In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from lightgbm import LGBMClassifier
from bayes_opt import BayesianOptimization
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, roc_auc_score, confusion_matrix, classification_report
import shap
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import fisher_exact
from scipy.stats import chi2_contingency
from scipy.stats import ttest_ind
import pingouin as pg
pd.set_option('display.max_rows', 1000)
pd.set_option('display.max_columns', 1000)

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

In [ ]:
# Load the clean dataset with all extracted features
file_path = 'df.csv'
df = pd.read_csv(file_path, na_values=['', ' ', 'NaN', 'nan'])

cols_to_exclude = ["Unnamed: 0"] 
df = df.drop(columns=cols_to_exclude)
print(f"Dataset shape: {df.shape}")
print(f"Columns: {len(df.columns)}")
print("\nFirst few rows:")
df.head()

In [ ]:
y=df['one_year_mortality']
x = df.drop(columns=['one_year_mortality'])
x1 = df.drop(columns=['one_year_mortality'])

# Convert all object columns to category dtype for LightGBM
categorical_cols = ['GENDER',	'ETHCAT',	'DIAB', 'ABO', 'CMV_STATUS',	'CMV_HIGH_RISK_MISMATCH',	'ECMO_TRR', 'FUNC_STAT_GROUP',
                    'GROUPING',	'PRIOR_LUNG_SURG_TRR',	'STEROID',	'TRACHEOSTOMY_TRR',	'TRANSFUSIONS',	
                    'TX_YEAR',	'VENTILATOR_TRR', 'GENDER_DON',	'ETHCAT_DON', 'CONTIN_CIG_DON',	'ABO_DON',	'ABO_MATCH',	'CLIN_INFECT_DON',
                    'CMV_DON',	'BRONCHO_LT_DON_CAT',	'BRONCHO_RT_DON_CAT',	'DEATH_CIRCUM_DON_MERGED',	'DEATH_MECH_DON_MERGED',	
                    'DIABETES_DON',	'ECD_DONOR',	'HEP_C_ANTI_DON',	'MED_COND_TRR',	'DONOR_TYPE_BINARY', 'TX_TYPE', 'PERFUSED_PRIOR',	
                    'PERFUSED_BY',	'REGION',	'SHARE_TY',	'era' ]  

for col in categorical_cols:
    if not pd.api.types.is_integer_dtype(x[col]):
        col_series = x[col].astype("string")
        unique_vals = [v for v in col_series.dropna().unique() if v != "U"]
        mapping = {val: i for i, val in enumerate(unique_vals)}
        mapping["U"] = -1
        x[col] = col_series.map(mapping).astype(int)
        x1[col] = col_series.map(mapping).astype(int)

feature_names = x.columns.tolist()
idn=np.array(range(0, len(x)))
print('input shape', x.shape)
print('output shape', y.shape)
print('idn shape', idn.shape)

In [ ]:
seed=200
# Split data into train - val - test
X_train0, X_test, y_train0, y_test, idn_X_train0, idn_test = train_test_split(x, y, idn, stratify=y, test_size=0.20, random_state=seed)
X_train, X_val, y_train, y_val, idn_train, idn_val = train_test_split(X_train0, y_train0, idn_X_train0, stratify=y_train0, test_size=0.25,random_state=seed)

for col in categorical_cols:
    x[col]= x[col].astype('category')
    X_train[col] = X_train[col].astype('category')
    X_val[col] = X_val[col].astype('category')
    X_test[col] = X_test[col].astype('category')

print('size of data partitions')
print('...............INPUT..............OUTPUT..........')
print('Train      : ', X_train.shape, '         ', y_train.shape)
print('Validation : ', X_val.shape, '         ', y_val.shape)
print('Test       : ', X_test.shape, '         ', y_test.shape)

In [ ]:
def lgb_evaluate(max_depth, num_leaves, learning_rate, max_bin, colsample_bytree, reg_alpha, 
                 reg_lambda, subsample, min_child_samples, min_child_weight, n_estimators, scale_pos_weight):
  
    params = {'max_depth': int(max_depth),
              'num_leaves':int(num_leaves),
              'learning_rate' :learning_rate,
              'max_bin':int(max_bin),
              'colsample_bytree': colsample_bytree,
              'random_state': seed,
              'reg_alpha':reg_alpha,
              'reg_lambda':reg_lambda,
              'subsample': subsample,
              'min_child_samples' : int(min_child_samples),
              'min_child_weight' : min_child_weight,
              'n_estimators' :int(n_estimators),
              'scale_pos_weight':scale_pos_weight
             }
    modellgb = LGBMClassifier(verbose=-1)
    modellgb.set_params(**params)
    clf_lg= modellgb.fit(X_train,y_train)
    p_lg_train=clf_lg.predict_proba(X_train)[:,1]
    p_lg_val=clf_lg.predict_proba(X_val)[:,1]
    p_lg_test=clf_lg.predict_proba(X_test)[:,1]
    lg_auc_train = roc_auc_score(y_train, p_lg_train)
    lg_auc_val = roc_auc_score(y_val, p_lg_val)
    lg_auc_test = roc_auc_score(y_test, p_lg_test)
    explainer = shap.TreeExplainer(clf_lg)
    expected_value = explainer.expected_value
    shap_values = explainer.shap_values(x)

    #shap.summary_plot(shap_values, x1, feature_names =feature_names,max_display=50)
    #print ('Model Parameters', modellgb.get_params())
    print('************************************')
    print('Train AUC : ', np.round(lg_auc_train, decimals=4))
    print('Validation AUC : ', np.round(lg_auc_val, decimals=4))
    print('Test AUC       : ', np.round(lg_auc_test, decimals=4)) 
    print (modellgb.get_params())
    return np.round(lg_auc_val, decimals=4)

In [ ]:
#bayesian optimization of hyperparameters for lg_boost
gp_params = {"alpha": 1e-4}

lgb_bo = BayesianOptimization(lgb_evaluate, {'max_depth': (1, 10), 
                                             'num_leaves': (2,20),
                                             'learning_rate':(0.01, 0.4),
                                             'max_bin':(50, 800),
                                             'colsample_bytree': (0.01, 1.0),
                                             'reg_alpha':(1,10),
                                             'reg_lambda':(0.01,1),
                                             'subsample':(0.01,1),
                                             'min_child_samples' : (30, 100),
                                             'min_child_weight':(1, 5),
                                             'n_estimators' :(100,1000),
                                             'scale_pos_weight': (1,4)
                                            })

# Optimally needs quite a few more initiation points and number of iterations
lgb_bo.maximize(init_points=200, n_iter=50)

In [ ]:
best_param={'boosting_type': 'gbdt', 'class_weight': None, 'colsample_bytree': 0.3053951189060516, 'importance_type': 'split', 'learning_rate': 0.042803763894080075, 'max_depth': 3, 'min_child_samples': 50, 'min_child_weight': 3.118886132912796, 'min_split_gain': 0.0, 'n_estimators': 440, 'n_jobs': None, 'num_leaves': 8, 'objective': None, 'random_state': 200, 'reg_alpha': 5.525079739129535, 'reg_lambda': 0.7334346969076605, 'subsample': 0.6224768074882098, 'subsample_for_bin': 200000, 'subsample_freq': 0, 'verbose': -1, 'max_bin': 770, 'scale_pos_weight': 2.529780329318757}


In [ ]:
modellgb = LGBMClassifier(colsample_bytree=best_param['colsample_bytree'], learning_rate=best_param['learning_rate'], 
                          max_depth=int(best_param['max_depth']) , 
                          min_child_samples= int(best_param['min_child_samples']), 
                          min_child_weight= best_param['min_child_weight'], 
                          n_estimators= int(best_param['n_estimators']), 
                          num_leaves= int(best_param['num_leaves']), random_state= seed, 
                          reg_alpha=best_param['reg_alpha'], reg_lambda= best_param['reg_lambda'], 
                          subsample=  best_param['subsample'], max_bin= int(best_param['max_bin']),  
                          scale_pos_weight= best_param['scale_pos_weight'],verbose=-1)

clf_lg= modellgb.fit(X_train,y_train)

explainer = shap.TreeExplainer(clf_lg)
expected_value = explainer.expected_value
print('------------------------------------------------------')
print('Expected Value of Model (E(y_hat)) : ', expected_value)


In [ ]:
# METRICS
from sklearn.metrics import confusion_matrix


p_lg_train=clf_lg.predict_proba(X_train)[:,1]
p_lg_val=clf_lg.predict_proba(X_val)[:,1]
p_lg_test=clf_lg.predict_proba(X_test)[:,1]

y_pred_train = (clf_lg.predict_proba(X_train)[:,1] >= 0.39).astype(bool)
y_pred_val = (clf_lg.predict_proba(X_val)[:,1] >= 0.39).astype(bool)
y_pred_test = (clf_lg.predict_proba(X_test)[:,1] >= 0.39).astype(bool)

cm_train = confusion_matrix(y_train, y_pred_train)
cm_val = confusion_matrix(y_val, y_pred_val)
cm_test = confusion_matrix(y_test, y_pred_test)
#print(cm_train)
lg_auc_train = np.round(roc_auc_score(y_train, p_lg_train), decimals=4)
lg_auc_val = np.round(roc_auc_score(y_val, p_lg_val), decimals=4)
lg_auc_test = np.round(roc_auc_score(y_test, p_lg_test), decimals=4)

acc_train = np.round(accuracy_score(y_train, y_pred_train), decimals=4)
acc_val = np.round(accuracy_score(y_val, y_pred_val), decimals=4)
acc_test = np.round(accuracy_score(y_test, y_pred_test), decimals=4)

prec_train = np.round(precision_score(y_train, y_pred_train), decimals=4)
prec_val = np.round(precision_score(y_val, y_pred_val), decimals=4)
prec_test = np.round(precision_score(y_test, y_pred_test), decimals=4)

rec_train = np.round(cm_train[1,1]/(cm_train[1,1]+cm_train[0,1]), decimals=4)
rec_val = np.round(cm_val[1,1]/(cm_val[1,1]+cm_val[0,1]), decimals=4)
rec_test = np.round(cm_test[1,1]/(cm_test[1,1]+cm_test[0,1]), decimals=4)

specificity_train = np.round(cm_train[0,0]/(cm_train[0,0]+cm_train[1,0]), decimals=4)
specificity_val = np.round(cm_val[0,0]/(cm_val[0,0]+cm_val[1,0]), decimals=4)
specificity_test = np.round(cm_test[0,0]/(cm_test[0,0]+cm_test[1,0]), decimals=4)

print('........................MODEL PERFORMANCE......................')
print('              Train            Validation               Test')
print('AUC         :', lg_auc_train, '         ', lg_auc_val, '                 ', lg_auc_test)
print('Accuracy    :', acc_train, '         ', acc_val, '                 ', acc_test) 
print('Precision   :', prec_train, '            ', prec_val, '                 ', prec_test) 
print('Recall       :', rec_train, '          ', rec_val, '                 ', rec_test)
print('Specificity  :', specificity_train, '      ', specificity_val, '                 ', specificity_test)

print()
print('train')
print(classification_report(y_pred_train, y_train))

print()
print('val')
print(classification_report(y_pred_val, y_val))

print()
print('test')
print(classification_report(y_pred_test, y_test))



In [ ]:
feature_names = [
    "Recipient Age",
    "Recipient Gender",
    "Recipient Ethnicity",
    "Recipient Diabetes",
    "Recipient eGFR at Transplant",
    "Recipient Blood Type",
    "Recipient BMI at Match",
    "Recipient CMV Serostatus",
    "High-risk CMV Mismatch (D+/R-)",
    "Need for ECMO at Transplant",
    "LAS at Match",
    "CAS Sub-score at Match",
    "Change in LAS (Pre-CAS)",
    "Change in LAS (Post-CAS)",
    "Last FEV1",
    "Last FVC",
    "Last Mean PA Pressure",
    "Last Cardiac Output",
    "Functional Status",
    "Diagnosis Group",
    "Prior Lung Surgery",
    "Chronic Steroid Use",
    "Tracheostomy at Transplant",
    "Blood Transfusions After Listing",
    "Transplant Year",
    "Need for Ventilator at Transplant",
    "Donor Age",
    "Donor Gender",
    "Donor Ethnicity",
    "Donor BMI",
    "Donor Cigarette Smoking",
    "Donor Blood Type",
    "ABO Match",
    "Donor Clinical Infection",
    "Donor CMV Serostatus",
    "Left Bronchoscopy Abnormality",
    "Right Bronchoscopy Abnormality",
    "Donor Circumstances of Death",
    "Donor Mechanism of Death",
    "Donor Diabetes",
    "Extended Criteria Donor",
    "Donor Hepatitis C Seropositive",
    "Recipient Medical Condition",
    "Donor Type (DCD vs DBD)",
    "Donor PaO₂ (challenge)",
    "Transplant Type",
    "Days on Waitlist",
    "Distance (nautical miles)",
    "Out of Body Time (hours)",
    "Machine Perfusion Used",
    "Perfusion Entity",
    "Region",
    "Share Type",
    "Allocation Era"
]

In [ ]:
!pip install matplotlib

In [ ]:
shap_values = explainer.shap_values(x)

# index of features in order of decending importance 
imp_ordered_ind = np.argsort(-np.abs(shap_values).mean(0))
max_feature = 54

shap.summary_plot(shap_values, x1, feature_names =feature_names,max_display=max_feature,show=False)
#[print(feature_names[k]) for k in imp_ordered_ind]
plt.savefig("shap_summary_plot.png", dpi=300, bbox_inches='tight')  # or .pdf for vector output
plt.close()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---------------------------------------------------------------
# Indices
# ---------------------------------------------------------------
ischemic_index   = 2
donor_type_index = 43
perfusion_index  = 46
tx_year_index    = 9
era_index        = 38
cutoff = 400

x = np.array(x)

# ---------------------------------------------------------------
# SHAP SUM
# ---------------------------------------------------------------
shap_sum = (
    shap_values[:, imp_ordered_ind[ischemic_index]] +
    shap_values[:, imp_ordered_ind[donor_type_index]] +
    shap_values[:, imp_ordered_ind[perfusion_index]] +
    shap_values[:, imp_ordered_ind[tx_year_index]] +
    shap_values[:, imp_ordered_ind[era_index]]
)

donor = x[:, imp_ordered_ind[donor_type_index]].astype(int)

perf = x[:, imp_ordered_ind[perfusion_index]].astype(int)
perf = np.where(perf == -1, 0, perf)

ischemia = x[:, imp_ordered_ind[ischemic_index]]*60
era = x[:, imp_ordered_ind[era_index]].astype(int)
tx_year = x[:, imp_ordered_ind[tx_year_index]].astype(int)

# ---------------------------------------------------------------
# Colors
# ---------------------------------------------------------------
color_tones = {
    (0, 0): "#FFFFFF",
    (0, 1): "#7F7F7F",
    (0, 2): "#000000",
    (1, 0): "#FFFFE6",
    (1, 1): "#CCFF66",
    (1, 2): "#00AA00"
}

unique_years = np.sort(np.unique(tx_year))

# ---------------------------------------------------------------
# Figure layout
# ---------------------------------------------------------------
fig = plt.figure(figsize=(max(24, 2 * len(unique_years)), 16))
gs = fig.add_gridspec(2, 1, height_ratios=[2.0, 2.0], hspace=0.38)

ax_perf = fig.add_subplot(gs[0, 0])
ax_noperf = fig.add_subplot(gs[1, 0])

# ---------------------------------------------------------------
# Box plot function
# ---------------------------------------------------------------
def make_group_boxplot(ax, perf_flag, title):
    group_labels = ["DBD – Short", "DBD – Long", "DCD – Short", "DCD – Long"]
    base_positions = [0, 1.7, 3.6, 5.3]

    data = []
    colors_bp = []
    positions = []
    year_tick_labels = []

    n_years = len(unique_years)
    box_width = max(0.08, 0.30 / n_years)
    offsets = np.linspace(-0.5, 0.5, n_years)

    idx = 0

    for d in [0, 1]:
        for flag in ["short", "long"]:
            base_x = base_positions[idx]
            idx += 1

            for j, yr in enumerate(unique_years):
                if flag == "short":
                    mask = (
                        (donor == d) &
                        (tx_year == yr) &
                        (perf == perf_flag) &
                        (ischemia < cutoff)
                    )
                else:
                    mask = (
                        (donor == d) &
                        (tx_year == yr) &
                        (perf == perf_flag) &
                        (ischemia >= cutoff)
                    )

                vals = shap_sum[mask]
                data.append(vals)

                xpos = base_x + offsets[j]
                positions.append(xpos)
                year_tick_labels.append(str(yr))

                era_vals = era[mask]
                era_mode = np.bincount(era_vals).argmax() if len(era_vals) > 0 else 0
                colors_bp.append(color_tones[(d, era_mode)])

    bp = ax.boxplot(
        data,
        positions=positions,
        widths=box_width,
        patch_artist=True,
        manage_ticks=False
    )

    for patch, c in zip(bp["boxes"], colors_bp):
        patch.set_facecolor(c)
        patch.set_edgecolor("k")
        patch.set_linewidth(1.2)
        patch.set_alpha(1.0)

    # Highlight long ischemic time boxes
    long_idxs = [i for i in range(len(data)) if (i // n_years) in [1, 3]]

    for i in long_idxs:
        bp["boxes"][i].set_edgecolor("red")
        bp["boxes"][i].set_linewidth(2.5)

    # Add n above each box
    for vals, xpos in zip(data, positions):
        n = len(vals)
        label_color = "red" if n < 14 else "black"

        ax.text(
            xpos,
            0.58,
            f"n={n}",
            rotation=90,
            fontsize=10,
            color=label_color,
            ha="center",
            va="bottom"
        )

    ax.axhline(0, color="gray", linestyle="--", linewidth=1)

    ax.set_ylim(-0.40, 0.68)
    ax.set_xlim(min(base_positions) - 0.75, max(base_positions) + 0.75)

    # Major x-ticks: group labels
    ax.set_xticks(base_positions)
    ax.set_xticklabels(group_labels, fontsize=13)

    # Minor x-ticks: years, aligned exactly under each box
    ax.set_xticks(positions, minor=True)
    ax.set_xticklabels(
        year_tick_labels,
        minor=True,
        rotation=90,
        fontsize=10
    )

    ax.tick_params(axis="x", which="minor", length=0, pad=2)
    ax.tick_params(axis="x", which="major", length=0, pad=28)

    ax.set_title(title, fontsize=18)
    ax.set_ylabel("Sum of SHAP Values", fontsize=14)
    ax.grid(axis="y", linestyle="--", alpha=0.4)

# ---------------------------------------------------------------
# Plot panels
# ---------------------------------------------------------------
make_group_boxplot(ax_perf, perf_flag=1, title="Perfusion")
make_group_boxplot(ax_noperf, perf_flag=0, title="No Perfusion")

plt.tight_layout()
plt.savefig("figure4.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# Feature indices 
ischemic_index = 2
donor_type_index = 43
perfusion_index = 46

cutoff = 400
x = np.array(x)

# Perfusion unknown (-1) → No perfusion (0)
x[:, imp_ordered_ind[perfusion_index]] = np.where(
    x[:, imp_ordered_ind[perfusion_index]] == -1,
    0,
    x[:, imp_ordered_ind[perfusion_index]]
)

# Convert ischemic time from hours to minutes
ischemia_minutes = x[:, imp_ordered_ind[ischemic_index]] * 60

# Color palette
colors = {
    (0, 0): '#1b9e77',  
    (0, 1): '#7570b3',  
    (1, 0): '#d95f02',  
    (1, 1): '#e7298a'   
}
labels = ['DBD / No Perfusion', 'DBD / Perfusion',
          'DCD / No Perfusion', 'DCD / Perfusion']
categories = [(0, 0), (0, 1), (1, 0), (1, 1)]

# ================================================
# FIGURE 1 — SCATTER (LEFT PANEL)
# ================================================
plt.figure(figsize=(8, 7))

sizes = x[:, imp_ordered_ind[perfusion_index]]
bubble_sizes = 30 * sizes + 10

y_values = (
    shap_values[:, imp_ordered_ind[ischemic_index]] +
    shap_values[:, imp_ordered_ind[donor_type_index]] +
    shap_values[:, imp_ordered_ind[perfusion_index]]
)

colors_scatter = np.where(
    x[:, imp_ordered_ind[donor_type_index]] == 1,
    'green', 'white'
)
edge_colors = np.where(
    x[:, imp_ordered_ind[donor_type_index]] == 1,
    'darkgreen', 'gray'
)

scatter = plt.scatter(
    ischemia_minutes,
    y_values,
    s=bubble_sizes,
    c=colors_scatter,
    edgecolor=edge_colors,
    alpha=0.8
)

# Cutoff
plt.axvline(cutoff, color='red', linestyle='--', linewidth=1)
plt.text(cutoff + 0.1, 0.55, 'Cutoff = 400min', color='red', fontsize=9)

plt.xlabel('Out of Body Time (hours)')
plt.ylabel('Sum of SHAP Values\n(Out of Body Time + Donor Type + Perfusion Prior)')
plt.title('')
plt.grid(alpha=0.4)
plt.ylim(-0.2, 0.6)

# -------------------------------------------------
# 🔹 MINI LEGEND 
# -------------------------------------------------
legend_elements = [
    # Donor type (color)
    Line2D([0], [0], color='none', label='Bubble color:', linestyle=''),
    Line2D([0], [0], marker='o', linestyle='None', 
           markerfacecolor='green', markeredgecolor='darkgreen',
           label='DCD (green)', markersize=9),
    Line2D([0], [0], marker='o', linestyle='None',
           markerfacecolor='white', markeredgecolor='gray',
           label='DBD (white)', markersize=9),

    # Perfusion (bubble size)
    Line2D([0], [0], color='none', label='Bubble size:', linestyle=''),
    Line2D([0], [0], marker='o', linestyle='None',
           markerfacecolor='gray', markeredgecolor='black',
           label='No Perfusion / Unknown (small)', markersize=6),
    Line2D([0], [0], marker='o', linestyle='None',
           markerfacecolor='gray', markeredgecolor='black',
           label='Perfusion (large)', markersize=12)
]

legend = plt.legend(
    handles=legend_elements,
    title="",
    loc='lower right',
    fontsize=9,
    frameon=True,
    borderpad=1.2,
    labelspacing=0.9
)

legend.get_title().set_fontsize(10)


plt.tight_layout()
plt.savefig("Figure1_Scatter.png", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# ================================================
# FIGURE 2 — BOXPLOTS (RIGHT PANEL)
# ================================================
plt.figure(figsize=(10, 7))

# Convert ischemic time from hours to minutes
ischemia_minutes = x[:, imp_ordered_ind[ischemic_index]] * 60

positions, plot_data, plot_colors = [], [], []
group_counts, mortality_rates = [], []

for j, (donor, perf) in enumerate(categories):
    for k in range(2):  # < cutoff and >= cutoff
        if k == 0:
            mask = (
                (ischemia_minutes < cutoff) &
                (x[:, imp_ordered_ind[donor_type_index]] == donor) &
                (x[:, imp_ordered_ind[perfusion_index]] == perf)
            )
        else:
            mask = (
                (ischemia_minutes >= cutoff) &
                (x[:, imp_ordered_ind[donor_type_index]] == donor) &
                (x[:, imp_ordered_ind[perfusion_index]] == perf)
            )

        shap_sum = (
            shap_values[mask, imp_ordered_ind[ischemic_index]] +
            shap_values[mask, imp_ordered_ind[donor_type_index]] +
            shap_values[mask, imp_ordered_ind[perfusion_index]]
        )

        plot_data.append(shap_sum)
        plot_colors.append(colors[(donor, perf)])
        group_counts.append(len(shap_sum))
        mortality_rates.append(y[mask].mean() * 100 if len(shap_sum) > 0 else np.nan)

        positions.append(j + k * (len(categories) + 1))

bp = plt.boxplot(
    plot_data,
    positions=positions,
    patch_artist=True,
    widths=0.6
)

for patch, color in zip(bp['boxes'], plot_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

for pos, data, n, mort_rate in zip(positions, plot_data, group_counts, mortality_rates):
    if n > 0:
        y_text = min(0.58, np.max(data) + 0.02)
        plt.text(
            pos,
            y_text,
            f'N={n}\n{mort_rate:.1f}%',
            ha='center',
            fontsize=8
        )

plt.xticks(
    [1.5, 6.5],
    ['Out of Body Time < 400 min', 'Out of Body Time ≥ 400 min'],
    fontsize=10
)

plt.ylabel('Sum of SHAP Values\n(Out of Body Time + Donor Type + Perfusion Prior)')
plt.title('')
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.ylim(-0.2, 0.6)

handles = [Line2D([0], [0], color=color, lw=10) for color in colors.values()]
plt.legend(handles, labels, loc='upper left', fontsize=9)

plt.tight_layout()
plt.savefig("ischemic_time_boxplot.png", dpi=300)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# Indices
grouping_index = 25
donor_type_index = 43
perfusion_index = 46

# Arrays
X = np.array(x)
SV = np.array(shap_values)

def col(idx): return X[:, imp_ordered_ind[idx]]
def shv(idx): return SV[:, imp_ordered_ind[idx]]

# ==============================
# 1) Extract variables
# ==============================

# GROUPING → round and cast to int
grouping = np.rint(col(grouping_index)).astype(int)

# 0→B, 1→D, 2→A, 3→C mapping
group_labels = {0: 'B', 1: 'D', 2: 'A', 3: 'C'}
group_order = [2, 0, 3, 1]   # A,B,C,D görsel sırası

# donor type
donor = (np.nan_to_num(col(donor_type_index), nan=0.0) > 0.5).astype(int)

# perfusion (unknown=-1 → 0)
perf_raw = np.nan_to_num(col(perfusion_index), nan=-1)
perf = np.where(perf_raw == -1, 0, perf_raw).astype(int)

# ==============================
# 2) Map group numbers to plot positions
# ==============================
# A,B,C,D = 0,1,2,3 olacak şekilde
group_to_pos = {2: 0, 0: 1, 3: 2, 1: 3}
x_positions = np.array([group_to_pos[g] for g in grouping], dtype=float)

# jitter
rng = np.random.default_rng(42)
x_jittered = x_positions + (rng.random(len(x_positions)) - 0.5) * 0.25

# ==============================
# 3) SHAP sum
# ==============================
shap_sum = (
    shv(grouping_index) +
    shv(donor_type_index) +
    shv(perfusion_index)
)

# ==============================
# 4) SCATTER colors + bubble sizes
# ==============================
bubble_sizes = 30 * perf + 10

colors_scatter = np.where(donor == 1, 'green', 'white')
edge_colors = np.where(donor == 1, 'darkgreen', 'gray')

# ==============================
# 5) PLOT
# ==============================
plt.figure(figsize=(8, 7))

plt.scatter(
    x_jittered,
    shap_sum,
    s=bubble_sizes,
    c=colors_scatter,
    edgecolor=edge_colors,
    alpha=0.82,
    linewidth=0.7
)

plt.xlabel("Diagnosis Group")
plt.ylabel("Sum of SHAP Values\n(Diagnosis Group + Donor Type + Perfusion Prior)")
plt.title("")
plt.grid(alpha=0.4)
plt.ylim(-0.1, 0.5)

plt.xlim(-0.5, 3.5)
plt.xticks([0, 1, 2, 3], ['A', 'B', 'C', 'D'])

# ==============================
# 6) LEGEND
# ==============================
legend_elements = [
    Line2D([0], [0], color='none', label='Bubble color:', linestyle=''),
    Line2D([0], [0], marker='o', linestyle='None',
           markerfacecolor='green', markeredgecolor='darkgreen',
           label='DCD (green)', markersize=9),
    Line2D([0], [0], marker='o', linestyle='None',
           markerfacecolor='white', markeredgecolor='gray',
           label='DBD (white)', markersize=9),
    Line2D([0], [0], color='none', label='Bubble size:', linestyle=''),
    Line2D([0], [0], marker='o', linestyle='None',
           markerfacecolor='gray', markeredgecolor='black',
           label='No Perfusion (small)', markersize=6),
    Line2D([0], [0], marker='o', linestyle='None',
           markerfacecolor='gray', markeredgecolor='black',
           label='Perfusion (large)', markersize=12),
]

plt.legend(
    handles=legend_elements,
    loc='upper left',
    fontsize=9,
    frameon=True,
    borderpad=1.2,
    labelspacing=0.9
)

plt.tight_layout()
plt.savefig("Figure_grouping_Scatter_updated.png", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# Indices
grouping_index = 25
donor_type_index = 43

# Arrays
X = np.array(x)
SV = np.array(shap_values)

def col(idx): return X[:, imp_ordered_ind[idx]]
def shv(idx): return SV[:, imp_ordered_ind[idx]]

# ==============================
# 1) Extract variables
# ==============================

# GROUPING → round and cast to int
grouping = np.rint(col(grouping_index)).astype(int)

# 0→B, 1→D, 2→A, 3→C mapping
group_labels = {0: 'B', 1: 'D', 2: 'A', 3: 'C'}
group_order = [2, 0, 3, 1]   

# donor type (binary)
donor = (np.nan_to_num(col(donor_type_index), nan=0.0) > 0.5).astype(int)

# ==============================
# 2) Map group numbers to plot positions
# ==============================
group_to_pos = {2: 0, 0: 1, 3: 2, 1: 3}   # A,B,C,D
x_positions = np.array([group_to_pos[g] for g in grouping], dtype=float)

# jitter
rng = np.random.default_rng(42)
x_jittered = x_positions + (rng.random(len(x_positions)) - 0.5) * 0.25

# ==============================
# 3) SHAP sum (NO PERFUSION)
# ==============================
shap_sum = (
    shv(grouping_index) +
    shv(donor_type_index)
)

# ==============================
# 4) SCATTER colors + bubble sizes
# ==============================
bubble_sizes = 40 * np.ones(len(donor))   # SABİT boyut (perf yok)
colors_scatter = np.where(donor == 1, 'green', 'white')
edge_colors = np.where(donor == 1, 'darkgreen', 'gray')

# ==============================
# 5) PLOT
# ==============================
plt.figure(figsize=(8, 7))

plt.scatter(
    x_jittered,
    shap_sum,
    s=bubble_sizes,
    c=colors_scatter,
    edgecolor=edge_colors,
    alpha=0.82,
    linewidth=0.7
)

plt.xlabel("Diagnosis Group")
plt.ylabel("Sum of SHAP Values\n(Diagnosis Group + Donor Type)")
plt.title("")
plt.grid(alpha=0.4)
plt.ylim(-0.1, 0.5)

plt.xlim(-0.5, 3.5)
plt.xticks([0, 1, 2, 3], ['A', 'B', 'C', 'D'])

# ==============================
# 6) LEGEND (Perfusion REMOVED)
# ==============================
legend_elements = [
    Line2D([0], [0], color='none', label='Bubble color:', linestyle=''),
    Line2D([0], [0], marker='o', linestyle='None',
           markerfacecolor='green', markeredgecolor='darkgreen',
           label='DCD (green)', markersize=9),
    Line2D([0], [0], marker='o', linestyle='None',
           markerfacecolor='white', markeredgecolor='gray',
           label='DBD (white)', markersize=9),
]

plt.legend(
    handles=legend_elements,
    loc='upper left',
    fontsize=9,
    frameon=True,
    borderpad=1.2,
    labelspacing=0.9
)

plt.tight_layout()
plt.savefig("Figure_grouping_Scatter_noPerfusion.png", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# ---------------------------
# Indices
# ---------------------------
grouping_index = 25
donor_type_index = 43
perfusion_index = 46

# ---------------------------
# Pull arrays
# ---------------------------
X = np.array(x)
SV = np.array(shap_values)
Y = np.array(y)

def col(idx): return X[:, imp_ordered_ind[idx]]
def shv(idx): return SV[:, imp_ordered_ind[idx]]

# ---------------------------
# GROUPING VALUES (mapped to A,B,C,D)
# ---------------------------
# 0→B, 1→D, 2→A, 3→C
group_labels = {2: 'A', 0: 'B', 3: 'C', 1: 'D'}
group_order  = [2, 0, 3, 1]   # A, B, C, D visual order

grouping = np.rint(col(grouping_index)).astype(int)

# ---------------------------
# Donor & Perfusion
# ---------------------------
donor = (np.nan_to_num(col(donor_type_index), nan=0.0) > 0.5).astype(int)

perf_raw = np.nan_to_num(col(perfusion_index), nan=-1)
perf = np.where(perf_raw == -1, 0, perf_raw).astype(int)

# ---------------------------
# 4 categories inside each group
# ---------------------------
categories = [(0,0), (0,1), (1,0), (1,1)]
cat_labels = ["DBD / No Perfusion", "DBD / Perfusion",
               "DCD / No Perfusion", "DCD / Perfusion"]

colors = {
    (0, 0): '#1b9e77',
    (0, 1): '#7570b3',
    (1, 0): '#d95f02',
    (1, 1): '#e7298a'
}

# ---------------------------
# Collect data for plot
# ---------------------------
positions = []
plot_data = []
plot_colors = []
group_counts = []
mortality_rates = []

cluster_spacing = 6        # distance between A,B,C,D clusters
within_spacing  = [-1.5, -0.5, 0.5, 1.5]  # position offsets within each cluster

for gi, g in enumerate(group_order):  # A,B,C,D in correct visual order
    cluster_center = gi * cluster_spacing
    
    for offset, (d, p) in zip(within_spacing, categories):
        
        mask = (grouping == g) & (donor == d) & (perf == p)

        shap_sum = (
            shv(grouping_index)[mask] +
            shv(donor_type_index)[mask] +
            shv(perfusion_index)[mask]
        )

        positions.append(cluster_center + offset)
        plot_data.append(shap_sum)
        plot_colors.append(colors[(d, p)])
        group_counts.append(len(shap_sum))
        mortality_rates.append(Y[mask].mean() * 100 if mask.any() else np.nan)

# ---------------------------
# Plot
# ---------------------------
plt.figure(figsize=(16, 8))

bp = plt.boxplot(
    plot_data,
    positions=positions,
    patch_artist=True,
    widths=0.8
)

# Set box colors
for patch, color in zip(bp['boxes'], plot_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)

# Add N and mortality
for pos, vals, n, mr in zip(positions, plot_data, group_counts, mortality_rates):
    if n > 0:
        ymax = np.nanmax(vals)
        plt.text(
            pos,
            min(0.55, ymax + 0.03),
            f"N={n}\n{mr:.1f}%",
            ha='center',
            fontsize=8
        )

# ---------------------------
# Axis setup
# ---------------------------
xticks = [i * cluster_spacing for i in range(len(group_order))]
xticklabels = [group_labels[g] for g in group_order]

plt.xticks(xticks, xticklabels, fontsize=12)
plt.ylabel("Sum of SHAP Values\n(Diagnosis Group + Donor Type + Perfusion Prior)", fontsize=11)
plt.grid(axis='y', alpha=0.5)
plt.ylim(-0.15, 0.55)

# Legend
handles = [Line2D([0], [0], color=colors[c], lw=10) for c in categories]
plt.legend(handles, cat_labels, loc='upper left', fontsize=10)

plt.tight_layout()
plt.savefig("grouping_16_boxplots.png", dpi=300)
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# ---------------------------
# Indices
# ---------------------------
grouping_index = 25
donor_type_index = 43

# ---------------------------
# Pull arrays
# ---------------------------
X = np.array(x)
SV = np.array(shap_values)
Y = np.array(y)

def col(idx): 
    return X[:, imp_ordered_ind[idx]]

def shv(idx): 
    return SV[:, imp_ordered_ind[idx]]

# ---------------------------
# GROUPING (A,B,C,D)
# ---------------------------
group_labels = {2: 'A', 0: 'B', 3: 'C', 1: 'D'}
group_order  = [2, 0, 3, 1]

grouping = np.rint(col(grouping_index)).astype(int)

# ---------------------------
# Donor (0=DBD white, 1=DCD green)
# ---------------------------
donor = (np.nan_to_num(col(donor_type_index), nan=0.0) > 0.5).astype(int)

categories = [0, 1]  # 0=DBD, 1=DCD
cat_labels = ["DBD", "DCD"]

# COLORS matching scatter plot
colors = {
    0: ("white", "gray"),       # face, edge
    1: ("green", "darkgreen")
}

# ---------------------------
# Collect data for plot
# ---------------------------
positions = []
plot_data = []
plot_facecolors = []
plot_edgecolors = []
group_counts = []
mortality_rates = []

cluster_spacing = 6
within_spacing  = [-0.7, 0.7]

for gi, g in enumerate(group_order):
    cluster_center = gi * cluster_spacing
    
    for offset, d in zip(within_spacing, categories):
        
        mask = (grouping == g) & (donor == d)

        shap_sum = (
            shv(grouping_index)[mask] +
            shv(donor_type_index)[mask]
        )

        positions.append(cluster_center + offset)
        plot_data.append(shap_sum)
        plot_facecolors.append(colors[d][0])
        plot_edgecolors.append(colors[d][1])
        group_counts.append(len(shap_sum))
        mortality_rates.append(Y[mask].mean() * 100 if mask.any() else np.nan)

# ---------------------------
# Plot
# ---------------------------
plt.figure(figsize=(14, 8))

bp = plt.boxplot(
    plot_data,
    positions=positions,
    patch_artist=True,
    widths=1.0
)

# Color each box with scatter colors
for patch, face, edge in zip(bp['boxes'], plot_facecolors, plot_edgecolors):
    patch.set_facecolor(face)
    patch.set_edgecolor(edge)
    patch.set_linewidth(1.3)
    patch.set_alpha(0.85)

# Add N and mortality
for pos, vals, n, mr in zip(positions, plot_data, group_counts, mortality_rates):
    if n > 0:
        ymax = np.nanmax(vals)
        plt.text(
            pos,
            min(0.55, ymax + 0.03),
            f"N={n}\n{mr:.1f}%",
            ha='center',
            fontsize=9
        )

# ---------------------------
# Axis
# ---------------------------
xticks = [i * cluster_spacing for i in range(len(group_order))]
xticklabels = [group_labels[g] for g in group_order]

plt.xticks(xticks, xticklabels, fontsize=13)
plt.ylabel("Sum of SHAP Values\n(Diagnosis Group + Donor Type)", fontsize=12)
plt.grid(axis='y', alpha=0.5)
plt.ylim(-0.15, 0.55)

# ---------------------------
# Legend (white & green)
# ---------------------------
handles = [
    Line2D([0], [0], marker='s', color='gray', markerfacecolor='white',
           markeredgecolor='gray', markersize=12, linestyle='None', label="DBD"),
    Line2D([0], [0], marker='s', color='darkgreen', markerfacecolor='green',
           markeredgecolor='darkgreen', markersize=12, linestyle='None', label="DCD")
]

plt.legend(handles=handles, loc='upper left', fontsize=11)

plt.tight_layout()
plt.savefig("grouping_boxplot_white_green.png", dpi=300)
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# ---------------------------------------------------------------
# Indices
# ---------------------------------------------------------------
grouping_index   = 25   # diagnosis (A=2, D=1)
donor_type_index = 43
perfusion_index  = 46
tx_year_index    = 9
era_index        = 38

# ---------------------------------------------------------------
# Pull arrays
# ---------------------------------------------------------------
X = np.array(x)
SV = np.array(shap_values)
Y = np.array(y)

diag  = X[:, imp_ordered_ind[grouping_index]].astype(int)  # diagnosis group
donor = X[:, imp_ordered_ind[donor_type_index]].astype(int)

perf_raw = X[:, imp_ordered_ind[perfusion_index]].astype(int)
perf = np.where(perf_raw == -1, 0, perf_raw)

era = X[:, imp_ordered_ind[era_index]].astype(int)

# ---------------------------------------------------------------
# SHAP SUM (diagnosis + donor + perfusion + year + era)
# ---------------------------------------------------------------
shap_sum = (
    SV[:, imp_ordered_ind[grouping_index]] +
    SV[:, imp_ordered_ind[donor_type_index]] +
    SV[:, imp_ordered_ind[perfusion_index]] +
    SV[:, imp_ordered_ind[tx_year_index]] +
    SV[:, imp_ordered_ind[era_index]]
)

# ---------------------------------------------------------------
# Colors
# ---------------------------------------------------------------
color_tones = {
    (0,0): "#FFFFFF",
    (0,1): "#7F7F7F",
    (0,2): "#000000",
    (1,0): "#FFFFE6",
    (1,1): "#CCFF66",
    (1,2): "#00AA00"
}

era_names = {0:"LAS-DSA", 1:"LAS-non-DSA", 2:"CAS"}

# ---------------------------------------------------------------
# FIGURE (ONLY 2 PANELS)
# ---------------------------------------------------------------
fig, (ax_perf, ax_noperf) = plt.subplots(1, 2, figsize=(26, 10))

# ===============================================================
# BOX PLOT FUNCTION — ONLY A AND D GROUPS
# ===============================================================
def make_group_boxplot(ax, perf_flag, title, y):

    # (Diagnosis, Donor)
    group_defs = [
        (2, 0),   # A – DBD
        (2, 1),   # A – DCD
        (1, 0),   # D – DBD
        (1, 1)    # D – DCD
    ]

    group_labels = ["A – DBD", "A – DCD", "D – DBD", "D – DCD"]
    base_positions = [0, 1.7, 3.4, 5.1]

    box_width = 0.22

    final_positions = []
    data = []
    colors_bp = []

    # -------------------------
    # Collect values for boxes
    # -------------------------
    for (dx_val, d_val), base_x in zip(group_defs, base_positions):
        for e_val in [0,1,2]:

            mask = (
                (diag == dx_val) &
                (donor == d_val) &
                (perf == perf_flag) &
                (era == e_val)
            )

            vals = shap_sum[mask]
            xpos = base_x + e_val * box_width

            data.append(vals)
            final_positions.append(xpos)
            colors_bp.append(color_tones[(d_val, e_val)])

    # -------------------------
    # DRAW BOXES
    # -------------------------
    bp = ax.boxplot(
        data,
        positions=final_positions,
        widths=box_width,
        patch_artist=True,
        manage_ticks=False
    )

    for patch, col in zip(bp["boxes"], colors_bp):
        patch.set_facecolor(col)
        patch.set_edgecolor("k")
        patch.set_linewidth(1.3)

    # -------------------------
    # N and % labels
    # -------------------------
    idx = 0
    for (dx_val, d_val), base_x in zip(group_defs, base_positions):
        for e_val in [0,1,2]:

            xpos = base_x + e_val * box_width

            mask = (
                (diag == dx_val) &
                (donor == d_val) &
                (perf == perf_flag) &
                (era == e_val)
            )

            vals = shap_sum[mask]

            if len(vals) == 0:
                ax.text(xpos, -0.25, "N=0\nNA", ha="center", fontsize=10)
            else:
                mort = y[mask].mean() * 100
                ypos = np.nanmax(vals) + 0.03
                ax.text(xpos, ypos, f"N={len(vals)}\n{mort:.1f}%",
                        ha="center", fontsize=10)

            idx += 1

    ax.set_ylim(-0.30, 0.62)

    # -------------------------
    # ERA LABELS BELOW BOXES
    # -------------------------
    era_y = -0.33   # position below the boxes

    for base_x in base_positions:
        for e_val in [0,1,2]:
            xpos = base_x + e_val * box_width
            ax.text(
                xpos, era_y,
                era_names[e_val],
                ha="center",
                va="top",
                rotation=90,
                fontsize=11
            )

    # -------------------------
    # X labels
    # -------------------------
    ax.set_xticks([p + 0.2 for p in base_positions])
    ax.set_xticklabels(group_labels, fontsize=14)

    ax.set_title(title, fontsize=18)
    ax.grid(axis="y", linestyle="--", alpha=0.4)

# ---------------------------------------------------------------
# PANELS
# ---------------------------------------------------------------
make_group_boxplot(ax_perf,    perf_flag=1, title="Perfusion",    y=Y)
make_group_boxplot(ax_noperf,  perf_flag=0, title="No Perfusion", y=Y)

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# ----------------------------------------------------------
# Indices
# ----------------------------------------------------------
grouping_index   = 25    # diagnosis group (A=2, D=1)
donor_type_index = 43
tx_year_index    = 9
era_index        = 38    # 0,1,2

# ----------------------------------------------------------
# Load arrays
# ----------------------------------------------------------
X  = np.array(x)
SV = np.array(shap_values)
Y  = np.array(y)

diag  = X[:, imp_ordered_ind[grouping_index]].astype(int)
donor = X[:, imp_ordered_ind[donor_type_index]].astype(int)
era   = X[:, imp_ordered_ind[era_index]].astype(int)

# ----------------------------------------------------------
# SHAP SUM (perfusion excluded)
# ----------------------------------------------------------
shap_sum = (
    SV[:, imp_ordered_ind[grouping_index]] +
    SV[:, imp_ordered_ind[donor_type_index]] +
    SV[:, imp_ordered_ind[tx_year_index]] +
    SV[:, imp_ordered_ind[era_index]]
)

# ----------------------------------------------------------
# ERA names
# ----------------------------------------------------------
era_names = {0:"LAS-DSA", 1:"LAS-non-DSA", 2:"CAS"}

# Groups of interest:
# dx_group = 2 → A
# dx_group = 1 → D
dx_names = {2:"A", 1:"D"}

# Donor types: 0=DBD, 1=DCD
donor_names = {0:"DBD", 1:"DCD"}

# Final internal order within each ERA:
# [A-DBD], [D-DBD], [A-DCD], [D-DCD]
ordered_pairs = [
    (2, 0),  # A-DBD
    (1, 0),  # D-DBD
    (2, 1),  # A-DCD
    (1, 1)   # D-DCD
]

# ----------------------------------------------------------
# Colors 
# ----------------------------------------------------------
face_colors = {0: "white", 1: "green"}
edge_colors = {0: "gray",  1: "darkgreen"}

# ----------------------------------------------------------
# Spacing to avoid overlap
# ----------------------------------------------------------
cluster_spacing = 10
within_spacing  = [-2.0, -0.7, 0.7, 2.0]

# ----------------------------------------------------------
# Collect data for boxplots
# ----------------------------------------------------------
positions = []
plot_data = []
face_list = []
edge_list = []
counts = []
mortality = []

for era_val in [0, 1, 2]:

    cluster_center = era_val * cluster_spacing

    for offset, (dx_val, donor_val) in zip(within_spacing, ordered_pairs):

        mask = (
            (diag == dx_val) &
            (donor == donor_val) &
            (era == era_val)
        )

        vals = shap_sum[mask]

        positions.append(cluster_center + offset)
        plot_data.append(vals)
        face_list.append(face_colors[donor_val])
        edge_list.append(edge_colors[donor_val])
        counts.append(len(vals))
        mortality.append(Y[mask].mean() * 100 if mask.any() else np.nan)

# ----------------------------------------------------------
# Plot
# ----------------------------------------------------------
plt.figure(figsize=(22, 10))

bp = plt.boxplot(
    plot_data,
    positions=positions,
    widths=1.1,
    patch_artist=True,
    manage_ticks=False
)

# Color boxes
for patch, fc, ec in zip(bp["boxes"], face_list, edge_list):
    patch.set_facecolor(fc)
    patch.set_edgecolor(ec)
    patch.set_linewidth(1.6)
    patch.set_alpha(0.9)

# ----------------------------------------------------------
# Add N and mortality labels
# ----------------------------------------------------------
for pos, vals, n, mr in zip(positions, plot_data, counts, mortality):

    if n > 0:
        ymax = np.nanmax(vals)
        plt.text(
            pos,
            min(0.55, ymax + 0.03),
            f"N={n}\n{mr:.1f}%",
            ha="center",
            fontsize=10
        )
    else:
        plt.text(pos, -0.22, "N=0\nNA", ha="center", fontsize=10)

# ----------------------------------------------------------
# X-axis group labels (A-DBD, D-DBD, A-DCD, D-DCD)
# ----------------------------------------------------------
group_labels = ["A", "D", "A", "D"]
xtick_labels = [group_labels[i % 4] for i in range(len(positions))]

plt.xticks(positions, xtick_labels, rotation=0, fontsize=12)

# ----------------------------------------------------------
# ERA labels centered under clusters
# ----------------------------------------------------------
era_centers = [i * cluster_spacing for i in [0, 1, 2]]

for c, name in zip(era_centers, ["LAS-DSA", "LAS-non-DSA", "CAS"]):
    plt.text(c, -0.28, name, ha="center", va="top", fontsize=15)

# ----------------------------------------------------------
# Axes and figure adjustments
# ----------------------------------------------------------
plt.ylabel("Sum of SHAP Values\n(Diagnosis Group + Donor Type + Transplantation Year + Era)", fontsize=14)
plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.ylim(-0.25, 0.60)

plt.subplots_adjust(bottom=0.25)

# ----------------------------------------------------------
# Legend
# ----------------------------------------------------------
legend_handles = [
    Line2D([0], [0], marker='s', color='gray',
           markerfacecolor='white', markeredgecolor='gray',
           markersize=12, linestyle='None', label="DBD"),
    Line2D([0], [0], marker='s', color='darkgreen',
           markerfacecolor='green', markeredgecolor='darkgreen',
           markersize=12, linestyle='None', label="DCD")
]

plt.legend(handles=legend_handles, fontsize=12, loc='upper left')

plt.tight_layout()
plt.savefig("ERA_boxplot_final.png", dpi=300)
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# ----------------------------------------------------------
# Indices
# ----------------------------------------------------------
grouping_index   = 25    # diagnosis group (A=2, D=1)
donor_type_index = 43
tx_year_index    = 9
era_index        = 38

# ----------------------------------------------------------
# Load arrays
# ----------------------------------------------------------
X  = np.array(x)
SV = np.array(shap_values)
Y  = np.array(y)

diag    = X[:, imp_ordered_ind[grouping_index]].astype(int)
donor   = X[:, imp_ordered_ind[donor_type_index]].astype(int)
tx_year = X[:, imp_ordered_ind[tx_year_index]].astype(int)
era     = X[:, imp_ordered_ind[era_index]].astype(int)

# ----------------------------------------------------------
# SHAP SUM (perfusion excluded)
# ----------------------------------------------------------
shap_sum = (
    SV[:, imp_ordered_ind[grouping_index]] +
    SV[:, imp_ordered_ind[donor_type_index]] +
    SV[:, imp_ordered_ind[tx_year_index]] +
    SV[:, imp_ordered_ind[era_index]]
)

# ----------------------------------------------------------
# Group definitions
# ----------------------------------------------------------
ordered_pairs = [
    (2, 0),  # A-DBD
    (1, 0),  # D-DBD
    (2, 1),  # A-DCD
    (1, 1)   # D-DCD
]

group_labels = ["A", "D", "A", "D"]

# ----------------------------------------------------------
# Colors (same as scatter plot)
# ----------------------------------------------------------
face_colors = {0: "white", 1: "green"}
edge_colors = {0: "gray",  1: "darkgreen"}

# ----------------------------------------------------------
# Years
# ----------------------------------------------------------
unique_years = np.sort(np.unique(tx_year))

# ----------------------------------------------------------
# Spacing
# ----------------------------------------------------------
cluster_spacing = 10               # distance between years
within_spacing  = [-2.0, -0.7, 0.7, 2.0]  # A-DBD, D-DBD, A-DCD, D-DCD

# ----------------------------------------------------------
# Collect data
# ----------------------------------------------------------
positions = []
plot_data = []
face_list = []
edge_list = []
counts = []
mortality = []
year_label_positions = []

for i, year in enumerate(unique_years):

    cluster_center = i * cluster_spacing
    year_label_positions.append(cluster_center)

    for offset, (dx_val, donor_val) in zip(within_spacing, ordered_pairs):

        mask = (
            (tx_year == year) &
            (diag == dx_val) &
            (donor == donor_val)
        )

        vals = shap_sum[mask]

        positions.append(cluster_center + offset)
        plot_data.append(vals)
        face_list.append(face_colors[donor_val])
        edge_list.append(edge_colors[donor_val])
        counts.append(len(vals))
        mortality.append(Y[mask].mean() * 100 if mask.any() else np.nan)

# ----------------------------------------------------------
# Plot
# ----------------------------------------------------------
plt.figure(figsize=(32, 10))

bp = plt.boxplot(
    plot_data,
    positions=positions,
    widths=1.4,
    patch_artist=True,
    manage_ticks=False
)

# ----------------------------------------------------------
# Color the boxes
# ----------------------------------------------------------
for patch, fc, ec in zip(bp["boxes"], face_list, edge_list):
    patch.set_facecolor(fc)
    patch.set_edgecolor(ec)
    patch.set_linewidth(1.4)
    patch.set_alpha(0.9)

# ----------------------------------------------------------
# Add N and mortality (tightly above boxes)
# ----------------------------------------------------------
for idx, (pos, vals, n, mr) in enumerate(zip(positions, plot_data, counts, mortality)):

    if len(vals) > 0:
        ymax = np.nanmax(vals)
    else:
        ymax = -0.05

    # group id inside each cluster: 0,1,2,3 → tiny separation
    group_id = idx % 4

    label_y = ymax + 0.025 + group_id * 0.015

    if n > 0:
        plt.text(
            pos,
            label_y,
            f"N={n}\n{mr:.1f}%",
            ha="center",
            fontsize=9
        )

# ----------------------------------------------------------
# Diagnosis labels (A D A D repeated)
# ----------------------------------------------------------
xtick_labels = [group_labels[i % 4] for i in range(len(positions))]
plt.xticks(positions, xtick_labels, fontsize=12)

# ----------------------------------------------------------
# YEAR labels — placed at bottom level
# ----------------------------------------------------------
for c, year in zip(year_label_positions, unique_years):
    plt.text(
        c, -0.33,
        str(year),
        ha="center",
        va="top",
        fontsize=15
    )

# ----------------------------------------------------------
# Axes formatting
# ----------------------------------------------------------
plt.ylim(-0.35, 0.50)
plt.grid(axis='y', linestyle='--', alpha=0.35)

plt.ylabel(
    "Sum of SHAP Values\n(Diagnosis Group + Donor Type + Transplantation Year + Era)",
    fontsize=15
)

plt.subplots_adjust(bottom=0.24)

# ----------------------------------------------------------
# Legend
# ----------------------------------------------------------
legend_handles = [
    Line2D([0], [0], marker='s', color='gray',
           markerfacecolor='white', markeredgecolor='gray',
           markersize=14, linestyle='None', label="DBD"),
    Line2D([0], [0], marker='s', color='darkgreen',
           markerfacecolor='green', markeredgecolor='darkgreen',
           markersize=14, linestyle='None', label="DCD")
]

plt.legend(handles=legend_handles, fontsize=14, loc='upper left')

plt.tight_layout()
plt.savefig("YEAR_boxplot_final_clean.png", dpi=300)
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# ---------------------------------------------------------------
# PULL ARRAYS
# ---------------------------------------------------------------
X  = np.array(x)
SV = np.array(shap_values)
Y  = np.array(y)

diag  = X[:, imp_ordered_ind[grouping_index]].astype(int)   # 1=D, 2=A
donor = X[:, imp_ordered_ind[donor_type_index]].astype(int) # 0=DBD, 1=DCD

perf_raw = X[:, imp_ordered_ind[perfusion_index]].astype(int)
perf = np.where(perf_raw == -1, 0, perf_raw)

year = X[:, imp_ordered_ind[tx_year_index]].astype(int)
era  = X[:, imp_ordered_ind[era_index]].astype(int)

years = np.sort(np.unique(year))

# ---------------------------------------------------------------
# SHAP SUM
# ---------------------------------------------------------------
shap_sum = (
    SV[:, imp_ordered_ind[grouping_index]] +
    SV[:, imp_ordered_ind[donor_type_index]] +
    SV[:, imp_ordered_ind[perfusion_index]] +
    SV[:, imp_ordered_ind[tx_year_index]] +
    SV[:, imp_ordered_ind[era_index]]
)

# ---------------------------------------------------------------
# COLOR PALETTE FOR ERA × DONOR
# ---------------------------------------------------------------
color_tones = {
    (0,0): "#FFFFFF",
    (0,1): "#7F7F7F",
    (0,2): "#000000",
    (1,0): "#DDFFDD",
    (1,1): "#66DD44",
    (1,2): "#009900"
}

# ---------------------------------------------------------------
# GROUP DEFINITIONS (A/D instead of short/long)
# ---------------------------------------------------------------
group_defs = [
    (2, 0, "A–DBD"),
    (2, 1, "A–DCD"),
    (1, 0, "D–DBD"),
    (1, 1, "D–DCD")
]

# fixed x-positions per group
base_positions = [0, 1.7, 3.6, 5.3]

# ===============================================================
# DRAW FUNCTION — with dummy boxes for missing years
# ===============================================================
def make_group_boxplot(ax, perf_flag, title):

    data = []
    positions = []
    colors_bp = []

    offsets = np.linspace(-0.45, +0.45, len(years))
    box_width = 0.70 / len(years)

    for g_idx, (diag_val, donor_val, label) in enumerate(group_defs):

        base_x = base_positions[g_idx]

        for j, yy in enumerate(years):

            mask = (
                (diag == diag_val) &
                (donor == donor_val) &
                (perf == perf_flag) &
                (year == yy)
            )

            vals = shap_sum[mask]

            # ---------- key fix: dummy box when empty ----------
            if len(vals) == 0:
                vals = np.array([np.nan, np.nan])   # draw an empty placeholder
                color = "#DDDDDD"  # light grey for empty years
            else:
                era_mode = np.bincount(era[mask]).argmax()
                color = color_tones[(donor_val, era_mode)]

            data.append(vals)

            xpos = base_x + offsets[j]
            positions.append(xpos)
            colors_bp.append(color)

            # mortality numbers only if non-empty
            if np.sum(mask) > 0:
                mort = Y[mask].mean()*100
                ax.text(
                    xpos, np.nanmax(vals) + 0.03,
                    f"{np.sum(mask)}\n{mort:.1f}%",
                    ha="center", fontsize=10
                )

            # year label under each box
            ax.text(
                xpos, -0.42, str(yy),
                rotation=90, fontsize=11,
                ha="center", va="top"
            )

    # Draw boxes
    bp = ax.boxplot(
        data,
        positions=positions,
        widths=box_width,
        patch_artist=True,
        manage_ticks=False
    )

    for patch, col in zip(bp["boxes"], colors_bp):
        patch.set_facecolor(col)
        patch.set_edgecolor("k")
        patch.set_linewidth(1.3)

    ax.set_ylim(-0.35, 0.60)
    ax.set_xticks(base_positions)
    ax.set_xticklabels([g[2] for g in group_defs], fontsize=15)
    ax.set_title(title, fontsize=20)
    ax.grid(axis="y", alpha=0.3)


# ---------------------------------------------------------------
# FIGURE — 2 PANELS (Perfusion / No Perfusion)
# ---------------------------------------------------------------
fig, (ax_p, ax_np) = plt.subplots(2, 1, figsize=(32, 18), sharey=True)

make_group_boxplot(ax_p,  perf_flag=1, title="Perfusion")
make_group_boxplot(ax_np, perf_flag=0, title="No Perfusion")
xmin = base_positions[0] - 1
xmax = base_positions[-1] + 1
ax_p.set_xlim(xmin, xmax)
ax_np.set_xlim(xmin, xmax)

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# ---------------------------------------------------------------
# Indices
# ---------------------------------------------------------------
grouping_index   = 25   # diagnosis (A=2, D=1)
donor_type_index = 43
perfusion_index  = 46
type_index       = 30

# ---------------------------------------------------------------
# Pull arrays
# ---------------------------------------------------------------
X = np.array(x)
SV = np.array(shap_values)
Y = np.array(y)

diag  = X[:, imp_ordered_ind[grouping_index]].astype(int)  # diagnosis group
donor = X[:, imp_ordered_ind[donor_type_index]].astype(int)

perf_raw = X[:, imp_ordered_ind[perfusion_index]].astype(int)
perf = np.where(perf_raw == -1, 0, perf_raw)

type = X[:, imp_ordered_ind[type_index]].astype(int)   # transplant type (0/1)

# ---------------------------------------------------------------
# SHAP SUM (diagnosis + donor + perfusion + type)
# ---------------------------------------------------------------
shap_sum = (
    SV[:, imp_ordered_ind[grouping_index]] +
    SV[:, imp_ordered_ind[donor_type_index]] +
    SV[:, imp_ordered_ind[perfusion_index]] +
    SV[:, imp_ordered_ind[type_index]]
)

# ---------------------------------------------------------------
# Colors
# ---------------------------------------------------------------
color_tones = {
    (0,0): "#7F7F7F",
    (0,1): "#000000",
    (1,0): "#CCFF66",
    (1,1): "#00AA00"
}

# TX TYPE LABELS
type_names = {1: "Single", 1: "Bilateral"}

# ---------------------------------------------------------------
# FIGURE (ONLY 2 PANELS)
# ---------------------------------------------------------------
fig, (ax_perf, ax_noperf) = plt.subplots(1, 2, figsize=(26, 10))


# ===============================================================
# BOX PLOT FUNCTION — ONLY A AND D GROUPS
# ===============================================================
def make_group_boxplot(ax, perf_flag, title, y):

    # (Diagnosis, Donor)
    group_defs = [
        (2, 0),   # A – DBD
        (2, 1),   # A – DCD
        (1, 0),   # D – DBD
        (1, 1)    # D – DCD
    ]

    group_labels = ["A – DBD", "A – DCD", "D – DBD", "D – DCD"]
    base_positions = [0, 1.7, 3.4, 5.1]

    box_width = 0.22

    final_positions = []
    data = []
    colors_bp = []

    # -------------------------
    # Collect values for boxes
    # -------------------------
    for (dx_val, d_val), base_x in zip(group_defs, base_positions):
        for t_val in [0,1]:

            mask = (
                (diag == dx_val) &
                (donor == d_val) &
                (perf == perf_flag) &
                (type == t_val)
            )

            vals = shap_sum[mask]
            xpos = base_x + t_val * box_width

            data.append(vals)
            final_positions.append(xpos)
            colors_bp.append(color_tones[(d_val, t_val)])

    # -------------------------
    # DRAW BOXES
    # -------------------------
    bp = ax.boxplot(
        data,
        positions=final_positions,
        widths=box_width,
        patch_artist=True,
        manage_ticks=False
    )

    for patch, col in zip(bp["boxes"], colors_bp):
        patch.set_facecolor(col)
        patch.set_edgecolor("k")
        patch.set_linewidth(1.3)

    # -------------------------
    # N and % labels
    # -------------------------
    idx = 0
    for (dx_val, d_val), base_x in zip(group_defs, base_positions):
        for t_val in [0,1]:

            xpos = base_x + t_val * box_width

            mask = (
                (diag == dx_val) &
                (donor == d_val) &
                (perf == perf_flag) &
                (type == t_val)
            )

            vals = shap_sum[mask]

            if len(vals) == 0:
                ax.text(xpos, -0.25, "N=0\nNA", ha="center", fontsize=10)
            else:
                mort = y[mask].mean() * 100
                ypos = np.nanmax(vals) + 0.03
                ax.text(xpos, ypos, f"N={len(vals)}\n{mort:.1f}%",
                        ha="center", fontsize=10)

            idx += 1

    ax.set_ylim(-0.30, 0.62)

    # -------------------------
    # TX TYPE LABELS BELOW BOXES
    # -------------------------
    label_y = -0.33  # position below boxes

    for base_x in base_positions:
        for t_val in [0,1]:
            xpos = base_x + t_val * box_width
            ax.text(
                xpos, label_y,
                type_names[t_val],   
                ha="center",
                va="top",
                rotation=90,
                fontsize=12
            )

    # -------------------------
    # X labels
    # -------------------------
    ax.set_xticks([p + 0.1 for p in base_positions])
    ax.set_xticklabels(group_labels, fontsize=14)

    ax.set_title(title, fontsize=18)
    ax.grid(axis="y", linestyle="--", alpha=0.4)


# ---------------------------------------------------------------
# PANELS
# ---------------------------------------------------------------
make_group_boxplot(ax_perf,    perf_flag=1, title="Perfusion",    y=Y)
make_group_boxplot(ax_noperf,  perf_flag=0, title="No Perfusion", y=Y)

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# ---------------------------------------------------------------
# Indices
# ---------------------------------------------------------------
grouping_index   = 25   # diagnosis (A=2, D=1)
donor_type_index = 43
type_index       = 30   # transplant type (0=Single, 1=Bilateral)

# ---------------------------------------------------------------
# Pull arrays
# ---------------------------------------------------------------
X  = np.array(x)
SV = np.array(shap_values)
Y  = np.array(y)

diag  = X[:, imp_ordered_ind[grouping_index]].astype(int)
donor = X[:, imp_ordered_ind[donor_type_index]].astype(int)
tx_type = X[:, imp_ordered_ind[type_index]].astype(int)   # 0/1

# ---------------------------------------------------------------
# SHAP SUM (NO PERFUSION ANYMORE)
# ---------------------------------------------------------------
shap_sum = (
    SV[:, imp_ordered_ind[grouping_index]] +
    SV[:, imp_ordered_ind[donor_type_index]] +
    SV[:, imp_ordered_ind[type_index]]
)

# ---------------------------------------------------------------
# Categories inside each TX TYPE
# ---------------------------------------------------------------
# Order: A-DBD, D-DBD, A-DCD, D-DCD
ordered_pairs = [
    (2, 0),
    (1, 0),
    (2, 1),
    (1, 1)
]

subgroup_labels = ["A", "D", "A", "D"]

# ---------------------------------------------------------------
# Colors 
# donor=0 → white, donor=1 → green
# ---------------------------------------------------------------
face_colors = {0: "white", 1: "green"}
edge_colors = {0: "gray",  1: "darkgreen"}

# ---------------------------------------------------------------
# Spacing
# ---------------------------------------------------------------
group_spacing   = 5        # SPACE BETWEEN SINGLE ↔ BILATERAL
within_spacing  = [-1.5, -0.5, 0.5, 1.5]

tx_groups = {0: "Single", 1: "Bilateral"}

# ---------------------------------------------------------------
# Collect data for plotting
# ---------------------------------------------------------------
positions = []
plot_data = []
face_list = []
edge_list = []
counts = []
mortality = []

for g in [0, 1]:   # Single then Bilateral

    center = g * group_spacing

    for offset, (dx_val, donor_val) in zip(within_spacing, ordered_pairs):

        mask = (
            (tx_type == g) &
            (diag == dx_val) &
            (donor == donor_val)
        )

        vals = shap_sum[mask]

        positions.append(center + offset)
        plot_data.append(vals)
        face_list.append(face_colors[donor_val])
        edge_list.append(edge_colors[donor_val])
        counts.append(len(vals))
        mortality.append(Y[mask].mean() * 100 if mask.any() else np.nan)

# ---------------------------------------------------------------
# PLOT
# ---------------------------------------------------------------
plt.figure(figsize=(10, 6))

bp = plt.boxplot(
    plot_data,
    positions=positions,
    widths=0.4,
    patch_artist=True,
    manage_ticks=False
)

# Color boxes
for patch, fc, ec in zip(bp["boxes"], face_list, edge_list):
    patch.set_facecolor(fc)
    patch.set_edgecolor(ec)
    patch.set_linewidth(1.5)
    patch.set_alpha(0.9)

# ---------------------------------------------------------------
# Add N and mortality labels
# ---------------------------------------------------------------
for pos, vals, n, mr in zip(positions, plot_data, counts, mortality):
    if n > 0:
        ymax = np.nanmax(vals)
        plt.text(
            pos,
            ymax + 0.03,
            f"N={n}\n{mr:.1f}%",
            ha="center",
            fontsize=9
        )
    else:
        plt.text(pos, -0.15, "N=0\nNA", ha="center", fontsize=9)

# ---------------------------------------------------------------
# X-axis labels: A–DBD, D–DBD, A–DCD, D–DCD for each TX group
# ---------------------------------------------------------------
xticks = positions
xticklabels = subgroup_labels * 2   # repeat for second group
plt.xticks(xticks, xticklabels, fontsize=12)

# ---------------------------------------------------------------
# TX TYPE GROUP LABELS ("Single", "Bilateral")
# ---------------------------------------------------------------
for g in [0, 1]:
    center = g * group_spacing
    plt.text(center, -0.22, tx_groups[g],
             ha='center', va='top', fontsize=14)

# ---------------------------------------------------------------
# Axes and styling
# ---------------------------------------------------------------
plt.ylim(-0.2, 0.3)
plt.grid(axis='y', linestyle='--', alpha=0.4)

plt.ylabel("Sum of SHAP Values\n(Diagnosis Group + Donor Type + Transplantation Type)", fontsize=13)

# Legend
legend_handles = [
    Line2D([0], [0], marker='s', color='gray',
           markerfacecolor='white', markeredgecolor='gray',
           markersize=12, linestyle='None', label="DBD"),
    Line2D([0], [0], marker='s', color='darkgreen',
           markerfacecolor='green', markeredgecolor='darkgreen',
           markersize=12, linestyle='None', label="DCD")
]

plt.legend(handles=legend_handles, fontsize=12, loc='upper left')

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# ---------------------------------------------------------------
# Indices
# ---------------------------------------------------------------
grouping_index   = 25
donor_type_index = 43
vent_index       = 37
ecmo_index       = 44

# ---------------------------------------------------------------
# Pull arrays
# ---------------------------------------------------------------
X = np.array(x)
SV = np.array(shap_values)
Y = np.array(y)

diag  = X[:, imp_ordered_ind[grouping_index]].astype(int)
donor = X[:, imp_ordered_ind[donor_type_index]].astype(int)
vent  = X[:, imp_ordered_ind[vent_index]].astype(int)
ecmo  = X[:, imp_ordered_ind[ecmo_index]].astype(int)

# Support = YES if vent==1 OR ecmo==1
support = np.where((vent == 1) | (ecmo == 1), 1, 0)

# ---------------------------------------------------------------
# SHAP SUM
# ---------------------------------------------------------------
shap_sum = (
    SV[:, imp_ordered_ind[grouping_index]] +
    SV[:, imp_ordered_ind[donor_type_index]] +
    SV[:, imp_ordered_ind[vent_index]] +
    SV[:, imp_ordered_ind[ecmo_index]]
)

# ---------------------------------------------------------------
# Groups (A, D, A, D)
# ---------------------------------------------------------------
groups = [
    (2, 0),   # A–DBD
    (1, 0),   # D–DBD
    (2, 1),   # A–DCD
    (1, 1)    # D–DCD
]

group_labels = ["A", "D", "A", "D"]

# ---------------------------------------------------------------
# Colors
# ---------------------------------------------------------------
face_colors = {0: "white", 1: "green"}
edge_colors = {0: "gray",  1: "darkgreen"}

# ---------------------------------------------------------------
# Build plot data
# ---------------------------------------------------------------
data = []
positions = []
face_list = []
edge_list = []
n_list = []
mort_list = []

box_width = 0.35
support_spacing = 2.0     # distance between NO block and YES block
group_spacing   = 0.45    # spacing within each block

# Compute positions for 8 groups total
pos_counter = 0
for support_flag in [0, 1]:             # NO block, YES block
    for g_idx, (dx_val, d_val) in enumerate(groups):

        mask = (
            (diag == dx_val) &
            (donor == d_val) &
            (support == support_flag)
        )

        vals = shap_sum[mask]

        xpos = pos_counter * group_spacing + support_flag * support_spacing

        data.append(vals)
        positions.append(xpos)
        face_list.append(face_colors[d_val])
        edge_list.append(edge_colors[d_val])
        n_list.append(len(vals))
        mort_list.append(Y[mask].mean() * 100 if mask.any() else np.nan)

        pos_counter += 1

# ---------------------------------------------------------------
# FIGURE
# ---------------------------------------------------------------
plt.figure(figsize=(14, 7))

bp = plt.boxplot(
    data,
    positions=positions,
    widths=box_width,
    patch_artist=True,
    manage_ticks=False
)

for patch, fc, ec in zip(bp["boxes"], face_list, edge_list):
    patch.set_facecolor(fc)
    patch.set_edgecolor(ec)
    patch.set_linewidth(1.5)

# ---------------------------------------------------------------
# N and % labels
# ---------------------------------------------------------------
for pos, vals, n, mr in zip(positions, data, n_list, mort_list):
    if n > 0:
        plt.text(pos, np.nanmax(vals) + 0.02, f"N={n}\n{mr:.1f}%",
                 ha="center", fontsize=10)
    else:
        plt.text(pos, -0.20, "N=0\nNA", ha="center", fontsize=10)

# ---------------------------------------------------------------
# X-axis tick labels (A D A D A D A D)
# ---------------------------------------------------------------
plt.xticks(positions, group_labels * 2, fontsize=14)

# ---------------------------------------------------------------
# Add "No" and "Yes" centered under their 4-box blocks
# ---------------------------------------------------------------
mid_no  = np.mean(positions[0:4])
mid_yes = np.mean(positions[4:8])

plt.text(mid_no, -0.30, "No", ha="center", fontsize=16)
plt.text(mid_yes, -0.30, "Yes", ha="center", fontsize=16)

plt.ylabel("Sum of SHAP Values\n(Diagnosis group + Donor type + Ventilation + ECMO)", fontsize=14)
plt.ylim(-0.25, 0.45)
plt.grid(axis='y', linestyle='--', alpha=0.4)

# Legend
legend_handles = [
    Line2D([0], [0], marker='s', color='gray',
           markerfacecolor='white', markeredgecolor='gray',
           markersize=12, linestyle='None', label="DBD"),
    Line2D([0], [0], marker='s', color='darkgreen',
           markerfacecolor='green', markeredgecolor='darkgreen',
           markersize=12, linestyle='None', label="DCD"),
]
plt.legend(handles=legend_handles, fontsize=12, loc='upper left')

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# ---------------------------------------------------------------
# Indices
# ---------------------------------------------------------------
grouping_index   = 25   # diagnosis (A=2, D=1)
donor_type_index = 43
perfusion_index  = 46
vent_index       = 37

# ---------------------------------------------------------------
# Pull arrays
# ---------------------------------------------------------------
X = np.array(x)
SV = np.array(shap_values)
Y = np.array(y)

diag  = X[:, imp_ordered_ind[grouping_index]].astype(int)  # diagnosis group
donor = X[:, imp_ordered_ind[donor_type_index]].astype(int)

perf_raw = X[:, imp_ordered_ind[perfusion_index]].astype(int)
perf = np.where(perf_raw == -1, 0, perf_raw)

vent = X[:, imp_ordered_ind[vent_index]].astype(int)   

# ---------------------------------------------------------------
# SHAP SUM (diagnosis + donor + perfusion + type)
# ---------------------------------------------------------------
shap_sum = (
    SV[:, imp_ordered_ind[grouping_index]] +
    SV[:, imp_ordered_ind[donor_type_index]] +
    SV[:, imp_ordered_ind[perfusion_index]] +
    SV[:, imp_ordered_ind[vent_index]]
)

# ---------------------------------------------------------------
# Colors
# ---------------------------------------------------------------
color_tones = {
    (0,0): "#7F7F7F",
    (0,1): "#000000",
    (1,0): "#CCFF66",
    (1,1): "#00AA00"
}

# TX TYPE LABELS
vent_names = {0: "No", 1: "Yes"}

# ---------------------------------------------------------------
# FIGURE (ONLY 2 PANELS)
# ---------------------------------------------------------------
fig, (ax_perf, ax_noperf) = plt.subplots(1, 2, figsize=(26, 10))


# ===============================================================
# BOX PLOT FUNCTION — ONLY A AND D GROUPS
# ===============================================================
def make_group_boxplot(ax, perf_flag, title, y):

    # (Diagnosis, Donor)
    group_defs = [
        (2, 0),   # A – DBD
        (2, 1),   # A – DCD
        (1, 0),   # D – DBD
        (1, 1)    # D – DCD
    ]

    group_labels = ["A – DBD", "A – DCD", "D – DBD", "D – DCD"]
    base_positions = [0, 1.7, 3.4, 5.1]

    box_width = 0.22

    final_positions = []
    data = []
    colors_bp = []

    # -------------------------
    # Collect values for boxes
    # -------------------------
    for (dx_val, d_val), base_x in zip(group_defs, base_positions):
        for v_val in [0,1]:

            mask = (
                (diag == dx_val) &
                (donor == d_val) &
                (perf == perf_flag) &
                (vent == v_val)
            )

            vals = shap_sum[mask]
            xpos = base_x + v_val * box_width

            data.append(vals)
            final_positions.append(xpos)
            colors_bp.append(color_tones[(d_val, v_val)])

    # -------------------------
    # DRAW BOXES
    # -------------------------
    bp = ax.boxplot(
        data,
        positions=final_positions,
        widths=box_width,
        patch_artist=True,
        manage_ticks=False
    )

    for patch, col in zip(bp["boxes"], colors_bp):
        patch.set_facecolor(col)
        patch.set_edgecolor("k")
        patch.set_linewidth(1.3)

    # -------------------------
    # N and % labels
    # -------------------------
    idx = 0
    for (dx_val, d_val), base_x in zip(group_defs, base_positions):
        for v_val in [0,1]:

            xpos = base_x + v_val * box_width

            mask = (
                (diag == dx_val) &
                (donor == d_val) &
                (perf == perf_flag) &
                (vent == v_val)
            )

            vals = shap_sum[mask]

            if len(vals) == 0:
                ax.text(xpos, -0.25, "N=0\nNA", ha="center", fontsize=10)
            else:
                mort = y[mask].mean() * 100
                ypos = np.nanmax(vals) + 0.03
                ax.text(xpos, ypos, f"N={len(vals)}\n{mort:.1f}%",
                        ha="center", fontsize=10)

            idx += 1

    ax.set_ylim(-0.30, 0.62)

    # -------------------------
    # VENTILATION LABELS BELOW BOXES
    # -------------------------
    label_y = -0.33  # position below boxes

    for base_x in base_positions:
        for v_val in [0,1]:
            xpos = base_x + v_val * box_width
            ax.text(
                xpos, label_y,
                vent_names[v_val],   
                ha="center",
                va="top",
                rotation=90,
                fontsize=12
            )

    # -------------------------
    # X labels
    # -------------------------
    ax.set_xticks([p + 0.1 for p in base_positions])
    ax.set_xticklabels(group_labels, fontsize=14)

    ax.set_title(title, fontsize=18)
    ax.grid(axis="y", linestyle="--", alpha=0.4)


# ---------------------------------------------------------------
# PANELS
# ---------------------------------------------------------------
make_group_boxplot(ax_perf,    perf_flag=1, title="Perfusion",    y=Y)
make_group_boxplot(ax_noperf,  perf_flag=0, title="No Perfusion", y=Y)

plt.tight_layout()
plt.show()


In [ ]:
#era, box plots for isch time, allocation strategy, grouping, 
x_n=pd.DataFrame(x1)
#print(x_n)
for i in range(x_n.shape[1]):
    #print(x_n.iloc[:,i])
    print(feature_names[i])
    print(x_n.iloc[:,i].describe())
    q0=x_n.iloc[:,i].describe()[3]
    q1=x_n.iloc[:,i].describe()[4]
    q2=x_n.iloc[:,i].describe()[5]
    q3=x_n.iloc[:,i].describe()[6]
    x_n.iloc[:,i] = np.digitize(x_n.iloc[:,i], [q0,q1,q2,q3], right=True)
x_n=np.array(x_n)
print(x_n)

In [ ]:
sizes = x[:, imp_ordered_ind[f3]]
sizes[sizes == -1] = 0 
print(sizes[sizes==1]) 


In [ ]:
# Indices from X by DONOR_TYPE_BINARY (no if/else on column name)
mask_dbd = (x["DONOR_TYPE_BINARY"].to_numpy() == 0)
mask_dcd = (x["DONOR_TYPE_BINARY"].to_numpy() == 1)

shap_values_dbd=shap_values[mask_dbd]
shap_values_dcd=shap_values[mask_dcd]
x_dbd=x.loc[mask_dbd, :]
x_dcd=x.loc[mask_dcd, :]
x_n_dbd=x_n.loc[mask_dbd, :]
x_n_dcd=x_n.loc[mask_dcd, :]

In [ ]:
shap_values1 = pd.DataFrame(shap_values)
shap_values1.to_csv('shap_values.csv')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Feature indices
f1 = 2    # ISCHTIME
f2 = 43   # DONOR_TYPE_BINARY
f3 = 46   # PERFUSED_PRIOR

x = np.array(x)
print('Dependency Plots for', feature_names[imp_ordered_ind[f1]])
print()
i = 0

# Scatter plot showing the sum of 3 SHAP values on the y-axis
for f2_ in range(f2, f2+1):
    for f3_ in range(f3, f3+1):
        if ((f1 != f2_) and (f2_ != f3_) and (f1 != f3_)):
            plt.clf()
            plt.figure(figsize=(7, 8))

            # bubble size based on f3 feature values
            sizes = x[:, imp_ordered_ind[f3_]].copy()
            sizes[sizes == -1] = 0  # convert -1 to 0
            bubble_sizes = 30 * sizes + 5

            # Y-axis: sum of 3 SHAP values (f1 + f2 + f3)
            y_values = (
                shap_values[:, imp_ordered_ind[f1]] +
                shap_values[:, imp_ordered_ind[f2_]] +
                shap_values[:, imp_ordered_ind[f3_]]
            )

            plt.scatter(
                x[:, imp_ordered_ind[f1]], 
                y_values,
                s=bubble_sizes,
                c=x[:, imp_ordered_ind[f2_]],
                edgecolor='k',
                cmap=plt.cm.Greens,
                alpha=0.8
            )

            plt.ylabel('Sum of SHAP values (Isch time + Donor Type + Perfusion Prior)')
            plt.xlabel(feature_names[imp_ordered_ind[f1]])
            plt.colorbar().ax.set_ylabel(feature_names[imp_ordered_ind[f2_]])
            plt.ylim(y_values.min() - 0.1, y_values.max() + 0.1)

            print('bubble size:', feature_names[imp_ordered_ind[f3_]])
            plt.tight_layout()
            plt.savefig("dependency_isch.png", dpi=300)
            plt.show()
            i += 1

print('num of figures', i)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# Feature indices
ischemic_index = 2           # Ischemic Time
donor_type_index = 43        # DONOR_TYPE_BINARY
perfusion_index = 46         # PERFUSED_PRIOR

cutoff = 6.62
x = np.array(x)

# Color palette
colors = {
    (0, 0): '#1b9e77',  # DBD / No Perfusion
    (0, 1): '#7570b3',  # DBD / Perfusion
    (1, 0): '#d95f02',  # DCD / No Perfusion
    (1, 1): '#e7298a'   # DCD / Perfusion
}

categories = [(0, 0), (0, 1), (1, 0), (1, 1)]
labels = ['DBD / No Perfusion', 'DBD / Perfusion',
          'DCD / No Perfusion', 'DCD / Perfusion']

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# ---------------------------------------------------------------------------
# LEFT PANEL: Scatter plot
# ---------------------------------------------------------------------------
sizes = x[:, imp_ordered_ind[perfusion_index]].copy()
sizes[sizes == -1] = 0
bubble_sizes = 30 * sizes + 10  

y_values = (
    shap_values[:, imp_ordered_ind[ischemic_index]] +
    shap_values[:, imp_ordered_ind[donor_type_index]] +
    shap_values[:, imp_ordered_ind[perfusion_index]]
)

# Color: donor type (DCD=1 green, DBD=0 white)
colors_scatter = np.where(x[:, imp_ordered_ind[donor_type_index]] == 1, 'green', 'white')
edge_colors = np.where(x[:, imp_ordered_ind[donor_type_index]] == 1, 'darkgreen', 'gray')

axes[0].scatter(
    x[:, imp_ordered_ind[ischemic_index]],
    y_values,
    s=bubble_sizes,
    c=colors_scatter,
    edgecolor=edge_colors,
    alpha=0.8,
    linewidth=0.7
)

axes[0].set_xlabel('Ischemic Time (hours)')
axes[0].set_ylabel('Sum of SHAP Values (Ischemic Time + Donor Type + Perfusion Prior)')
axes[0].set_title('Sum of SHAP Values by Ischemic Time')
axes[0].grid(alpha=0.4)
axes[0].set_ylim(-0.2, 0.6)

# Add cutoff line
axes[0].axvline(cutoff, color='red', linestyle='--', linewidth=1)
axes[0].text(cutoff + 0.1, 0.55, 'Cutoff = 6.62', color='red', fontsize=9)

# Clean, left-aligned legend in bottom right
legend_elements = [
    Line2D([0], [0], color='none', label='Bubble color:', linestyle=''),
    Line2D([0], [0], marker='o', color='white', markerfacecolor='green',
           markeredgecolor='darkgreen', markersize=8, label='DCD (green)'),
    Line2D([0], [0], marker='o', color='white', markerfacecolor='white',
           markeredgecolor='gray', markersize=8, label='DBD (white)'),
    Line2D([0], [0], color='none', label='Bubble size:', linestyle=''),
    Line2D([0], [0], marker='o', color='white', markerfacecolor='lightgray',
           markeredgecolor='k', markersize=6, label='No Perfusion Prior / Unknown (small)'),
    Line2D([0], [0], marker='o', color='white', markerfacecolor='lightgray',
           markeredgecolor='k', markersize=12, label='Perfusion Prior (large)')
]
legend = axes[0].legend(
    handles=legend_elements,
    loc='lower right',
    fontsize=8,
    frameon=True,
    handletextpad=1.2,
    labelspacing=0.9,
    borderpad=1.0
)
for text in legend.get_texts():
    text.set_ha('left')

# ---------------------------------------------------------------------------
# RIGHT PANEL: Box plot
# ---------------------------------------------------------------------------
positions, plot_data, plot_colors = [], [], []
group_counts, mortality_rates = [], []

for j, (donor, perf) in enumerate(categories):
    for k, group_label in enumerate(['Ischemic Time < 6.62', 'Ischemic Time ≥ 6.62']):
        if k == 0:
            mask = (
                (x[:, imp_ordered_ind[ischemic_index]] < cutoff) &
                (x[:, imp_ordered_ind[donor_type_index]] == donor) &
                (x[:, imp_ordered_ind[perfusion_index]] == perf)
            )
        else:
            mask = (
                (x[:, imp_ordered_ind[ischemic_index]] >= cutoff) &
                (x[:, imp_ordered_ind[donor_type_index]] == donor) &
                (x[:, imp_ordered_ind[perfusion_index]] == perf)
            )

        shap_sum = (
            shap_values[mask, imp_ordered_ind[ischemic_index]] +
            shap_values[mask, imp_ordered_ind[donor_type_index]] +
            shap_values[mask, imp_ordered_ind[perfusion_index]]
        )

        plot_data.append(shap_sum)
        plot_colors.append(colors[(donor, perf)])
        n = len(shap_sum)
        group_counts.append(n)
        mort_rate = y[mask].mean() * 100 if n > 0 else np.nan
        mortality_rates.append(mort_rate)
        pos = j + (k * (len(categories) + 1))
        positions.append(pos)

bp = axes[1].boxplot(plot_data, positions=positions, patch_artist=True, widths=0.6)
for patch, color in zip(bp['boxes'], plot_colors * 2):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

# Add N and mortality above boxes
for pos, data, n, mort_rate in zip(positions, plot_data, group_counts, mortality_rates):
    if len(data) > 0:
        y_max = np.nanmax(data)
        text_y = min(0.58, y_max + 0.02)
        axes[1].text(pos, text_y, f'N={n}\n{mort_rate:.1f}%', ha='center', va='bottom',
                     fontsize=8, color='black')
    else:
        axes[1].text(pos, 0, 'N=0\nNA', ha='center', va='bottom', fontsize=8, color='gray')

xtick_positions = [1.5, 6.5]
axes[1].set_xticks(xtick_positions)
axes[1].set_xticklabels(['Ischemic Time < 6.62', 'Ischemic Time ≥ 6.62'], fontsize=10)
axes[1].set_xlim(-1, 9)
axes[1].set_ylim(-0.2, 0.6)
axes[1].set_ylabel('Sum of SHAP Values (Ischemic Time + Donor Type + Perfusion Prior)')
axes[1].set_title('Boxplots by Ischemic Time, Donor Type & Perfusion Prior')
axes[1].grid(axis='y', linestyle='--', alpha=0.6)

# Box legend + note inside plot under legend
handles = [Line2D([0], [0], color=color, lw=10) for color in colors.values()]
leg = axes[1].legend(handles, labels, loc='upper left', fontsize=8, frameon=True)

# Position the text *inside* the figure just below the legend
bbox = leg.get_window_extent(axes[1].figure.canvas.get_renderer())
bbox_data = bbox.transformed(axes[1].transData.inverted())
x_text = bbox_data.x0
y_text = bbox_data.y0 - 0.02
axes[1].text(x_text, y_text,
             'Text above each box shows N and 1-Year Mortality Rate',
             fontsize=8, color='black', ha='left', va='top')

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig("ischemic_time_scatter_box.png", dpi=300)
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# Feature indices
ischemic_index   = 2            # Ischemic Time
donor_type_index = 43           # DONOR_TYPE_BINARY
perfusion_index  = 46           # PERFUSED_PRIOR
tx_year_index    = 9            # TX Year (NEW)
era_index        = 38           # Era (NEW)

cutoff = 6.62
x = np.array(x)

# Color palette
colors = {
    (0, 0): '#1b9e77',  # DBD / No Perfusion
    (0, 1): '#7570b3',  # DBD / Perfusion
    (1, 0): '#d95f02',  # DCD / No Perfusion
    (1, 1): '#e7298a'   # DCD / Perfusion
}

categories = [(0, 0), (0, 1), (1, 0), (1, 1)]
labels = ['DBD / No Perfusion', 'DBD / Perfusion',
          'DCD / No Perfusion', 'DCD / Perfusion']

era_codes  = np.array([0, 1, 2])
era_labels = {0: 'LAS-DSA', 1: 'LAS-non-DSA', 2: 'CAS'}

# ----------------------------------------------------------------------------
# SHAP SUM with NEW features
# ----------------------------------------------------------------------------
y_shap_sum = (
    shap_values[:, imp_ordered_ind[ischemic_index]] +
    shap_values[:, imp_ordered_ind[donor_type_index]] +
    shap_values[:, imp_ordered_ind[perfusion_index]] +
    shap_values[:, imp_ordered_ind[tx_year_index]] +   
    shap_values[:, imp_ordered_ind[era_index]]          
)

# ----------------------------------------------------------------------------
# FIGURE
# ----------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# ----------------------------------------------------------------------------
# LEFT PANEL: Scatter
# ----------------------------------------------------------------------------
sizes = x[:, imp_ordered_ind[perfusion_index]].copy()
sizes[sizes == -1] = 0
bubble_sizes = 30 * sizes + 10

colors_scatter = np.where(
    x[:, imp_ordered_ind[donor_type_index]] == 1,
    'green', 'white'
)
edge_colors = np.where(
    x[:, imp_ordered_ind[donor_type_index]] == 1,
    'darkgreen', 'gray'
)

axes[0].scatter(
    x[:, imp_ordered_ind[ischemic_index]],
    y_shap_sum,
    s=bubble_sizes,
    c=colors_scatter,
    edgecolor=edge_colors,
    alpha=0.8,
    linewidth=0.7
)

axes[0].set_xlabel('Ischemic Time (hours)')
axes[0].set_ylabel('Sum of SHAP Values (Isch + Donor + Perf + TX_YEAR + ERA)')
axes[0].set_title('Sum of SHAP Values by Ischemic Time')
axes[0].grid(alpha=0.4)
axes[0].set_ylim(-0.2, 0.6)

axes[0].axvline(cutoff, color='red', linestyle='--', linewidth=1)
axes[0].text(cutoff + 0.1, 0.55, 'Cutoff = 6.62', color='red', fontsize=9)

legend_elements = [
    Line2D([0], [0], color='none', label='Bubble color:', linestyle=''),
    Line2D([0], [0], marker='o', color='white', markerfacecolor='green',
           markeredgecolor='darkgreen', markersize=8, label='DCD (green)'),
    Line2D([0], [0], marker='o', color='white', markerfacecolor='white',
           markeredgecolor='gray', markersize=8, label='DBD (white)'),
    Line2D([0], [0], color='none', label='Bubble size:', linestyle=''),
    Line2D([0], [0], marker='o', color='white', markerfacecolor='lightgray',
           markeredgecolor='k', markersize=6, label='No Perfusion Prior'),
    Line2D([0], [0], marker='o', color='white', markerfacecolor='lightgray',
           markeredgecolor='k', markersize=12, label='Perfusion Prior')
]

axes[0].legend(
    handles=legend_elements,
    loc='lower right',
    fontsize=8,
    frameon=True
)

# ----------------------------------------------------------------------------
# RIGHT PANEL: Boxplot
# ----------------------------------------------------------------------------
positions, plot_data, plot_colors = [], [], []
group_counts, mortality_rates = [], []

for j, (donor, perf) in enumerate(categories):
    for k, group_label in enumerate(['Ischemic Time < 6.62', 'Ischemic Time ≥ 6.62']):
        if k == 0:
            mask = (
                (x[:, imp_ordered_ind[ischemic_index]] < cutoff) &
                (x[:, imp_ordered_ind[donor_type_index]] == donor) &
                (x[:, imp_ordered_ind[perfusion_index]] == perf)
            )
        else:
            mask = (
                (x[:, imp_ordered_ind[ischemic_index]] >= cutoff) &
                (x[:, imp_ordered_ind[donor_type_index]] == donor) &
                (x[:, imp_ordered_ind[perfusion_index]] == perf)
            )

        shap_sum_group = y_shap_sum[mask]

        plot_data.append(shap_sum_group)
        plot_colors.append(colors[(donor, perf)])

        n = len(shap_sum_group)
        group_counts.append(n)
        mortality_rates.append(y[mask].mean() * 100 if n > 0 else np.nan)

        pos = j + (k * (len(categories) + 1))
        positions.append(pos)

bp = axes[1].boxplot(plot_data, positions=positions, patch_artist=True, widths=0.6)
for patch, color in zip(bp['boxes'], plot_colors * 2):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

axes[1].set_ylabel('Sum of SHAP (Isch + Donor + Perf + TX_YEAR + ERA)')
axes[1].set_title('Boxplots by Ischemic Time, Donor Type, Perfusion')
axes[1].set_ylim(-0.2, 0.6)

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import textwrap

# ---------------------------------------------------------------
# Indices
# ---------------------------------------------------------------
ischemic_index   = 2
donor_type_index = 43
perfusion_index  = 46
tx_year_index    = 9
era_index        = 38
cutoff = 6.62

x = np.array(x)

# ---------------------------------------------------------------
# SHAP SUM
# ---------------------------------------------------------------
shap_sum = (
    shap_values[:, imp_ordered_ind[ischemic_index]] +
    shap_values[:, imp_ordered_ind[donor_type_index]] +
    shap_values[:, imp_ordered_ind[perfusion_index]] +
    shap_values[:, imp_ordered_ind[tx_year_index]] +
    shap_values[:, imp_ordered_ind[era_index]]
)

donor  = x[:, imp_ordered_ind[donor_type_index]].astype(int)
perf   = x[:, imp_ordered_ind[perfusion_index]].astype(int)
perf   = np.where(perf == -1, 0, perf)
ischemia = x[:, imp_ordered_ind[ischemic_index]]
era = x[:, imp_ordered_ind[era_index]].astype(int)

bubble_sizes = 45 * perf + 15

# ---------------------------------------------------------------
# ✅ Color palette
# ---------------------------------------------------------------
color_tones = {
    (0,0): "#FFFFFF",
    (0,1): "#7F7F7F",
    (0,2): "#000000",
    (1,0): "#FFFFE6",
    (1,1): "#CCFF66",
    (1,2): "#00AA00"
}

colors = np.array([color_tones[(d,e)] for d,e in zip(donor, era)])
era_names = {0:"LAS-DSA", 1:"LAS-non-DSA", 2:"CAS"}

# Wrap ERA names for secondary labels
era_names_wrapped = {
    k : "\n".join(textwrap.wrap(v, width=6))
    for k,v in era_names.items()
}

# ---------------------------------------------------------------
# Figure layout 
# ---------------------------------------------------------------
fig = plt.figure(figsize=(36, 10))
gs = fig.add_gridspec(1, 3, width_ratios=[2.5, 1.25, 1.25], wspace=0.12)

ax_scatter = fig.add_subplot(gs[0,0])
ax_perf    = fig.add_subplot(gs[0,1])
ax_noperf  = fig.add_subplot(gs[0,2])

# ===============================================================
# ✅ SCATTER PLOT
# ===============================================================
for e_code in [0,1,2]:
    mask_e = (era == e_code)
    ax_scatter.scatter(
        ischemia[mask_e],
        shap_sum[mask_e],
        s=bubble_sizes[mask_e],
        c=colors[mask_e],
        edgecolor="k",
        alpha=0.85,
    )

ax_scatter.axvline(cutoff, color="red", linestyle="--", linewidth=1.5)
ax_scatter.text(cutoff+0.3, 0.55, "Cutoff = 6.62 h", color="red", fontsize=14)

ax_scatter.set_xlabel("Ischemic Time (hours)", fontsize=16)
ax_scatter.set_ylabel("Sum of SHAP Values", fontsize=16)
ax_scatter.set_ylim(-0.4, 0.6)   # ✅ updated
ax_scatter.grid(alpha=0.35)

# Legend (scatter)
legend_items = []
for (d,e), color in color_tones.items():
    donor_label = "DBD" if d==0 else "DCD"
    legend_items.append(Line2D([0],[0], marker='o', markersize=10,
                               markerfacecolor=color, markeredgecolor="k",
                               label=f"{donor_label} – {era_names[e]}"))

legend_items.append(Line2D([0],[0], marker='o', markersize=10,
                           markerfacecolor="white", markeredgecolor="k",
                           label="No Perfusion (small)"))
legend_items.append(Line2D([0],[0], marker='o', markersize=14,
                           markerfacecolor="white", markeredgecolor="k",
                           label="Perfusion (large)"))

ax_scatter.legend(handles=legend_items, loc="lower right", fontsize=13)

# ===============================================================
# BOX PLOT FUNCTION
# ===============================================================
def make_group_boxplot(ax, perf_flag, title, y):

    group_labels = ["DBD – Short", "DBD – Long", "DCD – Short", "DCD – Long"]
    base_positions = [0, 1.7, 3.6, 5.3]

    final_positions = []
    data = []
    colors_bp = []
    label_positions = []

    box_width = 0.22
    idx = 0

    for d in [0,1]:
        for ischemia_flag in ["short","long"]:
            base_x = base_positions[idx]
            idx += 1

            for e in [0,1,2]:

                if ischemia_flag=="short":
                    mask = (donor==d)&(era==e)&(perf==perf_flag)&(ischemia < cutoff)
                else:
                    mask = (donor==d)&(era==e)&(perf==perf_flag)&(ischemia >= cutoff)

                vals = shap_sum[mask]
                data.append(vals)

                xpos = base_x + e * box_width
                final_positions.append(xpos)
                colors_bp.append(color_tones[(d,e)])
                label_positions.append((mask, xpos, vals))


    # -------------------------------------------------
    # DRAW BOXES
    # -------------------------------------------------
    bp = ax.boxplot(
        data, positions=final_positions, widths=box_width,
        patch_artist=True, manage_ticks=False
    )

    for patch, c in zip(bp["boxes"], colors_bp):
        patch.set_facecolor(c)
        patch.set_edgecolor("k")
        patch.set_linewidth(1.3)
        patch.set_alpha(1.0)

    # red borders for long ischemia
    long_indices = [i for i in range(12) if (i//3 in [1,3])]
    for i in long_indices:
        bp["boxes"][i].set_edgecolor("red")
        bp["boxes"][i].set_linewidth(2.4)

    # -------------------------------------------------
    # NON-OVERLAPPING LABELS FOR N and %
    # -------------------------------------------------
    records = []
    x_offsets = {0: -0.04, 1: 0.0, 2: +0.04}  # ERA=0 sola, ERA=2 sağa minik kaydır

    idx = 0
    for g_idx, d in enumerate([0, 1]):               # 0: DBD, 1: DCD
        for is_idx, ischemia_flag in enumerate(["short", "long"]):
            base_x = base_positions[idx]
            idx += 1
            for e in [0, 1, 2]:
                if ischemia_flag == "short":
                    mask = (donor == d) & (era == e) & (perf == perf_flag) & (ischemia < cutoff)
                else:
                    mask = (donor == d) & (era == e) & (perf == perf_flag) & (ischemia >= cutoff)

                vals = shap_sum[mask]
                xpos = base_x + e * box_width + x_offsets[e]
                base_y = (np.nanmax(vals) if len(vals) else -0.35)
                records.append(( (g_idx*2 + is_idx), e, mask, xpos, vals, base_y))

    
    BASE_STEP = 0.012    
    MIN_GAP   = 0.030    
    MAX_ABOVE = 0.025    
    TOP_CAP   = 0.52     

    for group_id in range(4):  # 0: DBD-short, 1: DBD-long, 2: DCD-short, 3: DCD-long
        group_recs = [r for r in records if r[0] == group_id]
        # en yüksek kutudan başlayalım ki daha az yükseltelim
        group_recs.sort(key=lambda r: (r[5]), reverse=True)

        placed_ys = []
        for _, e, mask, xpos, vals, base_y in group_recs:
            if len(vals) == 0:
                label = "N=0\nNA"
                ypos = -0.32
            else:
                n = len(vals)
                mort = y[mask].mean() * 100
                ypos = base_y + MAX_ABOVE

                # grup içi çarpışma çözümü
                while any(abs(ypos - yy) < MIN_GAP for yy in placed_ys):
                    ypos += BASE_STEP
                    if ypos > TOP_CAP:
                        break

                label = f"N={n}\n{mort:.1f}%"
                placed_ys.append(ypos)

            ax.text(xpos, ypos, label, ha="center", fontsize=11)
        
    # -------------------------------------------------
    # ERA LABELS OUTSIDE PLOT 
    # -------------------------------------------------
    ax.set_ylim(-0.35, 0.62)

    era_y = -0.38
    for base_x in base_positions:
        for e_code in [0,1,2]:
            xpos = base_x + e_code * box_width
            ax.text(
                xpos, era_y, era_names[e_code],
                rotation=90,
                fontsize=11,
                ha="center", va="top"
            )


    # main group labels
    ax.set_xticks(base_positions)
    ax.set_xticklabels(group_labels, fontsize=13)

    ax.set_title(title, fontsize=18)
    ax.grid(axis="y", linestyle="--", alpha=0.4)
# ---------------------------------------------------------------
# PLOT PANELS
# ---------------------------------------------------------------
make_group_boxplot(ax_perf,    perf_flag=1, title="Perfusion",    y=y)
make_group_boxplot(ax_noperf,  perf_flag=0, title="No Perfusion", y=y)

plt.tight_layout()
plt.show()


In [ ]:
np.sort(np.unique(x[:, imp_ordered_ind[tx_year_index]].astype(int)))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import textwrap

# ---------------------------------------------------------------
# Indices
# ---------------------------------------------------------------
ischemic_index   = 2
donor_type_index = 43
perfusion_index  = 46
tx_year_index    = 9
era_index        = 38
cutoff = 6.62

x = np.array(x)

# ---------------------------------------------------------------
# SHAP SUM
# ---------------------------------------------------------------
shap_sum = (
    shap_values[:, imp_ordered_ind[ischemic_index]] +
    shap_values[:, imp_ordered_ind[donor_type_index]] +
    shap_values[:, imp_ordered_ind[perfusion_index]] +
    shap_values[:, imp_ordered_ind[tx_year_index]] +
    shap_values[:, imp_ordered_ind[era_index]]
)

donor  = x[:, imp_ordered_ind[donor_type_index]].astype(int)
perf   = x[:, imp_ordered_ind[perfusion_index]].astype(int)
perf   = np.where(perf == -1, 0, perf)
ischemia = x[:, imp_ordered_ind[ischemic_index]]
era = x[:, imp_ordered_ind[era_index]].astype(int)  
tx_year = x[:, imp_ordered_ind[tx_year_index]].astype(int)

# ===============================================================
# 🎨 ERA COLORS 
# ===============================================================
color_tones = {
    (0,0): "#FFFFFF",
    (0,1): "#7F7F7F",
    (0,2): "#000000",
    (1,0): "#FFFFE6",
    (1,1): "#CCFF66",
    (1,2): "#00AA00"
}

colors = np.array([color_tones[(d,e)] for d,e in zip(donor, era)])  

bubble_sizes = 45 * perf + 15

era_names = {0:"LAS-DSA", 1:"LAS-non-DSA", 2:"CAS"}

unique_years = np.sort(np.unique(tx_year))

# ---------------------------------------------------------------
# Figure layout
# ---------------------------------------------------------------
#fig = plt.figure(figsize=(20, 26))
fig = plt.figure(figsize=(max(24, 2 * len(unique_years)), 26))
gs = fig.add_gridspec(3, 1, height_ratios=[3.0, 2.0, 2.0], hspace=0.20)

ax_scatter = fig.add_subplot(gs[0, 0])
ax_perf    = fig.add_subplot(gs[1, 0])
ax_noperf  = fig.add_subplot(gs[2, 0])

# ===============================================================
# 🎯 SCATTER 
# ===============================================================
for e_code in [0,1,2]:
    mask_e = (era == e_code)
    ax_scatter.scatter(
        ischemia[mask_e],
        shap_sum[mask_e],
        s=bubble_sizes[mask_e],
        c=colors[mask_e],
        edgecolor="k",
        alpha=0.85,
    )

ax_scatter.axvline(cutoff, color="red", linestyle="--", linewidth=1.5)
ax_scatter.text(cutoff+0.3, 0.55, "Cutoff = 6.62 h", color="red", fontsize=14)

ax_scatter.set_xlabel("Ischemic Time (hours)", fontsize=16)
ax_scatter.set_ylabel("Sum of SHAP Values", fontsize=16)
ax_scatter.set_ylim(-0.4, 0.6)
ax_scatter.grid(alpha=0.35)

legend_items = []
for (d,e), color in color_tones.items():
    donor_label = "DBD" if d==0 else "DCD"
    legend_items.append(
        Line2D([0],[0], marker='o', markersize=10,
               markerfacecolor=color, markeredgecolor="k",
               label=f"{donor_label} – {era_names[e]}")
    )
legend_items.append(Line2D([0],[0], marker='o', markersize=10,
                           markerfacecolor="white", markeredgecolor="k",
                           label="No Perfusion (small)"))
legend_items.append(Line2D([0],[0], marker='o', markersize=14,
                           markerfacecolor="white", markeredgecolor="k",
                           label="Perfusion (large)"))
ax_scatter.legend(handles=legend_items, loc="lower right", fontsize=13)

# ===============================================================
# 📌 BOX-PLOT (EVERY YEAR SEPARATE, ERA COLOR)
# ===============================================================
def make_group_boxplot(ax, perf_flag, title, y):
    group_labels = ["DBD – Short", "DBD – Long", "DCD – Short", "DCD – Long"]
    base_positions = [0, 1.7, 3.6, 5.3]
    #box_width = 0.22
    
    data = []
    colors_bp = []
    positions = []

    n_years = len(unique_years)
    #offsets = np.linspace(-0.30, +0.30, n_years)  # side-by-side placement
    box_width = max(0.08, 0.30 / n_years)
    offsets = np.linspace(-0.5, +0.5, n_years)
    idx = 0
    for d in [0, 1]:
        for flag in ["short", "long"]:
            base_x = base_positions[idx]
            idx += 1

            for j, yr in enumerate(unique_years):
                if flag == "short":
                    mask = (donor==d)&(tx_year==yr)&(perf==perf_flag)&(ischemia < cutoff)
                else:
                    mask = (donor==d)&(tx_year==yr)&(perf==perf_flag)&(ischemia >= cutoff)

                vals = shap_sum[mask]
                data.append(vals)

                xpos = base_x + offsets[j]
                positions.append(xpos)

                era_vals = era[mask]
                era_mode = np.bincount(era_vals).argmax() if len(era_vals)>0 else 0
                colors_bp.append(color_tones[(d, era_mode)])

    bp = ax.boxplot(data, positions=positions, widths=box_width,
                    patch_artist=True, manage_ticks=False)

    for patch, c in zip(bp["boxes"], colors_bp):
        patch.set_facecolor(c)
        patch.set_edgecolor("k")
        patch.set_linewidth(1.2)
        patch.set_alpha(1.0)

    long_idxs = [i for i in range(len(data)) if (i // n_years) in [1,3]]
    for i in long_idxs:
        bp["boxes"][i].set_edgecolor("red")
        bp["boxes"][i].set_linewidth(2.5)

    # -------------------------------------------------
    # YEAR LABELS UNDER BOXES (vertical)
    # -------------------------------------------------
    for d in [0,1]:
        for flag_i, flag in enumerate(["short","long"]):
            base_x = base_positions[d*2 + flag_i]
            for j, yr in enumerate(unique_years):
                xpos = base_x + offsets[j]
                ax.text(
                    xpos, -0.45, str(yr),
                    rotation=90, fontsize=12, ha="center", va="top"
                )

    ax.set_ylim(-0.40, 0.62)
    ax.set_xticks(base_positions)
    ax.set_xticklabels(group_labels, fontsize=13)
    ax.set_title(title, fontsize=18)
    ax.grid(axis="y", linestyle="--", alpha=0.4)

# ---------------------------------------------------------------
# PLOT PANELS
# ---------------------------------------------------------------
make_group_boxplot(ax_perf,    perf_flag=1, title="Perfusion",    y=y)
make_group_boxplot(ax_noperf,  perf_flag=0, title="No Perfusion", y=y)

plt.tight_layout()
plt.show()


In [ ]:
shap_sum = (
    #shap_values[:, imp_ordered_ind[ischemic_index]] +
    shap_values[:, imp_ordered_ind[donor_type_index]] 
    #shap_values[:, imp_ordered_ind[perfusion_index]] +
    #shap_values[:, imp_ordered_ind[tx_year_index]] +
    #shap_values[:, imp_ordered_ind[era_index]]
)
print(shap_sum)

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu
import pingouin as pg

# =========================================================
# LABELS
# =========================================================
group_names = {
    (0,"short"): "DBD–Short",
    (0,"long"):  "DBD–Long",
    (1,"short"): "DCD–Short",
    (1,"long"):  "DCD–Long"
}

era_names = {0:"LAS-DSA", 1:"LAS-nonDSA", 2:"CAS"}

# =========================================================
# HELPERS
# =========================================================
def get_stats(vals):
    if len(vals)==0:
        return np.nan, np.nan, np.nan, "NA"
    med = np.nanmedian(vals)
    q1  = np.nanpercentile(vals, 25)
    q3  = np.nanpercentile(vals, 75)
    txt = f"{med:.3f} ({q1:.3f},{q3:.3f})"
    return med, q1, q3, txt

def asd_d(x, y):
    if len(x)==0 or len(y)==0:
        return np.nan
    return abs(pg.compute_effsize(x, y, eftype="cohen"))

def mw_p(x, y):
    if len(x)==0 or len(y)==0:
        return np.nan
    return mannwhitneyu(x, y, alternative="two-sided").pvalue

# =========================================================
# MAIN TABLE FUNCTION
# =========================================================
def build_era_table(perf_flag):

    rows = []

    for d in [0,1]:
        for ischemia_flag in ["short", "long"]:

            row = {}
            row["Group"] = f"{group_names[(d, ischemia_flag)]} – Perf={perf_flag}"

            era_vals = {}

            # Collect values for each era
            for e in [0,1,2]:
                if ischemia_flag=="short":
                    mask = (donor==d)&(era==e)&(perf==perf_flag)&(ischemia < cutoff)
                else:
                    mask = (donor==d)&(era==e)&(perf==perf_flag)&(ischemia >= cutoff)

                vals = shap_sum[mask]
                era_vals[e] = vals

                _,_,_,txt = get_stats(vals)
                row[era_names[e]] = txt

            # =================================================
            # ✅ PAIRWISE ASD (Cohen’s d)
            # =================================================
            row["ASD_LAS-nonDSA_vs_LAS-DSA"] = asd_d(era_vals[1], era_vals[0])
            row["ASD_CAS_vs_LAS-DSA"]        = asd_d(era_vals[2], era_vals[0])
            row["ASD_CAS_vs_LAS-nonDSA"]     = asd_d(era_vals[2], era_vals[1])

            # =================================================
            # ✅ PAIRWISE P-values (Mann–Whitney)
            # =================================================
            row["P_LAS-nonDSA_vs_LAS-DSA"] = mw_p(era_vals[1], era_vals[0])
            row["P_CAS_vs_LAS-DSA"]        = mw_p(era_vals[2], era_vals[0])
            row["P_CAS_vs_LAS-nonDSA"]     = mw_p(era_vals[2], era_vals[1])

            rows.append(row)

    return pd.DataFrame(rows)

# =========================================================
# BUILD TABLES
# =========================================================
table_perf   = build_era_table(perf_flag=1)
table_noperf = build_era_table(perf_flag=0)

table_perf.to_csv("SHAP_ERA_Table_Perfusion.csv", index=False, encoding="utf-8")
table_noperf.to_csv("SHAP_ERA_Table_NoPerfusion.csv", index=False, encoding="utf-8")


In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu
import pingouin as pg

# =========================================================
# HELPERS
# =========================================================
def get_stats(vals):
    if len(vals)==0:
        return np.nan, np.nan, np.nan, "NA"
    med = np.nanmedian(vals)
    q1  = np.nanpercentile(vals, 25)
    q3  = np.nanpercentile(vals, 75)
    txt = f"{med:.3f} ({q1:.3f},{q3:.3f})"
    return med, q1, q3, txt

def asd_d(x, y):
    if len(x)==0 or len(y)==0:
        return np.nan
    return abs(pg.compute_effsize(x, y, eftype="cohen"))

def mw_p(x, y):
    if len(x)==0 or len(y)==0:
        return np.nan
    return mannwhitneyu(x, y, alternative="two-sided").pvalue


# =========================================================
# ✅ GLOBAL ISCHEMIA TABLE
# =========================================================
def build_global_ischemia_table():

    rows = []

    # --- Short ischemia (<6.62)
    mask_short = ischemia < cutoff
    vals_short = shap_sum[mask_short]

    _,_,_,txt_short = get_stats(vals_short)

    rows.append({
        "Group": "Global – Short Ischemia (<6.62)",
        "Median (IQR)": txt_short
    })

    # --- Long ischemia (>=6.62)
    mask_long = ischemia >= cutoff
    vals_long = shap_sum[mask_long]

    _,_,_,txt_long = get_stats(vals_long)

    rows.append({
        "Group": "Global – Long Ischemia (>=6.62)",
        "Median (IQR)": txt_long
    })

    # --- Comparison (Cohen d + MW)
    d_value = asd_d(vals_short, vals_long)
    p_value = mw_p(vals_short, vals_long)

    rows.append({
        "Group": "Short vs Long Ischemia — Comparison",
        "Cohen_d": d_value,
        "MW_p_value": p_value
    })

    return pd.DataFrame(rows)


# =========================================================
# BUILD AND SAVE
# =========================================================
global_table = build_global_ischemia_table()

global_table.to_csv("SHAP_Global_Ischemia_Table.csv", index=False, encoding="utf-8")

print(global_table)
print("✅ Global ischemia table saved.")


In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu
import pingouin as pg

# =========================================================
# HELPERS
# =========================================================
def get_stats(vals):
    if len(vals)==0:
        return np.nan, np.nan, np.nan, "NA"
    med = np.nanmedian(vals)
    q1  = np.nanpercentile(vals, 25)
    q3  = np.nanpercentile(vals, 75)
    txt = f"{med:.3f} ({q1:.3f},{q3:.3f})"
    return med, q1, q3, txt

def asd_d(x, y):
    if len(x)==0 or len(y)==0:
        return np.nan
    return abs(pg.compute_effsize(x, y, eftype="cohen"))

def mw_p(x, y):
    if len(x)==0 or len(y)==0:
        return np.nan
    return mannwhitneyu(x, y, alternative="two-sided").pvalue


era_names = {0:"LAS-DSA", 1:"LAS-nonDSA", 2:"CAS"}

# =========================================================
# ✅ CRUDE ERA TABLE (NO donor/perfusion/ischemia stratification)
# =========================================================
def build_crude_era_stats():

    rows = []

    # Pull SHAP values by era
    era_vals = {e: shap_sum[era == e] for e in [0,1,2]}

    # === Era-specific stats ===
    for e in [0,1,2]:
        med, q1, q3, txt = get_stats(era_vals[e])
        rows.append({
            "Era": era_names[e],
            "Median (IQR)": txt
        })

    # === Pairwise comparisons ===
    rows.append({
        "Era": "LAS-nonDSA vs LAS-DSA",
        "Cohen_d": asd_d(era_vals[1], era_vals[0]),
        "MW_p_value": mw_p(era_vals[1], era_vals[0])
    })

    rows.append({
        "Era": "CAS vs LAS-DSA",
        "Cohen_d": asd_d(era_vals[2], era_vals[0]),
        "MW_p_value": mw_p(era_vals[2], era_vals[0])
    })

    rows.append({
        "Era": "CAS vs LAS-nonDSA",
        "Cohen_d": asd_d(era_vals[2], era_vals[1]),
        "MW_p_value": mw_p(era_vals[2], era_vals[1])
    })

    return pd.DataFrame(rows)

# =========================================================
# RUN & SAVE
# =========================================================
crude_era_table = build_crude_era_stats()
crude_era_table.to_csv("SHAP_Crude_Era_Statistics.csv", index=False, encoding="utf-8")

print(crude_era_table)
print("✅ Crude era summary table saved.")


In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu
import pingouin as pg

# =========================================================
# SETTINGS
# =========================================================
donor_shap = shap_values[:, imp_ordered_ind[donor_type_index]]  # ✅ Only donor-type SHAP values
donor_binary = donor.astype(int)  # 0=DBD, 1=DCD

# =========================================================
# HELPERS
# =========================================================
def get_stats(vals):
    if len(vals)==0:
        return "NA"
    med = np.nanmedian(vals)
    q1  = np.nanpercentile(vals, 25)
    q3  = np.nanpercentile(vals, 75)
    return f"{med:.3f} ({q1:.3f}, {q3:.3f})"

def asd_d(x, y):
    if len(x)==0 or len(y)==0:
        return np.nan
    return abs(pg.compute_effsize(x, y, eftype="cohen"))

def mw_p(x, y):
    if len(x)==0 or len(y)==0:
        return np.nan
    return mannwhitneyu(x, y, alternative="two-sided").pvalue

# =========================================================
# SPLIT GROUPS
# =========================================================
vals_DBD = donor_shap[donor_binary == 0]
vals_DCD = donor_shap[donor_binary == 1]

# =========================================================
# BUILD RESULT TABLE
# =========================================================
rows = [{
    "Group": "DBD",
    "SHAP Median (IQR)": get_stats(vals_DBD)
},
{
    "Group": "DCD",
    "SHAP Median (IQR)": get_stats(vals_DCD)
},
{
    "Group": "DCD vs DBD – Comparison",
    "Cohen_d": asd_d(vals_DCD, vals_DBD),
    "MW_p_value": mw_p(vals_DCD, vals_DBD)
}]

table_donor_shap = pd.DataFrame(rows)
print(table_donor_shap)

# Save
table_donor_shap.to_csv("SHAP_DonorType_Only.csv", index=False)


In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu
import pingouin as pg

# =========================================================
# PERFUSION SHAP VALUES
# =========================================================
perfusion_shap = shap_values[:, imp_ordered_ind[perfusion_index]]  # Only perfusion SHAP
perfusion_binary = perf.astype(int)  # 0 = No perfusion, 1 = Perfusion

# =========================================================
# HELPERS
# =========================================================
def get_stats(vals):
    if len(vals)==0:
        return "NA"
    med = np.nanmedian(vals)
    q1  = np.nanpercentile(vals, 25)
    q3  = np.nanpercentile(vals, 75)
    return f"{med:.3f} ({q1:.3f}, {q3:.3f})"

def asd_d(x, y):
    if len(x)==0 or len(y)==0:
        return np.nan
    return abs(pg.compute_effsize(x, y, eftype="cohen"))

def mw_p(x, y):
    if len(x)==0 or len(y)==0:
        return np.nan
    return mannwhitneyu(x, y, alternative="two-sided").pvalue


# =========================================================
# SPLIT GROUPS
# =========================================================
vals_NoPerf = perfusion_shap[perfusion_binary == 0]
vals_Perf   = perfusion_shap[perfusion_binary == 1]

# =========================================================
# BUILD RESULT TABLE
# =========================================================
rows = [
    {
        "Group": "No Perfusion (0)",
        "SHAP Median (IQR)": get_stats(vals_NoPerf)
    },
    {
        "Group": "Perfusion (1)",
        "SHAP Median (IQR)": get_stats(vals_Perf)
    },
    {
        "Group": "Perfusion vs No Perfusion — Comparison",
        "Cohen_d": asd_d(vals_Perf, vals_NoPerf),
        "MW_p_value": mw_p(vals_Perf, vals_NoPerf)
    }
]

table_perf_shap = pd.DataFrame(rows)
print(table_perf_shap)

# Save CSV
table_perf_shap.to_csv("SHAP_Perfusion_Only.csv", index=False)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# ---------------------------
# Indices
# ---------------------------
tx_year_index     = 9
era_index         = 38
donor_type_index  = 43
perfusion_index   = 46

# ---------------------------
# Pull arrays
# ---------------------------
X = np.array(x)
SV = np.array(shap_values)
y_mort = np.array(y)

def col(idx): return X[:, imp_ordered_ind[idx]]
def shv(idx): return SV[:, imp_ordered_ind[idx]]

# ---------------------------
# Clean variables
# ---------------------------
era = np.rint(col(era_index)).astype(float)
era[~np.isin(era, [0, 1, 2])] = np.nan

donor = np.nan_to_num(col(donor_type_index), nan=0.0)
donor[donor < 0] = 0
donor = (donor > 0.5).astype(int)

perf = np.nan_to_num(col(perfusion_index), nan=0.0)
perf[perf < 0] = 0
perf = (perf > 0.5).astype(int)

# ---------------------------
# Constants
# ---------------------------
era_codes  = np.array([0, 1, 2])
era_labels = {0: 'LAS-DSA', 1: 'LAS-non-DSA', 2: 'CAS'}

colors_box = {
    (0, 0): '#1b9e77',  # DBD / No Perfusion
    (0, 1): '#7570b3',  # DBD / Perfusion
    (1, 0): '#d95f02',  # DCD / No Perfusion
    (1, 1): '#e7298a'   # DCD / Perfusion
}
group_keys   = [(0,0),(0,1),(1,0),(1,1)]
group_labels = ['DBD / No Perfusion', 'DBD / Perfusion',
                'DCD / No Perfusion', 'DCD / Perfusion']

shap_sum4 = (shv(tx_year_index) +
             shv(era_index) +
             shv(donor_type_index) +
             shv(perfusion_index))

# ---------------------------
# Create figure
# ---------------------------
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# ==============================================================
# LEFT: Scatter
# ==============================================================
rng = np.random.default_rng(42)
era_valid = ~np.isnan(era)
x_scatter = era + (rng.random(size=era.shape) - 0.5) * 0.25

bubble_sizes = 30 * perf + 10
colors_scatter = np.where(donor == 1, 'green', 'white')
edge_colors    = np.where(donor == 1, 'darkgreen', 'gray')

axes[0].scatter(
    x_scatter[era_valid],
    shap_sum4[era_valid],
    s=bubble_sizes[era_valid],
    c=colors_scatter[era_valid],
    edgecolor=edge_colors[era_valid],
    alpha=0.82, linewidth=0.7
)

#axes[0].set_xlabel('ERA (0=LAS-DSA, 1=LAS-non-DSA, 2=CAS)')
axes[0].set_ylabel('Sum of SHAP Values (TX_YEAR + ERA + Donor Type + Perfusion Prior)')
axes[0].set_title('Sum of SHAP Values by ERA')
axes[0].grid(alpha=0.4)
axes[0].set_xlim(-0.5, 2.5)
axes[0].set_ylim(-0.25, 0.45)
axes[0].set_xticks(era_codes)
axes[0].set_xticklabels([era_labels[c] for c in era_codes])

# Legend (top-left)
scatter_leg = [
    Line2D([0], [0], color='none', label='Bubble color:', linestyle=''),
    Line2D([0], [0], marker='o', color='white', markerfacecolor='green',
           markeredgecolor='darkgreen', markersize=8, label='DCD (green)'),
    Line2D([0], [0], marker='o', color='white', markerfacecolor='white',
           markeredgecolor='gray', markersize=8, label='DBD (white)'),
    Line2D([0], [0], color='none', label='Bubble size:', linestyle=''),
    Line2D([0], [0], marker='o', color='white', markerfacecolor='lightgray',
           markeredgecolor='k', markersize=6,  label='No Perfusion Prior (small)'),
    Line2D([0], [0], marker='o', color='white', markerfacecolor='lightgray',
           markeredgecolor='k', markersize=12, label='Perfusion Prior (large)'),
]
leg0 = axes[0].legend(handles=scatter_leg, loc='upper left',
                      fontsize=8, frameon=True, handletextpad=1.2,
                      labelspacing=0.9, borderpad=1.0)
for t in leg0.get_texts():
    t.set_ha('left')

# ==============================================================
# RIGHT: Boxplots
# ==============================================================
positions, plot_data, plot_colors, Ns, mort_rates = [], [], [], [], []
offsets = [-1.5, -0.5, 0.5, 1.5]

for i, ecode in enumerate(era_codes):
    base = i * 5
    for (d, p), off in zip(group_keys, offsets):
        mask = (era == ecode) & (donor == d) & (perf == p)
        vals = shap_sum4[mask]
        if len(vals) == 0:
            vals = np.array([np.nan])
        positions.append(base + off)
        plot_data.append(vals)
        plot_colors.append(colors_box[(d, p)])
        Ns.append(int(mask.sum()))
        mort_rates.append((y_mort[mask].mean() * 100) if mask.any() else np.nan)

bp = axes[1].boxplot(plot_data, positions=positions, patch_artist=True, widths=0.8)
for patch, color in zip(bp['boxes'], plot_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

# Add labels
for pos, vals, n, mr in zip(positions, plot_data, Ns, mort_rates):
    if np.all(np.isnan(vals)):
        axes[1].text(pos, -0.25, 'N=0\nNA', ha='center', va='bottom', fontsize=8, color='gray')
    else:
        y_max = np.nanmax(vals)
        axes[1].text(pos, y_max + 0.05, f'N={n}\n{mr:.1f}%',
                     ha='center', va='bottom', fontsize=8, color='black')

axes[1].set_xlim(-2.5, (len(era_codes) - 1) * 5 + 2.5)
axes[1].set_ylim(-0.25, 0.45)
axes[1].set_ylabel('Sum of SHAP Values (TX_YEAR + ERA + Donor Type + Perfusion Prior)')
axes[1].set_title('Boxplots by ERA, Donor Type & Perfusion Prior')
axes[1].grid(axis='y', linestyle='--', alpha=0.6)
axes[1].set_xticks([i * 5 for i in range(len(era_codes))])
axes[1].set_xticklabels([era_labels[c] for c in era_codes])

# Legend (top-left)
handles = [Line2D([0], [0], color=colors_box[k], lw=10) for k in group_keys]
leg1 = axes[1].legend(handles=handles, labels=group_labels,
                      loc='upper left', fontsize=8, frameon=True)
axes[1].text(0.02, 0.88,
             'Text above each box shows N and 1-Year Mortality Rate',
             transform=axes[1].transAxes, fontsize=8, color='black',
             ha='left', va='top')

plt.tight_layout()
plt.savefig("era_scatter_box.png", dpi=300)
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# ---------------------------
# Indices
# ---------------------------
tx_year_index     = 9
era_index         = 38
donor_type_index  = 43
perfusion_index   = 46

# ---------------------------
# Pull arrays
# ---------------------------
X = np.array(x)
SV = np.array(shap_values)
y_mort = np.array(y)

def col(idx): return X[:, imp_ordered_ind[idx]]
def shv(idx): return SV[:, imp_ordered_ind[idx]]

# ---------------------------
# Extract variables
# ---------------------------
tx_year = np.rint(col(tx_year_index)).astype(float)
era = np.rint(col(era_index)).astype(float)
era[~np.isin(era, [0, 1, 2])] = np.nan

donor = np.nan_to_num(col(donor_type_index), nan=0.0)
donor[donor < 0] = 0
donor = (donor > 0.5).astype(int)

perf = np.nan_to_num(col(perfusion_index), nan=0.0)
perf[perf < 0] = 0
perf = (perf > 0.5).astype(int)

# ---------------------------
# Constants
# ---------------------------
colors_box = {
    (0, 0): '#1b9e77',  # DBD / No Perfusion
    (0, 1): '#7570b3',  # DBD / Perfusion
    (1, 0): '#d95f02',  # DCD / No Perfusion
    (1, 1): '#e7298a'   # DCD / Perfusion
}
group_keys   = [(0,0),(0,1),(1,0),(1,1)]
group_labels = ['DBD / No Perfusion', 'DBD / Perfusion',
                'DCD / No Perfusion', 'DCD / Perfusion']

# SHAP sum
shap_sum4 = (shv(tx_year_index) +
             shv(era_index) +
             shv(donor_type_index) +
             shv(perfusion_index))

# ---------------------------
# Unique years
# ---------------------------
years = np.unique(tx_year[~np.isnan(tx_year)])
years = np.sort(years)

# ---------------------------
# Create figure
# ---------------------------
fig, (ax_scatter, ax_box) = plt.subplots(
    1, 2,
    figsize=(30, 10),
    gridspec_kw={'width_ratios': [1, 2.7]}  # scatter : box = 1 : 2
)
# ==============================================================
# LEFT: Scatter Plot (spread like ERA)
# ==============================================================
rng = np.random.default_rng(42)
tx_year_valid = ~np.isnan(tx_year)
x_jittered = tx_year.copy()
x_jittered[tx_year_valid] += (rng.random(sum(tx_year_valid)) - 0.5) * 0.4  # horizontal jitter

bubble_sizes = 30 * perf + 10
colors_scatter = np.where(donor == 1, 'green', 'white')
edge_colors    = np.where(donor == 1, 'darkgreen', 'gray')

ax_scatter.scatter(
    x_jittered,
    shap_sum4,
    s=bubble_sizes,
    c=colors_scatter,
    edgecolor=edge_colors,
    alpha=0.82, linewidth=0.7
)

ax_scatter.set_xlabel('TX_YEAR')
ax_scatter.set_ylabel('Sum of SHAP Values (TX_YEAR + ERA + Donor Type + Perfusion Prior)')
ax_scatter.set_title('Sum of SHAP Values by TX_YEAR')
ax_scatter.grid(alpha=0.4)
ax_scatter.set_ylim(-0.25, 0.45)

# ✅ Fixed legend
handles = [
    Line2D([0], [0], color='none', linestyle='', label='Bubble color:'),
    Line2D([0], [0], marker='o', color='white',
           markerfacecolor='green', markeredgecolor='darkgreen', markersize=6,
           label='DCD (green)'),
    Line2D([0], [0], marker='o', color='white',
           markerfacecolor='white', markeredgecolor='gray', markersize=6,
           label='DBD (white)'),
    Line2D([0], [0], color='none', linestyle='', label='Bubble size:'),
    Line2D([0], [0], marker='o', color='white',
           markerfacecolor='lightgray', markeredgecolor='k', markersize=4,
           label='No Perfusion Prior (small)'),
    Line2D([0], [0], marker='o', color='white',
           markerfacecolor='lightgray', markeredgecolor='k', markersize=8,
           label='Perfusion Prior (large)')
]

ax_scatter.legend(handles=handles, loc='upper left',
                  fontsize=9, frameon=True, handletextpad=1.2,
                  labelspacing=0.9, borderpad=1.0)
# ==============================================================
# RIGHT: Boxplots (consistent spacing + staggered text)
# ==============================================================
positions, plot_data, plot_colors, Ns, mort_rates = [], [], [], [], []
offsets = [-2.0, -0.7, 0.7, 2.0]   # spacing inside year group
spacing = 7                        # spacing between years

for i, year in enumerate(years):
    base = i * spacing
    for (d, p), off in zip(group_keys, offsets):
        mask = (tx_year == year) & (donor == d) & (perf == p)
        vals = shap_sum4[mask]
        if len(vals) == 0:
            vals = np.array([np.nan])
        positions.append(base + off)
        plot_data.append(vals)
        plot_colors.append(colors_box[(d, p)])
        Ns.append(int(mask.sum()))
        mort_rates.append((y_mort[mask].mean() * 100) if mask.any() else np.nan)

bp = ax_box.boxplot(plot_data, positions=positions, patch_artist=True, widths=1.0)
for patch, color in zip(bp['boxes'], plot_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

# Add staggered labels
for j, (pos, vals, n, mr) in enumerate(zip(positions, plot_data, Ns, mort_rates)):
    if np.all(np.isnan(vals)):
        ax_box.text(pos, -0.25, 'N=0\nNA', ha='center', va='bottom', fontsize=8, color='gray')
    else:
        y_max = np.nanmax(vals)
        ax_box.text(pos, y_max + (0.05 if j % 2 == 0 else 0.09),
                    f'N={n}\n{mr:.1f}%', ha='center', va='bottom',
                    fontsize=8, color='black')

ax_box.set_xlim(-3, (len(years) - 1) * spacing + 4)
ax_box.set_ylim(-0.25, 0.45)
ax_box.set_ylabel('Sum of SHAP Values (TX_YEAR + ERA + Donor Type + Perfusion Prior)')
ax_box.set_title('Boxplots by TX_YEAR, Donor Type & Perfusion Prior')
ax_box.grid(axis='y', linestyle='--', alpha=0.6)
ax_box.set_xticks([i * spacing for i in range(len(years))])
ax_box.set_xticklabels([int(y) for y in years], rotation=45, fontsize=9)

# Legend
handles = [Line2D([0], [0], color=colors_box[k], lw=10) for k in group_keys]
ax_box.legend(handles=handles, labels=group_labels,
              loc='upper left', fontsize=9, frameon=True)
ax_box.text(0.02, 0.88,
            'Text above each box shows N and 1-Year Mortality Rate',
            transform=ax_box.transAxes, fontsize=8, color='black',
            ha='left', va='top')

plt.tight_layout()
plt.savefig("txyear_scatter_box.png", dpi=300)
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# ---------------------------
# Indices
# ---------------------------
grouping_index    = 25
donor_type_index  = 43
perfusion_index   = 46

# ---------------------------
# Pull arrays
# ---------------------------
X = np.array(x)
SV = np.array(shap_values)
y_mort = np.array(y)

def col(idx): return X[:, imp_ordered_ind[idx]]
def shv(idx): return SV[:, imp_ordered_ind[idx]]

# ---------------------------
# Extract variables
# ---------------------------
grouping = np.rint(col(grouping_index)).astype(float)
donor = (np.nan_to_num(col(donor_type_index), nan=0.0) > 0.5).astype(int)
perf = (np.nan_to_num(col(perfusion_index), nan=0.0) > 0.5).astype(int)

# ---------------------------
# Group mapping
# ---------------------------
# Data: 0→B, 1→D, 2→A, 3→C
group_labels = {0: 'B', 1: 'D', 2: 'A', 3: 'C'}
group_order = [2, 0, 3, 1]  # visual order: A, B, C, D

colors_box = {
    (0, 0): '#1b9e77',  # DBD / No Perfusion
    (0, 1): '#7570b3',  # DBD / Perfusion
    (1, 0): '#d95f02',  # DCD / No Perfusion
    (1, 1): '#e7298a'   # DCD / Perfusion
}
group_keys = [(0,0),(0,1),(1,0),(1,1)]
group_labels_legend = ['DBD / No Perfusion', 'DBD / Perfusion',
                       'DCD / No Perfusion', 'DCD / Perfusion']

# ---------------------------
# SHAP sum
# ---------------------------
shap_sum_grp = (shv(grouping_index) +
                shv(donor_type_index) +
                shv(perfusion_index))

# ---------------------------
# Create figure
# ---------------------------
fig, (ax_scatter, ax_box) = plt.subplots(
    1, 2, figsize=(20, 8), gridspec_kw={'width_ratios': [1, 1]}
)
plt.subplots_adjust(wspace=0.3)

# ==============================================================
# LEFT: Scatter Plot (fixed A–B–C–D order)
# ==============================================================

# Map grouping values to positions for correct visual order
group_to_pos = {2: 0, 0: 1, 3: 2, 1: 3}  # A,B,C,D order
x_positions = np.array([group_to_pos.get(g, np.nan) for g in grouping], dtype=float)

rng = np.random.default_rng(42)
valid = ~np.isnan(x_positions)
x_jittered = x_positions.astype(float)  # ensure float
x_jittered[valid] += (rng.random(sum(valid)) - 0.5) * 0.25  # small jitter

bubble_sizes = 30 * perf + 10
colors_scatter = np.where(donor == 1, 'green', 'white')
edge_colors    = np.where(donor == 1, 'darkgreen', 'gray')

ax_scatter.scatter(
    x_jittered[valid],
    shap_sum_grp[valid],
    s=bubble_sizes[valid],
    c=colors_scatter[valid],
    edgecolor=edge_colors[valid],
    alpha=0.82, linewidth=0.7
)

ax_scatter.set_xlabel('GROUPING')
ax_scatter.set_ylabel('Sum of SHAP Values (GROUPING + Donor Type + Perfusion Prior)')
ax_scatter.set_title('Sum of SHAP Values by GROUPING')
ax_scatter.grid(alpha=0.4)
ax_scatter.set_xlim(-0.5, 3.5)
ax_scatter.set_ylim(-0.1, 0.5)
ax_scatter.set_xticks(range(4))
ax_scatter.set_xticklabels(['A', 'B', 'C', 'D'])

# Legend
handles = [
    Line2D([0], [0], color='none', linestyle='', label='Bubble color:'),
    Line2D([0], [0], marker='o', color='white',
           markerfacecolor='green', markeredgecolor='darkgreen', markersize=6,
           label='DCD (green)'),
    Line2D([0], [0], marker='o', color='white',
           markerfacecolor='white', markeredgecolor='gray', markersize=6,
           label='DBD (white)'),
    Line2D([0], [0], color='none', linestyle='', label='Bubble size:'),
    Line2D([0], [0], marker='o', color='white',
           markerfacecolor='lightgray', markeredgecolor='k', markersize=4,
           label='No Perfusion Prior (small)'),
    Line2D([0], [0], marker='o', color='white',
           markerfacecolor='lightgray', markeredgecolor='k', markersize=8,
           label='Perfusion Prior (large)')
]

ax_scatter.legend(handles=handles, loc='upper left',
                  fontsize=9, frameon=True, handletextpad=1.2,
                  labelspacing=0.9, borderpad=1.0)

# ==============================================================
# RIGHT: Boxplots (aligned with A–B–C–D order)
# ==============================================================
positions, plot_data, plot_colors, Ns, mort_rates = [], [], [], [], []
offsets = [-1.5, -0.5, 0.5, 1.5]

for i, g in enumerate(group_order):  # A,B,C,D order visually
    base = i * 5
    for (d, p), off in zip(group_keys, offsets):
        mask = (grouping == g) & (donor == d) & (perf == p)
        vals = shap_sum_grp[mask]
        if len(vals) == 0:
            vals = np.array([np.nan])
        positions.append(base + off)
        plot_data.append(vals)
        plot_colors.append(colors_box[(d, p)])
        Ns.append(int(mask.sum()))
        mort_rates.append((y_mort[mask].mean() * 100) if mask.any() else np.nan)

bp = ax_box.boxplot(plot_data, positions=positions, patch_artist=True, widths=0.8)
for patch, color in zip(bp['boxes'], plot_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

# Labels (N and mortality above each box)
for pos, vals, n, mr in zip(positions, plot_data, Ns, mort_rates):
    if np.all(np.isnan(vals)):
        ax_box.text(pos, -0.08, 'N=0\nNA', ha='center', va='bottom', fontsize=8, color='gray')
    else:
        y_max = np.nanmax(vals)
        ax_box.text(pos, y_max + 0.03, f'N={n}\n{mr:.1f}%',
                     ha='center', va='bottom', fontsize=8, color='black')

ax_box.set_xlim(-2.5, (len(group_order) - 1) * 5 + 2.5)
ax_box.set_ylim(-0.1, 0.5)
ax_box.set_ylabel('Sum of SHAP Values (GROUPING + Donor Type + Perfusion Prior)')
ax_box.set_title('Boxplots by GROUPING, Donor Type & Perfusion Prior')
ax_box.grid(axis='y', linestyle='--', alpha=0.6)
ax_box.set_xticks([i * 5 for i in range(len(group_order))])
ax_box.set_xticklabels(['A', 'B', 'C', 'D'])

# Legend
handles = [Line2D([0], [0], color=colors_box[k], lw=10) for k in group_keys]
ax_box.legend(handles=handles, labels=group_labels_legend,
              loc='upper left', fontsize=9, frameon=True)
ax_box.text(0.02, 0.86,
            'Text above each box shows N and 1-Year Mortality Rate',
            transform=ax_box.transAxes, fontsize=8, color='black',
            ha='left', va='top')

plt.tight_layout()
plt.savefig("grouping_scatter_box.png", dpi=300)
plt.show()


In [ ]:
# 3D SHAP dependency plots for f1=1
f1=9
x=np.array(x)
print('Dependency Plots for', feature_names[imp_ordered_ind[f1]])
print()
i=0
for f2 in range(43,44):
    for f3 in range(46,47):
        if ((f1 != f2) and (f2!= f3) and (f1 != f3)):
            plt.clf()
            plt.figure(figsize=(7,8))
            sizes = x[:, imp_ordered_ind[f3]]
            sizes[sizes == -1] = 0  
            plt.scatter(x[:,imp_ordered_ind[f1]], shap_values[:,imp_ordered_ind[f1]], s = 30 * sizes + 5,  c = x[:,imp_ordered_ind[f2]], edgecolor = 'k', cmap=plt.cm.Greens, alpha=0.8)
            plt.ylabel('SHAP values')
            plt.ylim(-0.2,0.3)
            #plt.xticks(np.arange(0,101,20))
            plt.xlabel(feature_names[imp_ordered_ind[f1]])
            plt.colorbar().ax.set_ylabel(feature_names[imp_ordered_ind[f2]])
            print('bubble size: ', feature_names[imp_ordered_ind[f3]])
            plt.savefig("dependency_year.png",dpi=300)
            plt.show()
            i+=1
print('num of figures',i)

In [ ]:
# 3D SHAP dependency plots for f1=1
f1=2
x=np.array(x)
print('Dependency Plots for', feature_names[imp_ordered_ind[f1]])
print()
i=0
for f2 in range(43,44):
    for f3 in range(46,47):
        if ((f1 != f2) and (f2!= f3) and (f1 != f3)):
            plt.clf()
            plt.figure(figsize=(7,8))
            sizes = x[:, imp_ordered_ind[f3]]
            sizes[sizes == -1] = 0  
            plt.scatter(x[:,imp_ordered_ind[f1]], shap_values[:,imp_ordered_ind[f1]], s = 30 * sizes + 5,  c = x[:,imp_ordered_ind[f2]], edgecolor = 'k', cmap=plt.cm.Greens, alpha=0.8)
            plt.ylabel('SHAP values')
            plt.ylim(-0.2,0.3)
            #plt.xticks(np.arange(0,101,20))
            plt.xlabel(feature_names[imp_ordered_ind[f1]])
            plt.colorbar().ax.set_ylabel(feature_names[imp_ordered_ind[f2]])
            print('bubble size: ', feature_names[imp_ordered_ind[f3]])
            plt.savefig("dependency_year.png",dpi=300)
            plt.show()
            i+=1
print('num of figures',i)

In [ ]:
# 3D SHAP dependency plots for f1=1
f1=2
x=np.array(x)
print('Dependency Plots for', feature_names[imp_ordered_ind[f1]])
print()
i=0
for f2 in range(43,44):
    for f3 in range(46,47):
        if ((f1 != f2) and (f2!= f3) and (f1 != f3)):
            plt.clf()
            plt.figure(figsize=(7,8))
            sizes = x[:, imp_ordered_ind[f3]]
            sizes[sizes == -1] = 0  
            plt.scatter(x[:,imp_ordered_ind[f1]], shap_values[:,imp_ordered_ind[f1]], s = 30 * sizes + 5,  c = x[:,imp_ordered_ind[f2]], edgecolor = 'k', cmap=plt.cm.Greens, alpha=0.8)
            plt.ylabel('SHAP values')
            plt.ylim(-0.2,0.3)
            #plt.xticks(np.arange(0,101,20))
            plt.xlabel(feature_names[imp_ordered_ind[f1]])
            plt.colorbar().ax.set_ylabel(feature_names[imp_ordered_ind[f2]])
            print('bubble size: ', feature_names[imp_ordered_ind[f3]])
            plt.savefig("dependency_year.png",dpi=300)
            plt.show()
            i+=1
print('num of figures',i)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Feature indices
f1 = 9    # TX_YEAR
f2 = 43   # DONOR_TYPE_BINARY
f3 = 46   # PERFUSED_PRIOR

x = np.array(x)
print('Dependency Plots for', feature_names[imp_ordered_ind[f1]])
print()
i = 0

# Scatter plot showing the sum of 3 SHAP values on the y-axis
for f2_ in range(f2, f2+1):
    for f3_ in range(f3, f3+1):
        if ((f1 != f2_) and (f2_ != f3_) and (f1 != f3_)):
            plt.clf()
            plt.figure(figsize=(7, 8))

            # bubble size based on f3 feature values
            sizes = x[:, imp_ordered_ind[f3_]].copy()
            sizes[sizes == -1] = 0  # convert -1 to 0
            bubble_sizes = 30 * sizes + 5

            # Y-axis: sum of 3 SHAP values (f1 + f2 + f3)
            y_values = (
                shap_values[:, imp_ordered_ind[f1]] +
                shap_values[:, imp_ordered_ind[f2_]] +
                shap_values[:, imp_ordered_ind[f3_]]
            )

            plt.scatter(
                x[:, imp_ordered_ind[f1]], 
                y_values,
                s=bubble_sizes,
                c=x[:, imp_ordered_ind[f2_]],
                edgecolor='k',
                cmap=plt.cm.Greens,
                alpha=0.8
            )

            plt.ylabel('Sum of SHAP values (Year + Donor Type + Perfusion Prior)')
            plt.xlabel(feature_names[imp_ordered_ind[f1]])
            plt.colorbar().ax.set_ylabel(feature_names[imp_ordered_ind[f2_]])
            plt.ylim(y_values.min() - 0.1, y_values.max() + 0.1)

            print('bubble size:', feature_names[imp_ordered_ind[f3_]])
            plt.tight_layout()
            plt.savefig("dependency_sum3.png", dpi=300)
            plt.show()
            i += 1

print('num of figures', i)


In [ ]:

import numpy as np
import matplotlib.pyplot as plt

# Feature indices
year_index = 9               # TX_YEAR
perfusion_index = 46         # PERFUSED_PRIOR
donor_type_index = 43        # DONOR_TYPE_BINARY

years = np.unique(x[:, imp_ordered_ind[year_index]])
perfusion_cats = [0, 1]
donor_types = [0, 1]

# Color palette for 4 category combinations
colors = {
    (0, 0): '#1b9e77',  # green
    (0, 1): '#d95f02',  # orange
    (1, 0): '#7570b3',  # purple
    (1, 1): '#e7298a'   # pink
}

# Horizontal offsets for the 4 groups
offset_values = [-1.5, -0.5, 0.5, 1.5]
category_keys = list(colors.keys())

positions = []
plot_data = []
plot_colors = []
group_counts = []

# Collect data for plotting
for i, year in enumerate(years):
    base_pos = i * 5  # spacing between year groups

    for j, (p, d) in enumerate(category_keys):
        mask = (
            (x[:, imp_ordered_ind[year_index]] == year) &
            (x[:, imp_ordered_ind[perfusion_index]] == p) &
            (x[:, imp_ordered_ind[donor_type_index]] == d)
        )
        shap_vals = shap_values[mask, imp_ordered_ind[year_index]]

        pos = base_pos + offset_values[j]
        positions.append(pos)
        plot_data.append(shap_vals)
        plot_colors.append(colors[(p, d)])
        group_counts.append(len(shap_vals))

# Create box plot
fig, ax = plt.subplots(figsize=(16, 6))
bp = ax.boxplot(plot_data, positions=positions, patch_artist=True, widths=0.8)

# Apply colors to boxes
for patch, color in zip(bp['boxes'], plot_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

# Add sample size above each box
for pos, data, n in zip(positions, plot_data, group_counts):
    if len(data) > 0:
        y_max = np.nanmax(data)
        ax.text(pos, y_max + 0.01, f'n={n}', ha='center', va='bottom', fontsize=8, color='black')
    else:
        ax.text(pos, 0, 'n=0', ha='center', va='bottom', fontsize=8, color='gray')

# Axis settings
ax.set_xticks([i * 5 for i in range(len(years))])
ax.set_xticklabels([int(y) for y in years])
ax.set_xlabel('TX_YEAR')
ax.set_ylabel('SHAP Values')
ax.set_title('SHAP Boxplots by Year, Perfused Prior & Donor Type (with sample sizes)')
ax.grid(axis='y', linestyle='--', alpha=0.6)

# Custom legend labels
legend_labels = {
    (0, 0): 'No perfusion prior / DBD',
    (0, 1): 'No perfusion prior / DCD',
    (1, 0): 'Perfusion prior / DBD',
    (1, 1): 'Perfusion prior / DCD'
}

handles = [plt.Line2D([0], [0], color=colors[key], lw=10) for key in colors]
labels = [legend_labels[key] for key in colors]
ax.legend(handles, labels, title='Groups')

plt.tight_layout()
plt.savefig("boxplot_all_years_with_counts.png", dpi=300)
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Feature indices
year_index = 9               # TX_YEAR
perfusion_index = 46         # PERFUSED_PRIOR
donor_type_index = 43        # DONOR_TYPE_BINARY

years = np.unique(x[:, imp_ordered_ind[year_index]])
perfusion_cats = [0, 1]
donor_types = [0, 1]

# Color palette
colors = {
    (0, 0): '#1b9e77',  # green
    (0, 1): '#d95f02',  # orange
    (1, 0): '#7570b3',  # purple
    (1, 1): '#e7298a'   # pink
}

# Horizontal offsets for each category
offset_values = [-1.5, -0.5, 0.5, 1.5]
category_keys = list(colors.keys())

positions = []
plot_data = []
plot_colors = []
group_counts = []

# Collect data
for i, year in enumerate(years):
    base_pos = i * 5

    for j, (p, d) in enumerate(category_keys):
        mask = (
            (x[:, imp_ordered_ind[year_index]] == year) &
            (x[:, imp_ordered_ind[perfusion_index]] == p) &
            (x[:, imp_ordered_ind[donor_type_index]] == d)
        )

        # Sum of SHAP values for year + perfused_prior + donor_type
        shap_sum = (
            shap_values[mask, imp_ordered_ind[year_index]] +
            shap_values[mask, imp_ordered_ind[perfusion_index]] +
            shap_values[mask, imp_ordered_ind[donor_type_index]]
        )

        pos = base_pos + offset_values[j]
        positions.append(pos)
        plot_data.append(shap_sum)
        plot_colors.append(colors[(p, d)])
        group_counts.append(len(shap_sum))

# Plot
fig, ax = plt.subplots(figsize=(16, 6))
bp = ax.boxplot(plot_data, positions=positions, patch_artist=True, widths=0.8)

# Colors
for patch, color in zip(bp['boxes'], plot_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

# Add sample size above boxes
for pos, data, n in zip(positions, plot_data, group_counts):
    if len(data) > 0:
        y_max = np.nanmax(data)
        ax.text(pos, y_max + 0.01, f'n={n}', ha='center', va='bottom', fontsize=8, color='black')
    else:
        ax.text(pos, 0, 'n=0', ha='center', va='bottom', fontsize=8, color='gray')

# Axes and labels
ax.set_xticks([i * 5 for i in range(len(years))])
ax.set_xticklabels([int(y) for y in years])
ax.set_xlabel('TX_YEAR')
ax.set_ylabel('Sum of SHAP Values (Year + Perfusion Prior + Donor Type)')
ax.set_title('Total SHAP Boxplots by Year, Perfused Prior & Donor Type')
ax.grid(axis='y', linestyle='--', alpha=0.6)

# Legend
legend_labels = {
    (0, 0): 'No perfusion prior / DBD',
    (0, 1): 'No perfusion prior / DCD',
    (1, 0): 'Perfusion prior / DBD',
    (1, 1): 'Perfusion prior / DCD'
}
handles = [plt.Line2D([0], [0], color=colors[key], lw=10) for key in colors]
labels = [legend_labels[key] for key in colors]
ax.legend(handles, labels, title='Groups')

plt.tight_layout()
plt.savefig("boxplot_all_years_sum_shap.png", dpi=300)
plt.show()
#isch time 
 #type of donor, 

In [ ]:
shap_sum = shap_values[mask, :].sum(axis=1)
import numpy as np
import matplotlib.pyplot as plt

# Feature indices
year_index = 9               # TX_YEAR
perfusion_index = 46         # PERFUSED_PRIOR
donor_type_index = 43        # DONOR_TYPE_BINARY

years = np.unique(x[:, imp_ordered_ind[year_index]])
perfusion_cats = [0, 1]
donor_types = [0, 1]

# Color palette
colors = {
    (0, 0): '#1b9e77',  # green
    (0, 1): '#d95f02',  # orange
    (1, 0): '#7570b3',  # purple
    (1, 1): '#e7298a'   # pink
}

# Horizontal offsets for each category
offset_values = [-1.5, -0.5, 0.5, 1.5]
category_keys = list(colors.keys())

positions = []
plot_data = []
plot_colors = []
group_counts = []

# Collect data
for i, year in enumerate(years):
    base_pos = i * 5  # space between year groups

    for j, (p, d) in enumerate(category_keys):
        mask = (
            (x[:, imp_ordered_ind[year_index]] == year) &
            (x[:, imp_ordered_ind[perfusion_index]] == p) &
            (x[:, imp_ordered_ind[donor_type_index]] == d)
        )

        # Sum SHAP values across ALL features for each patient
        shap_sum = shap_values[mask, :].sum(axis=1)

        pos = base_pos + offset_values[j]
        positions.append(pos)
        plot_data.append(shap_sum)
        plot_colors.append(colors[(p, d)])
        group_counts.append(len(shap_sum))

# Plot
fig, ax = plt.subplots(figsize=(16, 6))
bp = ax.boxplot(plot_data, positions=positions, patch_artist=True, widths=0.8)

# Colors
for patch, color in zip(bp['boxes'], plot_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

# Add sample size above boxes
for pos, data, n in zip(positions, plot_data, group_counts):
    if len(data) > 0:
        y_max = np.nanmax(data)
        ax.text(pos, y_max + 0.01, f'n={n}', ha='center', va='bottom', fontsize=8, color='black')
    else:
        ax.text(pos, 0, 'n=0', ha='center', va='bottom', fontsize=8, color='gray')

# Axes and labels
ax.set_xticks([i * 5 for i in range(len(years))])
ax.set_xticklabels([int(y) for y in years])
ax.set_xlabel('TX_YEAR')
ax.set_ylabel('Sum of SHAP Values (All Features)')
ax.set_title('Total SHAP Boxplots by Year, Perfused Prior & Donor Type')
ax.grid(axis='y', linestyle='--', alpha=0.6)

# Legend
legend_labels = {
    (0, 0): 'No perfusion prior / DBD',
    (0, 1): 'No perfusion prior / DCD',
    (1, 0): 'Perfusion prior / DBD',
    (1, 1): 'Perfusion prior / DCD'
}
handles = [plt.Line2D([0], [0], color=colors[key], lw=10) for key in colors]
labels = [legend_labels[key] for key in colors]
ax.legend(handles, labels, title='Groups')

plt.tight_layout()
plt.savefig("boxplot_all_years_total_shap.png", dpi=300)
plt.show()

In [ ]:
# 3D SHAP dependency plots for f1=1
f1=9
x_dcd=np.array(x_dcd)
x_n_dcd=np.array(x_n_dcd)
print('Dependency Plots for', feature_names[imp_ordered_ind[f1]])
print()
i=0
for f2 in range(43,44):
    for f3 in range(46,47):
        if ((f1 != f2) and (f2!= f3) and (f1 != f3)):
            plt.clf()
            plt.figure(figsize=(7,8))
            sizes = x_dcd[:, imp_ordered_ind[f3]]
            sizes[sizes == -1] = 0  
            plt.scatter(x_dcd[:,imp_ordered_ind[f1]], shap_values_dcd[:,imp_ordered_ind[f1]], s = 30*sizes+5,  c = x_dcd[:,imp_ordered_ind[f2]], edgecolor = 'k', cmap=plt.cm.Greens, alpha=0.8)
            plt.ylabel('SHAP values')
            plt.ylim(-0.2,0.3)
            #plt.xticks(np.arange(0,101,20))
            plt.xlabel(feature_names[imp_ordered_ind[f1]])
            plt.colorbar().ax.set_ylabel(feature_names[imp_ordered_ind[f2]])
            print('bubble size: ', feature_names[imp_ordered_ind[f3]])
            plt.savefig("dependency_year.png",dpi=300)
            plt.show()
            i+=1
print('num of figures',i)

In [ ]:
# 3D SHAP dependency plots for f1=1
f1=9
x_dbd=np.array(x_dbd)
x_n_dbd=np.array(x_n_dbd)
print('Dependency Plots for', feature_names[imp_ordered_ind[f1]])
print()
i=0
for f2 in range(43,44):
    for f3 in range(46,47):
        if ((f1 != f2) and (f2!= f3) and (f1 != f3)):
            plt.clf()
            plt.figure(figsize=(7,8))
            sizes = x_dbd[:, imp_ordered_ind[f3]]
            sizes[sizes == -1] = 0  
            plt.scatter(x_dbd[:,imp_ordered_ind[f1]], shap_values_dbd[:,imp_ordered_ind[f1]], s = 30*sizes+5,  c = x_dbd[:,imp_ordered_ind[f2]], edgecolor = 'k', cmap=plt.cm.Greens, alpha=0.8)
            plt.ylabel('SHAP values')
            plt.ylim(-0.2,0.3)
            #plt.xticks(np.arange(0,101,20))
            plt.xlabel(feature_names[imp_ordered_ind[f1]])
            plt.colorbar().ax.set_ylabel(feature_names[imp_ordered_ind[f2]])
            print('bubble size: ', feature_names[imp_ordered_ind[f3]])
            plt.savefig("dependency_year.png",dpi=300)
            plt.show()
            i+=1
print('num of figures',i)

In [ ]:
# 3D SHAP dependency plots for f1=1
f1=9
x_dcd=np.array(x_dcd)
x_n_dcd=np.array(x_n_dcd)
print('Dependency Plots for', feature_names[imp_ordered_ind[f1]])
print()
i=0
for f2 in range(46,47):
    for f3 in range(6,49):
        if ((f1 != f2) and (f2!= f3) and (f1 != f3)):
            plt.clf()
            plt.figure(figsize=(7,8))
            plt.scatter(x_dcd[:,imp_ordered_ind[f1]], shap_values_dcd[:,imp_ordered_ind[f1]], s = 10*x_n_dcd[:,imp_ordered_ind[f3]]+1,  c = x_dcd[:,imp_ordered_ind[f2]], edgecolor = 'k', cmap=plt.cm.Greens, alpha=0.8)
            plt.ylabel('SHAP values')
            plt.ylim(-0.2,0.3)
            #plt.xticks(np.arange(0,101,20))
            plt.xlabel(feature_names[imp_ordered_ind[f1]])
            plt.colorbar().ax.set_ylabel(feature_names[imp_ordered_ind[f2]])
            print('bubble size: ', feature_names[imp_ordered_ind[f3]])
            plt.savefig("dependency_year.png",dpi=300)
            plt.show()
            i+=1
print('num of figures',i)

In [ ]:
x[:,imp_ordered_ind[f3]].sum()

In [ ]:

# Indices from X by DONOR_TYPE_BINARY (no if/else on column name)
mask_dbd = (x["DONOR_TYPE_BINARY"].to_numpy() == 0)
mask_dcd = (x["DONOR_TYPE_BINARY"].to_numpy() == 1)

# DBD
shap.summary_plot(
    shap_values[mask_dbd],
    x1.loc[mask_dbd, :],
    feature_names=feature_names,
    max_display=max_feature,
    show=False
)



In [ ]:
# DCD
shap.summary_plot(
    shap_values[mask_dcd],
    x1.loc[mask_dcd, :],
    feature_names=feature_names,
    max_display=max_feature,
    show=False
)


In [ ]:
# SHAP Dependence plots 
# interaction between important variables
max_feature = 50

for f1 in range(max_feature):
    plt.figure(figsize=(3,3))
    shap.dependence_plot(imp_ordered_ind[f1], shap_values, x, feature_names =feature_names)
   

In [ ]:
# Complete UMAP + DBSCAN Parameter Optimization for DBD Patients Only - 3D Version
# Tests all combinations and creates graphs for each with 3D visualization

import itertools
from sklearn.metrics import silhouette_score, calinski_harabasz_score
from sklearn.cluster import DBSCAN
import umap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from scipy.stats import chi2_contingency
from mpl_toolkits.mplot3d import Axes3D

print("=== Complete UMAP + DBSCAN Parameter Optimization for DBD Patients - 3D Version ===")
print("Goal: Find parameters that create distinct high/low mortality risk clusters for DBD patients")
print("Each combination will be tested and visualized in 3D")

# Create DBD mask
mask_dbd = (x["DONOR_TYPE_BINARY"].to_numpy() == 0)
mask_dcd = (x["DONOR_TYPE_BINARY"].to_numpy() == 1)

print(f"Total patients: {len(x)}")
print(f"DBD patients: {np.sum(mask_dbd)}")
print(f"DCD patients: {np.sum(mask_dcd)}")

# Filter data for DBD patients only
shap_values_dbd = shap_values[mask_dbd]
y_dbd = y[mask_dbd]

print(f"DBD SHAP values shape: {shap_values_dbd.shape}")
print(f"DBD outcome shape: {y_dbd.shape}")
print(f"DBD mortality rate: {np.mean(y_dbd):.3f}")

# Define parameter ranges to test - 3D VERSION
umap_params = {
    'n_neighbors': [15, 30],
    'min_dist': [0.1, 0.2, 0.3],
    'n_components': [3]  # 3D visualization
}

dbscan_params = {
    'eps': [0.2, 0.3, 0.4],
    'min_samples': [50, 100]
}

# Results storage
results = []
total_combinations = len(umap_params['n_neighbors']) * len(umap_params['min_dist']) * len(umap_params['n_components']) * len(dbscan_params['eps']) * len(dbscan_params['min_samples'])

print(f"Testing {total_combinations} parameter combinations for DBD patients...")

# Test all combinations
combination_count = 0
for n_neighbors in umap_params['n_neighbors']:
    for min_dist in umap_params['min_dist']:
        for n_components in umap_params['n_components']:
            for eps in dbscan_params['eps']:
                for min_samples in dbscan_params['min_samples']:
                    
                    combination_count += 1
                    print(f"\n--- Testing Combination {combination_count}/{total_combinations} ---")
                    print(f"UMAP: n_neighbors={n_neighbors}, min_dist={min_dist}, n_components={n_components}")
                    print(f"DBSCAN: eps={eps}, min_samples={min_samples}")
                    
                    try:
                        # Step 1: UMAP
                        umap_reducer = umap.UMAP(
                            n_components=n_components,
                            n_neighbors=n_neighbors,
                            min_dist=min_dist,
                            metric='euclidean',
                            random_state=42
                        )
                        
                        # Fit UMAP on DBD patients only
                        umap_embedding = umap_reducer.fit_transform(shap_values_dbd)
                        
                        # Step 2: DBSCAN
                        dbscan = DBSCAN(eps=eps, min_samples=min_samples)
                        clusters = dbscan.fit_predict(umap_embedding)
                        
                        # Calculate metrics
                        n_clusters = len(set(clusters)) - (1 if -1 in clusters else 0)
                        n_noise = list(clusters).count(-1)
                        
                        # Skip if too few clusters or too much noise
                        if n_clusters < 2 or n_noise > len(clusters) * 0.5:
                            print(f"❌ Skipped: {n_clusters} clusters, {n_noise} noise ({n_noise/len(clusters):.1%})")
                            continue
                        
                        # DEBUGGING: Cluster ID'leri kontrol et
                        print(f"\n=== CLUSTER ID DEBUGGING ===")
                        print(f"All cluster IDs: {sorted(set(clusters))}")
                        unique_clusters = sorted([c for c in set(clusters) if c != -1])
                        print(f"Unique clusters (excluding noise): {unique_clusters}")
                        print(f"Number of unique clusters: {len(unique_clusters)}")
                        
                        # Her cluster ID için sayıları kontrol edin
                        for cluster_id in sorted(set(clusters)):
                            count = np.sum(clusters == cluster_id)
                            if cluster_id == -1:
                                print(f"Noise (ID=-1): {count} patients")
                            else:
                                print(f"Cluster ID {cluster_id}: {count} patients")
                        
                        # Gerçek noise'u kontrol edin
                        noise_mask = clusters == -1
                        if np.sum(noise_mask) > 0:
                            print(f"Real noise points: {np.sum(noise_mask)}")
                            print(f"Noise mortality rate: {np.mean(y_dbd[noise_mask]):.3f}")
                        else:
                            print("No noise points found!")
                        
                        # Calculate clustering quality metrics - FIXED
                        if n_clusters > 1:
                            # Remove noise points for silhouette calculation
                            non_noise_mask = clusters != -1
                            if np.sum(non_noise_mask) > 1:
                                silhouette = silhouette_score(umap_embedding[non_noise_mask], 
                                                            clusters[non_noise_mask])
                                calinski = calinski_harabasz_score(umap_embedding[non_noise_mask], 
                                                                 clusters[non_noise_mask])
                            else:
                                silhouette = -1
                                calinski = 0
                        else:
                            silhouette = -1
                            calinski = 0
                        
                        # Calculate mortality risk separation for DBD patients
                        mortality_by_cluster = []
                        cluster_sizes = []
                        
                        for cluster_id in range(n_clusters):
                            cluster_mask = clusters == cluster_id
                            if np.sum(cluster_mask) > 0:
                                cluster_mortality = np.mean(y_dbd[cluster_mask])
                                cluster_size = np.sum(cluster_mask)
                                mortality_by_cluster.append(cluster_mortality)
                                cluster_sizes.append(cluster_size)
                        
                        # Calculate mortality risk variance and range
                        mortality_variance = np.var(mortality_by_cluster) if len(mortality_by_cluster) > 1 else 0
                        mortality_range = max(mortality_by_cluster) - min(mortality_by_cluster) if len(mortality_by_cluster) > 1 else 0
                        
                        # Print results
                        print(f"✅ Results: {n_clusters} clusters, {n_noise} noise ({n_noise/len(clusters):.1%})")
                        print(f"   Silhouette: {silhouette:.3f}")
                        print(f"   Calinski-Harabasz: {calinski:.2f}")
                        print(f"   Mortality Range: {mortality_range:.3f}")
                        print(f"   Mortality Rates: {[f'{x:.3f}' for x in mortality_by_cluster]}")
                        print(f"   Cluster Sizes: {cluster_sizes}")
                        
                        # Create visualization for this combination - 3D VERSION
                        fig = plt.figure(figsize=(20, 16))
                        fig.suptitle(f'DBD Patients - Combination {combination_count}: UMAP({n_neighbors},{min_dist},{n_components}) + DBSCAN({eps},{min_samples})', 
                                   fontsize=16, fontweight='bold')
                        
                        # Get unique clusters (excluding noise points with -1)
                        n_unique_clusters = len(unique_clusters)
                        
                        # Plot 1: 3D UMAP embedding with clusters - NOISE SEPARATED
                        ax1 = fig.add_subplot(2, 3, 1, projection='3d')
                        colors = cm.get_cmap('tab20', n_unique_clusters)
                        
                        # Plot clusters (non-noise points)
                        non_noise_mask = clusters != -1
                        scatter1 = ax1.scatter(umap_embedding[non_noise_mask, 0], 
                                             umap_embedding[non_noise_mask, 1], 
                                             umap_embedding[non_noise_mask, 2],
                                             c=clusters[non_noise_mask], cmap=colors, alpha=0.7, s=20)
                        
                        # Plot noise points separately in black
                        if np.sum(noise_mask) > 0:
                            ax1.scatter(umap_embedding[noise_mask, 0], 
                                       umap_embedding[noise_mask, 1], 
                                       umap_embedding[noise_mask, 2],
                                       c='black', alpha=0.7, s=20, label='Noise')
                        
                        ax1.set_title('3D UMAP Embedding (Clustered) - DBD Only', fontsize=14, fontweight='bold')
                        ax1.set_xlabel('UMAP 1')
                        ax1.set_ylabel('UMAP 2')
                        ax1.set_zlabel('UMAP 3')
                        
                        # Add legend for noise
                        if np.sum(noise_mask) > 0:
                            ax1.legend(loc='upper right')
                        
                        # Plot 2: 3D UMAP with outcome colors - NOISE SEPARATED
                        ax2 = fig.add_subplot(2, 3, 2, projection='3d')
                        colors_outcome = ['red' if outcome == 1 else 'blue' for outcome in y_dbd]
                        
                        # Plot clusters (non-noise points)
                        ax2.scatter(umap_embedding[non_noise_mask, 0], 
                                  umap_embedding[non_noise_mask, 1], 
                                  umap_embedding[non_noise_mask, 2],
                                  c=[colors_outcome[i] for i in range(len(colors_outcome)) if clusters[i] != -1], 
                                  alpha=0.7, s=20)
                        
                        # Plot noise points separately in black
                        if np.sum(noise_mask) > 0:
                            ax2.scatter(umap_embedding[noise_mask, 0], 
                                       umap_embedding[noise_mask, 1], 
                                       umap_embedding[noise_mask, 2],
                                       c='black', alpha=0.7, s=20, label='Noise')
                        
                        ax2.set_title('3D UMAP Embedding (Outcome) - DBD Only', fontsize=14, fontweight='bold')
                        ax2.set_xlabel('UMAP 1')
                        ax2.set_ylabel('UMAP 2')
                        ax2.set_zlabel('UMAP 3')
                        
                        # Add legend for noise
                        if np.sum(noise_mask) > 0:
                            ax2.legend(loc='upper right')
                        
                        # Plot 3: 2D Projection (UMAP 1 vs UMAP 2)
                        ax3 = fig.add_subplot(2, 3, 3)
                        scatter3 = ax3.scatter(umap_embedding[non_noise_mask, 0], umap_embedding[non_noise_mask, 1], 
                                             c=clusters[non_noise_mask], cmap=colors, alpha=0.7, s=20)
                        
                        if np.sum(noise_mask) > 0:
                            ax3.scatter(umap_embedding[noise_mask, 0], umap_embedding[noise_mask, 1], 
                                       c='black', alpha=0.7, s=20, label='Noise')
                        
                        ax3.set_title('2D Projection (UMAP 1 vs UMAP 2)', fontsize=14, fontweight='bold')
                        ax3.set_xlabel('UMAP 1')
                        ax3.set_ylabel('UMAP 2')
                        
                        # Create colorbar for 2D projection
                        cbar3 = plt.colorbar(scatter3, ax=ax3)
                        cbar3.set_label('Cluster ID', rotation=270, labelpad=15)
                        cbar3.set_ticks(unique_clusters)
                        cbar3.set_ticklabels([str(cluster_id) for cluster_id in unique_clusters])
                        
                        if np.sum(noise_mask) > 0:
                            ax3.legend(loc='upper right')
                        
                        # Plot 4: 2D Projection (UMAP 1 vs UMAP 3)
                        ax4 = fig.add_subplot(2, 3, 4)
                        scatter4 = ax4.scatter(umap_embedding[non_noise_mask, 0], umap_embedding[non_noise_mask, 2], 
                                             c=clusters[non_noise_mask], cmap=colors, alpha=0.7, s=20)
                        
                        if np.sum(noise_mask) > 0:
                            ax4.scatter(umap_embedding[noise_mask, 0], umap_embedding[noise_mask, 2], 
                                       c='black', alpha=0.7, s=20, label='Noise')
                        
                        ax4.set_title('2D Projection (UMAP 1 vs UMAP 3)', fontsize=14, fontweight='bold')
                        ax4.set_xlabel('UMAP 1')
                        ax4.set_ylabel('UMAP 3')
                        
                        if np.sum(noise_mask) > 0:
                            ax4.legend(loc='upper right')
                        
                        # Plot 5: 2D Projection (UMAP 2 vs UMAP 3)
                        ax5 = fig.add_subplot(2, 3, 5)
                        scatter5 = ax5.scatter(umap_embedding[non_noise_mask, 1], umap_embedding[non_noise_mask, 2], 
                                             c=clusters[non_noise_mask], cmap=colors, alpha=0.7, s=20)
                        
                        if np.sum(noise_mask) > 0:
                            ax5.scatter(umap_embedding[noise_mask, 1], umap_embedding[noise_mask, 2], 
                                       c='black', alpha=0.7, s=20, label='Noise')
                        
                        ax5.set_title('2D Projection (UMAP 2 vs UMAP 3)', fontsize=14, fontweight='bold')
                        ax5.set_xlabel('UMAP 2')
                        ax5.set_ylabel('UMAP 3')
                        
                        if np.sum(noise_mask) > 0:
                            ax5.legend(loc='upper right')
                        
                        # Plot 6: Cluster size distribution
                        ax6 = fig.add_subplot(2, 3, 6)
                        cluster_counts = np.bincount(clusters + 1)  # +1 to handle -1 noise points
                        cluster_counts_no_noise = cluster_counts[1:]  # Remove noise
                        
                        ax6.bar(range(len(cluster_counts_no_noise)), cluster_counts_no_noise, color='skyblue', alpha=0.7)
                        ax6.set_title('Cluster Size Distribution - DBD Only', fontsize=14, fontweight='bold')
                        ax6.set_xlabel('Cluster ID')
                        ax6.set_ylabel('Number of Patients')
                        ax6.grid(True, alpha=0.3)
                        
                        # Set x-axis ticks to show REAL cluster IDs
                        ax6.set_xticks(range(len(cluster_counts_no_noise)))
                        ax6.set_xticklabels([str(unique_clusters[i]) for i in range(len(cluster_counts_no_noise))])
                        
                        # Add value labels on bars
                        for i, v in enumerate(cluster_counts_no_noise):
                            ax6.text(i, v + max(cluster_counts_no_noise)*0.01, str(v), ha='center', va='bottom', fontweight='bold')
                        
                        plt.tight_layout()
                        plt.show()
                        
                        # Additional 3D plot with mortality rates
                        fig2 = plt.figure(figsize=(15, 5))
                        fig2.suptitle(f'Mortality Analysis - Combination {combination_count}', fontsize=16, fontweight='bold')
                        
                        # 3D plot colored by mortality rate
                        ax_mort = fig2.add_subplot(1, 3, 1, projection='3d')
                        mortality_colors = [np.mean(y_dbd[clusters == cluster_id]) if cluster_id != -1 else 0.5 
                                          for cluster_id in clusters]
                        scatter_mort = ax_mort.scatter(umap_embedding[:, 0], umap_embedding[:, 1], umap_embedding[:, 2],
                                                     c=mortality_colors, cmap='RdYlBu_r', alpha=0.7, s=20)
                        ax_mort.set_title('3D UMAP (Mortality Rate)', fontsize=14, fontweight='bold')
                        ax_mort.set_xlabel('UMAP 1')
                        ax_mort.set_ylabel('UMAP 2')
                        ax_mort.set_zlabel('UMAP 3')
                        plt.colorbar(scatter_mort, ax=ax_mort, label='Mortality Rate')
                        
                        # 2D projection with mortality
                        ax_mort2d = fig2.add_subplot(1, 3, 2)
                        scatter_mort2d = ax_mort2d.scatter(umap_embedding[:, 0], umap_embedding[:, 1],
                                                         c=mortality_colors, cmap='RdYlBu_r', alpha=0.7, s=20)
                        ax_mort2d.set_title('2D Projection (Mortality Rate)', fontsize=14, fontweight='bold')
                        ax_mort2d.set_xlabel('UMAP 1')
                        ax_mort2d.set_ylabel('UMAP 2')
                        plt.colorbar(scatter_mort2d, ax=ax_mort2d, label='Mortality Rate')
                        
                        # Mortality rate by cluster bar chart
                        ax_mort_bar = fig2.add_subplot(1, 3, 3)
                        cluster_outcomes = []
                        cluster_ids = []
                        for i, cluster_id in enumerate(unique_clusters):
                            cluster_mask = clusters == cluster_id
                            if np.sum(cluster_mask) > 0:
                                cluster_outcome_rate = np.mean(y_dbd[cluster_mask])
                                cluster_outcomes.append(cluster_outcome_rate)
                                cluster_ids.append(cluster_id)
                        
                        ax_mort_bar.bar(cluster_ids, cluster_outcomes, color='lightcoral', alpha=0.7)
                        ax_mort_bar.set_title('Mortality Rate by Cluster - DBD Only', fontsize=14, fontweight='bold')
                        ax_mort_bar.set_xlabel('Cluster ID')
                        ax_mort_bar.set_ylabel('Mortality Rate')
                        ax_mort_bar.grid(True, alpha=0.3)
                        ax_mort_bar.set_xticks(cluster_ids)
                        ax_mort_bar.set_xticklabels(cluster_ids)
                        
                        # Add value labels on bars
                        for i, v in enumerate(cluster_outcomes):
                            ax_mort_bar.text(cluster_ids[i], v + max(cluster_outcomes)*0.01, f'{v:.3f}',
                                           ha='center', va='bottom', fontweight='bold', fontsize=9)
                        
                        plt.tight_layout()
                        plt.show()
                        
                        # Store results
                        result = {
                            'combination_id': combination_count,
                            'n_neighbors': n_neighbors,
                            'min_dist': min_dist,
                            'n_components': n_components,
                            'eps': eps,
                            'min_samples': min_samples,
                            'n_clusters': n_clusters,
                            'n_noise': n_noise,
                            'noise_ratio': n_noise / len(clusters),
                            'silhouette_score': silhouette,
                            'calinski_harabasz_score': calinski,
                            'mortality_variance': mortality_variance,
                            'mortality_range': mortality_range,
                            'mortality_by_cluster': mortality_by_cluster,
                            'cluster_sizes': cluster_sizes,
                            'min_cluster_size': min(cluster_sizes) if cluster_sizes else 0,
                            'max_cluster_size': max(cluster_sizes) if cluster_sizes else 0
                        }
                        
                        results.append(result)
                        
                    except Exception as e:
                        print(f"❌ Error: {e}")
                        import traceback
                        traceback.print_exc()
                        continue

print(f"\n=== OPTIMIZATION COMPLETE FOR DBD PATIENTS - 3D VERSION ===")
print(f"Completed testing. Found {len(results)} valid parameter combinations.")

# Convert to DataFrame for analysis
results_df = pd.DataFrame(results)

if len(results_df) > 0:
    print(f"\n=== TOP 10 PARAMETER COMBINATIONS BY MORTALITY RANGE (DBD ONLY - 3D) ===")
    top_results = results_df.nlargest(10, 'mortality_range')
    
    for i, (_, row) in enumerate(top_results.iterrows()):
        print(f"\n{i+1}. Combination {row['combination_id']}: Mortality Range = {row['mortality_range']:.3f}")
        print(f"   UMAP: n_neighbors={row['n_neighbors']}, min_dist={row['min_dist']}, n_components={row['n_components']}")
        print(f"   DBSCAN: eps={row['eps']}, min_samples={row['min_samples']}")
        print(f"   Clusters: {row['n_clusters']}, Noise: {row['n_noise']} ({row['noise_ratio']:.1%})")
        print(f"   Silhouette: {row['silhouette_score']:.3f}")
        print(f"   CH Score: {row['calinski_harabasz_score']:.2f}")
        print(f"   Mortality rates: {[f'{x:.3f}' for x in row['mortality_by_cluster']]}")
        print(f"   Cluster sizes: {row['cluster_sizes']}")
        print(f"   Size range: {row['min_cluster_size']}-{row['max_cluster_size']}")
    
    # Create summary visualization
    print(f"\n=== CREATING SUMMARY VISUALIZATION FOR DBD PATIENTS - 3D VERSION ===")
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Parameter Optimization Summary - DBD Patients Only (3D)', fontsize=16, fontweight='bold')
    
    # Plot 1: Mortality Range vs Silhouette Score
    scatter = axes[0,0].scatter(results_df['mortality_range'], results_df['silhouette_score'], 
                               c=results_df['n_clusters'], cmap='viridis', alpha=0.7, s=100)
    axes[0,0].set_xlabel('Mortality Range')
    axes[0,0].set_ylabel('Silhouette Score')
    axes[0,0].set_title('Mortality Range vs Silhouette Score (DBD Only - 3D)')
    plt.colorbar(scatter, ax=axes[0,0], label='Number of Clusters')
    
    # Plot 2: Parameter combinations heatmap
    param_matrix = results_df.pivot_table(values='mortality_range', 
                                        index='eps', 
                                        columns='min_samples', 
                                        aggfunc='mean')
    sns.heatmap(param_matrix, annot=True, fmt='.3f', cmap='YlOrRd', ax=axes[0,1])
    axes[0,1].set_title('Mortality Range by DBSCAN Parameters (DBD Only - 3D)')
    
    # Plot 3: Cluster count distribution
    axes[1,0].hist(results_df['n_clusters'], bins=range(1, results_df['n_clusters'].max()+2), 
                   alpha=0.7, color='skyblue', edgecolor='black')
    axes[1,0].set_xlabel('Number of Clusters')
    axes[1,0].set_ylabel('Frequency')
    axes[1,0].set_title('Distribution of Cluster Counts (DBD Only - 3D)')
    
    # Plot 4: Top 5 combinations comparison
    top_5 = results_df.nlargest(5, 'mortality_range')
    x_pos = range(len(top_5))
    bars = axes[1,1].bar(x_pos, top_5['mortality_range'], alpha=0.7, color='lightcoral')
    axes[1,1].set_xlabel('Top 5 Combinations')
    axes[1,1].set_ylabel('Mortality Range')
    axes[1,1].set_title('Top 5 Parameter Combinations (DBD Only - 3D)')
    axes[1,1].set_xticks(x_pos)
    axes[1,1].set_xticklabels([f'#{int(row["combination_id"])}' for _, row in top_5.iterrows()], rotation=45)
    
    # Add value labels
    for i, v in enumerate(top_5['mortality_range']):
        axes[1,1].text(i, v + max(top_5['mortality_range'])*0.01, f'{v:.3f}',
                      ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n=== BEST PARAMETERS RECOMMENDATION FOR DBD PATIENTS - 3D VERSION ===")
    best_combo = results_df.loc[results_df['mortality_range'].idxmax()]
    print(f"Best Combination: #{int(best_combo['combination_id'])}")
    print(f"UMAP: n_neighbors={best_combo['n_neighbors']}, min_dist={best_combo['min_dist']}, n_components={best_combo['n_components']}")
    print(f"DBSCAN: eps={best_combo['eps']}, min_samples={best_combo['min_samples']}")
    print(f"Mortality Range: {best_combo['mortality_range']:.3f}")
    print(f"Silhouette Score: {best_combo['silhouette_score']:.3f}")
    print(f"CH Score: {best_combo['calinski_harabasz_score']:.2f}")
    print(f"Cluster Sizes: {best_combo['cluster_sizes']}")
    print(f"Mortality Rates: {[f'{x:.3f}' for x in best_combo['mortality_by_cluster']]}")
    
    # Save best parameters for future use
    best_params_dbd_3d = {
        'n_neighbors': best_combo['n_neighbors'],
        'min_dist': best_combo['min_dist'],
        'n_components': best_combo['n_components'],
        'eps': best_combo['eps'],
        'min_samples': best_combo['min_samples']
    }
    
    print(f"\n=== BEST PARAMETERS FOR DBD PATIENTS - 3D VERSION ===")
    print(f"Best UMAP + DBSCAN parameters: {best_params_dbd_3d}")
    
else:
    print("No valid parameter combinations found for DBD patients!")

In [ ]:
# Complete UMAP + DBSCAN Parameter Optimization for DBD Patients Only
# Tests all combinations and creates graphs for each

import itertools
from sklearn.metrics import silhouette_score, calinski_harabasz_score
from sklearn.cluster import DBSCAN
import umap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from scipy.stats import chi2_contingency

print("=== Complete UMAP + DBSCAN Parameter Optimization for DBD Patients ===")
print("Goal: Find parameters that create distinct high/low mortality risk clusters for DBD patients")
print("Each combination will be tested and visualized")

# Create DBD mask
mask_dbd = (x["DONOR_TYPE_BINARY"].to_numpy() == 0)
mask_dcd = (x["DONOR_TYPE_BINARY"].to_numpy() == 1)

print(f"Total patients: {len(x)}")
print(f"DBD patients: {np.sum(mask_dbd)}")
print(f"DCD patients: {np.sum(mask_dcd)}")

# Filter data for DBD patients only
shap_values_dbd = shap_values[mask_dbd]
y_dbd = y[mask_dbd]

print(f"DBD SHAP values shape: {shap_values_dbd.shape}")
print(f"DBD outcome shape: {y_dbd.shape}")
print(f"DBD mortality rate: {np.mean(y_dbd):.3f}")

# Define parameter ranges to test
umap_params = {
    'n_neighbors': [15, 30],
    'min_dist': [0.1, 0.2, 0.3],
    'n_components': [2]
}

dbscan_params = {
    'eps': [0.2, 0.3, 0.4],
    'min_samples': [50, 100]
}

# Results storage
results = []
total_combinations = len(umap_params['n_neighbors']) * len(umap_params['min_dist']) * len(umap_params['n_components']) * len(dbscan_params['eps']) * len(dbscan_params['min_samples'])

print(f"Testing {total_combinations} parameter combinations for DBD patients...")

# Test all combinations
combination_count = 0
for n_neighbors in umap_params['n_neighbors']:
    for min_dist in umap_params['min_dist']:
        for n_components in umap_params['n_components']:
            for eps in dbscan_params['eps']:
                for min_samples in dbscan_params['min_samples']:
                    
                    combination_count += 1
                    print(f"\n--- Testing Combination {combination_count}/{total_combinations} ---")
                    print(f"UMAP: n_neighbors={n_neighbors}, min_dist={min_dist}, n_components={n_components}")
                    print(f"DBSCAN: eps={eps}, min_samples={min_samples}")
                    
                    try:
                        # Step 1: UMAP
                        umap_reducer = umap.UMAP(
                            n_components=n_components,
                            n_neighbors=n_neighbors,
                            min_dist=min_dist,
                            metric='euclidean',
                            random_state=42
                        )
                        
                        # Fit UMAP on DBD patients only
                        umap_embedding = umap_reducer.fit_transform(shap_values_dbd)
                        
                        # Step 2: DBSCAN
                        dbscan = DBSCAN(eps=eps, min_samples=min_samples)
                        clusters = dbscan.fit_predict(umap_embedding)
                        
                        # Calculate metrics
                        n_clusters = len(set(clusters)) - (1 if -1 in clusters else 0)
                        n_noise = list(clusters).count(-1)
                        
                        # Skip if too few clusters or too much noise
                        if n_clusters < 2 or n_noise > len(clusters) * 0.5:
                            print(f"❌ Skipped: {n_clusters} clusters, {n_noise} noise ({n_noise/len(clusters):.1%})")
                            continue
                        
                        # DEBUGGING: Cluster ID'leri kontrol et
                        print(f"\n=== CLUSTER ID DEBUGGING ===")
                        print(f"All cluster IDs: {sorted(set(clusters))}")
                        unique_clusters = sorted([c for c in set(clusters) if c != -1])
                        print(f"Unique clusters (excluding noise): {unique_clusters}")
                        print(f"Number of unique clusters: {len(unique_clusters)}")
                        
                        # Her cluster ID için sayıları kontrol edin
                        for cluster_id in sorted(set(clusters)):
                            count = np.sum(clusters == cluster_id)
                            if cluster_id == -1:
                                print(f"Noise (ID=-1): {count} patients")
                            else:
                                print(f"Cluster ID {cluster_id}: {count} patients")
                        
                        # Gerçek noise'u kontrol edin
                        noise_mask = clusters == -1
                        if np.sum(noise_mask) > 0:
                            print(f"Real noise points: {np.sum(noise_mask)}")
                            print(f"Noise mortality rate: {np.mean(y_dbd[noise_mask]):.3f}")
                        else:
                            print("No noise points found!")
                        
                        # Calculate clustering quality metrics - FIXED
                        if n_clusters > 1:
                            # Remove noise points for silhouette calculation
                            non_noise_mask = clusters != -1
                            if np.sum(non_noise_mask) > 1:
                                silhouette = silhouette_score(umap_embedding[non_noise_mask], 
                                                            clusters[non_noise_mask])
                                calinski = calinski_harabasz_score(umap_embedding[non_noise_mask], 
                                                                 clusters[non_noise_mask])
                            else:
                                silhouette = -1
                                calinski = 0
                        else:
                            silhouette = -1
                            calinski = 0
                        
                        # Calculate mortality risk separation for DBD patients
                        mortality_by_cluster = []
                        cluster_sizes = []
                        
                        for cluster_id in range(n_clusters):
                            cluster_mask = clusters == cluster_id
                            if np.sum(cluster_mask) > 0:
                                cluster_mortality = np.mean(y_dbd[cluster_mask])
                                cluster_size = np.sum(cluster_mask)
                                mortality_by_cluster.append(cluster_mortality)
                                cluster_sizes.append(cluster_size)
                        
                        # Calculate mortality risk variance and range
                        mortality_variance = np.var(mortality_by_cluster) if len(mortality_by_cluster) > 1 else 0
                        mortality_range = max(mortality_by_cluster) - min(mortality_by_cluster) if len(mortality_by_cluster) > 1 else 0
                        
                        # Print results
                        print(f"✅ Results: {n_clusters} clusters, {n_noise} noise ({n_noise/len(clusters):.1%})")
                        print(f"   Silhouette: {silhouette:.3f}")
                        print(f"   Calinski-Harabasz: {calinski:.2f}")
                        print(f"   Mortality Range: {mortality_range:.3f}")
                        print(f"   Mortality Rates: {[f'{x:.3f}' for x in mortality_by_cluster]}")
                        print(f"   Cluster Sizes: {cluster_sizes}")
                        
                        # Create visualization for this combination - NOISE SEPARATED
                        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
                        fig.suptitle(f'DBD Patients - Combination {combination_count}: UMAP({n_neighbors},{min_dist},{n_components}) + DBSCAN({eps},{min_samples})', 
                                   fontsize=16, fontweight='bold')
                        
                        # Get unique clusters (excluding noise points with -1)
                        n_unique_clusters = len(unique_clusters)
                        
                        # Plot 1: UMAP embedding with clusters - NOISE SEPARATED
                        # Create custom colormap that excludes noise
                        colors = cm.get_cmap('tab20', n_unique_clusters)
                        
                        # Plot clusters (non-noise points)
                        non_noise_mask = clusters != -1
                        scatter = axes[0,0].scatter(umap_embedding[non_noise_mask, 0], umap_embedding[non_noise_mask, 1], 
                                                   c=clusters[non_noise_mask], cmap=colors, alpha=0.7, s=20)
                        
                        # Plot noise points separately in black
                        if np.sum(noise_mask) > 0:
                            axes[0,0].scatter(umap_embedding[noise_mask, 0], umap_embedding[noise_mask, 1], 
                                             c='black', alpha=0.7, s=20, label='Noise')
                        
                        axes[0,0].set_title('UMAP Embedding of SHAP Values (Clustered) - DBD Only', fontsize=14, fontweight='bold')
                        axes[0,0].set_xlabel('UMAP 1')
                        axes[0,0].set_ylabel('UMAP 2')
                        
                        # Create custom colorbar with CORRECT labels - ONLY FOR CLUSTERS
                        cbar = plt.colorbar(scatter, ax=axes[0,0])
                        cbar.set_label('Cluster ID', rotation=270, labelpad=15)
                        # Set colorbar ticks to show REAL cluster IDs (excluding noise)
                        cbar.set_ticks(unique_clusters)
                        cbar.set_ticklabels([str(cluster_id) for cluster_id in unique_clusters])
                        # Adjust tick label position to be right next to colors
                        cbar.ax.tick_params(axis='y', which='major', pad=5)
                        
                        # Add legend for noise
                        if np.sum(noise_mask) > 0:
                            axes[0,0].legend(loc='upper right')
                        
                        # Plot 2: UMAP with outcome colors - NOISE SEPARATED
                        # Create color array for all points
                        colors_outcome = ['red' if outcome == 1 else 'blue' for outcome in y_dbd]
                        
                        # Plot clusters (non-noise points)
                        axes[0,1].scatter(umap_embedding[non_noise_mask, 0], umap_embedding[non_noise_mask, 1], 
                                        c=[colors_outcome[i] for i in range(len(colors_outcome)) if clusters[i] != -1], 
                                        alpha=0.7, s=20)
                        
                        # Plot noise points separately in black
                        if np.sum(noise_mask) > 0:
                            axes[0,1].scatter(umap_embedding[noise_mask, 0], umap_embedding[noise_mask, 1], 
                                             c='black', alpha=0.7, s=20, label='Noise')
                        
                        axes[0,1].set_title('UMAP Embedding Colored by Outcome - DBD Only', fontsize=14, fontweight='bold')
                        axes[0,1].set_xlabel('UMAP 1')
                        axes[0,1].set_ylabel('UMAP 2')
                        
                        # Add legend for noise
                        if np.sum(noise_mask) > 0:
                            axes[0,1].legend(loc='upper right')
                        
                        # Plot 3: Cluster size distribution - EXACT SAME AS WORKING CODE
                        cluster_counts = np.bincount(clusters + 1)  # +1 to handle -1 noise points
                        # Remove noise (index 0) and create labels starting from 1
                        cluster_counts_no_noise = cluster_counts[1:]  # Remove noise
                        cluster_labels = [f'Cluster {i+1}' for i in range(len(cluster_counts_no_noise))]
                        
                        axes[1,0].bar(cluster_labels, cluster_counts_no_noise, color='skyblue', alpha=0.7)
                        axes[1,0].set_title('Cluster Size Distribution - DBD Only', fontsize=14, fontweight='bold')
                        axes[1,0].set_xlabel('Cluster ID')
                        axes[1,0].set_ylabel('Number of Patients')
                        axes[1,0].tick_params(axis='x', rotation=0)
                        axes[1,0].grid(True, alpha=0.3)
                        
                        # Set x-axis ticks to show REAL cluster IDs
                        axes[1,0].set_xticks(range(len(cluster_counts_no_noise)))
                        axes[1,0].set_xticklabels([str(unique_clusters[i]) for i in range(len(cluster_counts_no_noise))])
                        
                        # Add value labels on bars
                        for i, v in enumerate(cluster_counts_no_noise):
                            axes[1,0].text(i, v + max(cluster_counts_no_noise)*0.01, str(v), ha='center', va='bottom', fontweight='bold')
                        
                        # Plot 4: Outcome distribution by cluster - EXACT SAME AS WORKING CODE
                        cluster_outcomes = []
                        cluster_ids = []
                        for i, cluster_id in enumerate(unique_clusters):
                            cluster_mask = clusters == cluster_id
                            if np.sum(cluster_mask) > 0:
                                cluster_outcome_rate = np.mean(y_dbd[cluster_mask])
                                cluster_outcomes.append(cluster_outcome_rate)
                                cluster_ids.append(cluster_id)
                        
                        axes[1,1].bar(cluster_ids, cluster_outcomes, color='lightcoral', alpha=0.7)
                        axes[1,1].set_title('Mortality Rate by Cluster - DBD Only', fontsize=14, fontweight='bold')
                        axes[1,1].set_xlabel('Cluster ID')
                        axes[1,1].set_ylabel('Mortality Rate')
                        axes[1,1].grid(True, alpha=0.3)
                        
                        # Set x-axis ticks to show REAL cluster IDs
                        axes[1,1].set_xticks(cluster_ids)
                        axes[1,1].set_xticklabels(cluster_ids)
                        
                        # Add value labels on bars
                        for i, v in enumerate(cluster_outcomes):
                            axes[1,1].text(cluster_ids[i], v + max(cluster_outcomes)*0.01, f'{v:.3f}',
                                           ha='center', va='bottom', fontweight='bold', fontsize=9)
                        
                        plt.tight_layout()
                        plt.show()
                        
                        # Store results
                        result = {
                            'combination_id': combination_count,
                            'n_neighbors': n_neighbors,
                            'min_dist': min_dist,
                            'n_components': n_components,
                            'eps': eps,
                            'min_samples': min_samples,
                            'n_clusters': n_clusters,
                            'n_noise': n_noise,
                            'noise_ratio': n_noise / len(clusters),
                            'silhouette_score': silhouette,
                            'calinski_harabasz_score': calinski,
                            'mortality_variance': mortality_variance,
                            'mortality_range': mortality_range,
                            'mortality_by_cluster': mortality_by_cluster,
                            'cluster_sizes': cluster_sizes,
                            'min_cluster_size': min(cluster_sizes) if cluster_sizes else 0,
                            'max_cluster_size': max(cluster_sizes) if cluster_sizes else 0
                        }
                        
                        results.append(result)
                        
                    except Exception as e:
                        print(f"❌ Error: {e}")
                        import traceback
                        traceback.print_exc()
                        continue

print(f"\n=== OPTIMIZATION COMPLETE FOR DBD PATIENTS ===")
print(f"Completed testing. Found {len(results)} valid parameter combinations.")

# Convert to DataFrame for analysis
results_df = pd.DataFrame(results)

if len(results_df) > 0:
    print(f"\n=== TOP 10 PARAMETER COMBINATIONS BY MORTALITY RANGE (DBD ONLY) ===")
    top_results = results_df.nlargest(10, 'mortality_range')
    
    for i, (_, row) in enumerate(top_results.iterrows()):
        print(f"\n{i+1}. Combination {row['combination_id']}: Mortality Range = {row['mortality_range']:.3f}")
        print(f"   UMAP: n_neighbors={row['n_neighbors']}, min_dist={row['min_dist']}, n_components={row['n_components']}")
        print(f"   DBSCAN: eps={row['eps']}, min_samples={row['min_samples']}")
        print(f"   Clusters: {row['n_clusters']}, Noise: {row['n_noise']} ({row['noise_ratio']:.1%})")
        print(f"   Silhouette: {row['silhouette_score']:.3f}")
        print(f"   CH Score: {row['calinski_harabasz_score']:.2f}")
        print(f"   Mortality rates: {[f'{x:.3f}' for x in row['mortality_by_cluster']]}")
        print(f"   Cluster sizes: {row['cluster_sizes']}")
        print(f"   Size range: {row['min_cluster_size']}-{row['max_cluster_size']}")
    
    # Create summary visualization
    print(f"\n=== CREATING SUMMARY VISUALIZATION FOR DBD PATIENTS ===")
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Parameter Optimization Summary - DBD Patients Only', fontsize=16, fontweight='bold')
    
    # Plot 1: Mortality Range vs Silhouette Score
    scatter = axes[0,0].scatter(results_df['mortality_range'], results_df['silhouette_score'], 
                               c=results_df['n_clusters'], cmap='viridis', alpha=0.7, s=100)
    axes[0,0].set_xlabel('Mortality Range')
    axes[0,0].set_ylabel('Silhouette Score')
    axes[0,0].set_title('Mortality Range vs Silhouette Score (DBD Only)')
    plt.colorbar(scatter, ax=axes[0,0], label='Number of Clusters')
    
    # Plot 2: Parameter combinations heatmap
    param_matrix = results_df.pivot_table(values='mortality_range', 
                                        index='eps', 
                                        columns='min_samples', 
                                        aggfunc='mean')
    sns.heatmap(param_matrix, annot=True, fmt='.3f', cmap='YlOrRd', ax=axes[0,1])
    axes[0,1].set_title('Mortality Range by DBSCAN Parameters (DBD Only)')
    
    # Plot 3: Cluster count distribution
    axes[1,0].hist(results_df['n_clusters'], bins=range(1, results_df['n_clusters'].max()+2), 
                   alpha=0.7, color='skyblue', edgecolor='black')
    axes[1,0].set_xlabel('Number of Clusters')
    axes[1,0].set_ylabel('Frequency')
    axes[1,0].set_title('Distribution of Cluster Counts (DBD Only)')
    
    # Plot 4: Top 5 combinations comparison
    top_5 = results_df.nlargest(5, 'mortality_range')
    x_pos = range(len(top_5))
    bars = axes[1,1].bar(x_pos, top_5['mortality_range'], alpha=0.7, color='lightcoral')
    axes[1,1].set_xlabel('Top 5 Combinations')
    axes[1,1].set_ylabel('Mortality Range')
    axes[1,1].set_title('Top 5 Parameter Combinations (DBD Only)')
    axes[1,1].set_xticks(x_pos)
    axes[1,1].set_xticklabels([f'#{int(row["combination_id"])}' for _, row in top_5.iterrows()], rotation=45)
    
    # Add value labels
    for i, v in enumerate(top_5['mortality_range']):
        axes[1,1].text(i, v + max(top_5['mortality_range'])*0.01, f'{v:.3f}',
                      ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n=== BEST PARAMETERS RECOMMENDATION FOR DBD PATIENTS ===")
    best_combo = results_df.loc[results_df['mortality_range'].idxmax()]
    print(f"Best Combination: #{int(best_combo['combination_id'])}")
    print(f"UMAP: n_neighbors={best_combo['n_neighbors']}, min_dist={best_combo['min_dist']}, n_components={best_combo['n_components']}")
    print(f"DBSCAN: eps={best_combo['eps']}, min_samples={best_combo['min_samples']}")
    print(f"Mortality Range: {best_combo['mortality_range']:.3f}")
    print(f"Silhouette Score: {best_combo['silhouette_score']:.3f}")
    print(f"CH Score: {best_combo['calinski_harabasz_score']:.2f}")
    print(f"Cluster Sizes: {best_combo['cluster_sizes']}")
    print(f"Mortality Rates: {[f'{x:.3f}' for x in best_combo['mortality_by_cluster']]}")
    
    # Save best parameters for future use
    best_params_dbd = {
        'n_neighbors': best_combo['n_neighbors'],
        'min_dist': best_combo['min_dist'],
        'n_components': best_combo['n_components'],
        'eps': best_combo['eps'],
        'min_samples': best_combo['min_samples']
    }
    
    print(f"\n=== BEST PARAMETERS FOR DBD PATIENTS ===")
    print(f"Best UMAP + DBSCAN parameters: {best_params_dbd}")
    
else:
    print("No valid parameter combinations found for DBD patients!")

In [ ]:
# Complete UMAP + DBSCAN Parameter Optimization for DBD Patients Only
# Tests all combinations and creates graphs for each

import itertools
from sklearn.metrics import silhouette_score, calinski_harabasz_score
from sklearn.cluster import DBSCAN
import umap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from scipy.stats import chi2_contingency

print("=== Complete UMAP + DBSCAN Parameter Optimization for DBD Patients ===")
print("Goal: Find parameters that create distinct high/low mortality risk clusters for DBD patients")
print("Each combination will be tested and visualized")

# Step 1: Combine SHAP values with outcome
print(f"\n=== Step 1: SHAP + Outcome Preparation ===")
# Normalize outcome to match SHAP scale
outcome_weight = 0.2  # Small weight to reduce dominance
shap_with_outcome = np.column_stack([shap_values, (y * outcome_weight)])
print(f"SHAP + Outcome shape: {shap_with_outcome.shape}")
print("Combined SHAP values with outcome as additional feature")

# Create DBD mask
mask_dbd = (x["DONOR_TYPE_BINARY"].to_numpy() == 0)
mask_dcd = (x["DONOR_TYPE_BINARY"].to_numpy() == 1)

print(f"Total patients: {len(x)}")
print(f"DBD patients: {np.sum(mask_dbd)}")
print(f"DCD patients: {np.sum(mask_dcd)}")

# Filter data for DBD patients only
shap_values_dbd = shap_with_outcome[mask_dbd]
y_dbd = y[mask_dbd]

print(f"DBD SHAP values shape: {shap_values_dbd.shape}")
print(f"DBD outcome shape: {y_dbd.shape}")
print(f"DBD mortality rate: {np.mean(y_dbd):.3f}")

# Define parameter ranges to test
umap_params = {
    'n_neighbors': [15, 30],
    'min_dist': [0.1, 0.2, 0.3],
    'n_components': [2]
}

dbscan_params = {
    'eps': [0.2, 0.3, 0.4],
    'min_samples': [50, 100]
}

# Results storage
results = []
total_combinations = len(umap_params['n_neighbors']) * len(umap_params['min_dist']) * len(umap_params['n_components']) * len(dbscan_params['eps']) * len(dbscan_params['min_samples'])

print(f"Testing {total_combinations} parameter combinations for DBD patients...")

# Test all combinations
combination_count = 0
for n_neighbors in umap_params['n_neighbors']:
    for min_dist in umap_params['min_dist']:
        for n_components in umap_params['n_components']:
            for eps in dbscan_params['eps']:
                for min_samples in dbscan_params['min_samples']:
                    
                    combination_count += 1
                    print(f"\n--- Testing Combination {combination_count}/{total_combinations} ---")
                    print(f"UMAP: n_neighbors={n_neighbors}, min_dist={min_dist}, n_components={n_components}")
                    print(f"DBSCAN: eps={eps}, min_samples={min_samples}")
                    
                    try:
                        # Step 1: UMAP
                        umap_reducer = umap.UMAP(
                            n_components=n_components,
                            n_neighbors=n_neighbors,
                            min_dist=min_dist,
                            metric='euclidean',
                            random_state=42
                        )
                        
                        # Fit UMAP on DBD patients only
                        umap_embedding = umap_reducer.fit_transform(shap_values_dbd)
                        
                        # Step 2: DBSCAN
                        dbscan = DBSCAN(eps=eps, min_samples=min_samples)
                        clusters = dbscan.fit_predict(umap_embedding)
                        
                        # Calculate metrics
                        n_clusters = len(set(clusters)) - (1 if -1 in clusters else 0)
                        n_noise = list(clusters).count(-1)
                        
                        # Skip if too few clusters or too much noise
                        if n_clusters < 2 or n_noise > len(clusters) * 0.5:
                            print(f"❌ Skipped: {n_clusters} clusters, {n_noise} noise ({n_noise/len(clusters):.1%})")
                            continue
                        
                        # DEBUGGING: Cluster ID'leri kontrol et
                        print(f"\n=== CLUSTER ID DEBUGGING ===")
                        print(f"All cluster IDs: {sorted(set(clusters))}")
                        unique_clusters = sorted([c for c in set(clusters) if c != -1])
                        print(f"Unique clusters (excluding noise): {unique_clusters}")
                        print(f"Number of unique clusters: {len(unique_clusters)}")
                        
                        # Her cluster ID için sayıları kontrol edin
                        for cluster_id in sorted(set(clusters)):
                            count = np.sum(clusters == cluster_id)
                            if cluster_id == -1:
                                print(f"Noise (ID=-1): {count} patients")
                            else:
                                print(f"Cluster ID {cluster_id}: {count} patients")
                        
                        # Gerçek noise'u kontrol edin
                        noise_mask = clusters == -1
                        if np.sum(noise_mask) > 0:
                            print(f"Real noise points: {np.sum(noise_mask)}")
                            print(f"Noise mortality rate: {np.mean(y_dbd[noise_mask]):.3f}")
                        else:
                            print("No noise points found!")
                        
                        # Calculate clustering quality metrics - FIXED
                        if n_clusters > 1:
                            # Remove noise points for silhouette calculation
                            non_noise_mask = clusters != -1
                            if np.sum(non_noise_mask) > 1:
                                silhouette = silhouette_score(umap_embedding[non_noise_mask], 
                                                            clusters[non_noise_mask])
                                calinski = calinski_harabasz_score(umap_embedding[non_noise_mask], 
                                                                 clusters[non_noise_mask])
                            else:
                                silhouette = -1
                                calinski = 0
                        else:
                            silhouette = -1
                            calinski = 0
                        
                        # Calculate mortality risk separation for DBD patients
                        mortality_by_cluster = []
                        cluster_sizes = []
                        
                        for cluster_id in range(n_clusters):
                            cluster_mask = clusters == cluster_id
                            if np.sum(cluster_mask) > 0:
                                cluster_mortality = np.mean(y_dbd[cluster_mask])
                                cluster_size = np.sum(cluster_mask)
                                mortality_by_cluster.append(cluster_mortality)
                                cluster_sizes.append(cluster_size)
                        
                        # Calculate mortality risk variance and range
                        mortality_variance = np.var(mortality_by_cluster) if len(mortality_by_cluster) > 1 else 0
                        mortality_range = max(mortality_by_cluster) - min(mortality_by_cluster) if len(mortality_by_cluster) > 1 else 0
                        
                        # Print results
                        print(f"✅ Results: {n_clusters} clusters, {n_noise} noise ({n_noise/len(clusters):.1%})")
                        print(f"   Silhouette: {silhouette:.3f}")
                        print(f"   Calinski-Harabasz: {calinski:.2f}")
                        print(f"   Mortality Range: {mortality_range:.3f}")
                        print(f"   Mortality Rates: {[f'{x:.3f}' for x in mortality_by_cluster]}")
                        print(f"   Cluster Sizes: {cluster_sizes}")
                        
                        # Create visualization for this combination - NOISE SEPARATED
                        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
                        fig.suptitle(f'DBD Patients - Combination {combination_count}: UMAP({n_neighbors},{min_dist},{n_components}) + DBSCAN({eps},{min_samples})', 
                                   fontsize=16, fontweight='bold')
                        
                        # Get unique clusters (excluding noise points with -1)
                        n_unique_clusters = len(unique_clusters)
                        
                        # Plot 1: UMAP embedding with clusters - NOISE SEPARATED
                        # Create custom colormap that excludes noise
                        colors = cm.get_cmap('tab20', n_unique_clusters)
                        
                        # Plot clusters (non-noise points)
                        non_noise_mask = clusters != -1
                        scatter = axes[0,0].scatter(umap_embedding[non_noise_mask, 0], umap_embedding[non_noise_mask, 1], 
                                                   c=clusters[non_noise_mask], cmap=colors, alpha=0.7, s=20)
                        
                        # Plot noise points separately in black
                        if np.sum(noise_mask) > 0:
                            axes[0,0].scatter(umap_embedding[noise_mask, 0], umap_embedding[noise_mask, 1], 
                                             c='black', alpha=0.7, s=20, label='Noise')
                        
                        axes[0,0].set_title('UMAP Embedding of SHAP Values (Clustered) - DBD Only', fontsize=14, fontweight='bold')
                        axes[0,0].set_xlabel('UMAP 1')
                        axes[0,0].set_ylabel('UMAP 2')
                        
                        # Create custom colorbar with CORRECT labels - ONLY FOR CLUSTERS
                        cbar = plt.colorbar(scatter, ax=axes[0,0])
                        cbar.set_label('Cluster ID', rotation=270, labelpad=15)
                        # Set colorbar ticks to show REAL cluster IDs (excluding noise)
                        cbar.set_ticks(unique_clusters)
                        cbar.set_ticklabels([str(cluster_id) for cluster_id in unique_clusters])
                        # Adjust tick label position to be right next to colors
                        cbar.ax.tick_params(axis='y', which='major', pad=5)
                        
                        # Add legend for noise
                        if np.sum(noise_mask) > 0:
                            axes[0,0].legend(loc='upper right')
                        
                        # Plot 2: UMAP with outcome colors - NOISE SEPARATED
                        # Create color array for all points
                        colors_outcome = ['red' if outcome == 1 else 'blue' for outcome in y_dbd]
                        
                        # Plot clusters (non-noise points)
                        axes[0,1].scatter(umap_embedding[non_noise_mask, 0], umap_embedding[non_noise_mask, 1], 
                                        c=[colors_outcome[i] for i in range(len(colors_outcome)) if clusters[i] != -1], 
                                        alpha=0.7, s=20)
                        
                        # Plot noise points separately in black
                        if np.sum(noise_mask) > 0:
                            axes[0,1].scatter(umap_embedding[noise_mask, 0], umap_embedding[noise_mask, 1], 
                                             c='black', alpha=0.7, s=20, label='Noise')
                        
                        axes[0,1].set_title('UMAP Embedding Colored by Outcome - DBD Only', fontsize=14, fontweight='bold')
                        axes[0,1].set_xlabel('UMAP 1')
                        axes[0,1].set_ylabel('UMAP 2')
                        
                        # Add legend for noise
                        if np.sum(noise_mask) > 0:
                            axes[0,1].legend(loc='upper right')
                        
                        # Plot 3: Cluster size distribution - EXACT SAME AS WORKING CODE
                        cluster_counts = np.bincount(clusters + 1)  # +1 to handle -1 noise points
                        # Remove noise (index 0) and create labels starting from 1
                        cluster_counts_no_noise = cluster_counts[1:]  # Remove noise
                        cluster_labels = [f'Cluster {i+1}' for i in range(len(cluster_counts_no_noise))]
                        
                        axes[1,0].bar(cluster_labels, cluster_counts_no_noise, color='skyblue', alpha=0.7)
                        axes[1,0].set_title('Cluster Size Distribution - DBD Only', fontsize=14, fontweight='bold')
                        axes[1,0].set_xlabel('Cluster ID')
                        axes[1,0].set_ylabel('Number of Patients')
                        axes[1,0].tick_params(axis='x', rotation=0)
                        axes[1,0].grid(True, alpha=0.3)
                        
                        # Set x-axis ticks to show REAL cluster IDs
                        axes[1,0].set_xticks(range(len(cluster_counts_no_noise)))
                        axes[1,0].set_xticklabels([str(unique_clusters[i]) for i in range(len(cluster_counts_no_noise))])
                        
                        # Add value labels on bars
                        for i, v in enumerate(cluster_counts_no_noise):
                            axes[1,0].text(i, v + max(cluster_counts_no_noise)*0.01, str(v), ha='center', va='bottom', fontweight='bold')
                        
                        # Plot 4: Outcome distribution by cluster - EXACT SAME AS WORKING CODE
                        cluster_outcomes = []
                        cluster_ids = []
                        for i, cluster_id in enumerate(unique_clusters):
                            cluster_mask = clusters == cluster_id
                            if np.sum(cluster_mask) > 0:
                                cluster_outcome_rate = np.mean(y_dbd[cluster_mask])
                                cluster_outcomes.append(cluster_outcome_rate)
                                cluster_ids.append(cluster_id)
                        
                        axes[1,1].bar(cluster_ids, cluster_outcomes, color='lightcoral', alpha=0.7)
                        axes[1,1].set_title('Mortality Rate by Cluster - DBD Only', fontsize=14, fontweight='bold')
                        axes[1,1].set_xlabel('Cluster ID')
                        axes[1,1].set_ylabel('Mortality Rate')
                        axes[1,1].grid(True, alpha=0.3)
                        
                        # Set x-axis ticks to show REAL cluster IDs
                        axes[1,1].set_xticks(cluster_ids)
                        axes[1,1].set_xticklabels(cluster_ids)
                        
                        # Add value labels on bars
                        for i, v in enumerate(cluster_outcomes):
                            axes[1,1].text(cluster_ids[i], v + max(cluster_outcomes)*0.01, f'{v:.3f}',
                                           ha='center', va='bottom', fontweight='bold', fontsize=9)
                        
                        plt.tight_layout()
                        plt.show()
                        
                        # Store results
                        result = {
                            'combination_id': combination_count,
                            'n_neighbors': n_neighbors,
                            'min_dist': min_dist,
                            'n_components': n_components,
                            'eps': eps,
                            'min_samples': min_samples,
                            'n_clusters': n_clusters,
                            'n_noise': n_noise,
                            'noise_ratio': n_noise / len(clusters),
                            'silhouette_score': silhouette,
                            'calinski_harabasz_score': calinski,
                            'mortality_variance': mortality_variance,
                            'mortality_range': mortality_range,
                            'mortality_by_cluster': mortality_by_cluster,
                            'cluster_sizes': cluster_sizes,
                            'min_cluster_size': min(cluster_sizes) if cluster_sizes else 0,
                            'max_cluster_size': max(cluster_sizes) if cluster_sizes else 0
                        }
                        
                        results.append(result)
                        
                    except Exception as e:
                        print(f"❌ Error: {e}")
                        import traceback
                        traceback.print_exc()
                        continue

print(f"\n=== OPTIMIZATION COMPLETE FOR DBD PATIENTS ===")
print(f"Completed testing. Found {len(results)} valid parameter combinations.")

# Convert to DataFrame for analysis
results_df = pd.DataFrame(results)

if len(results_df) > 0:
    print(f"\n=== TOP 10 PARAMETER COMBINATIONS BY MORTALITY RANGE (DBD ONLY) ===")
    top_results = results_df.nlargest(10, 'mortality_range')
    
    for i, (_, row) in enumerate(top_results.iterrows()):
        print(f"\n{i+1}. Combination {row['combination_id']}: Mortality Range = {row['mortality_range']:.3f}")
        print(f"   UMAP: n_neighbors={row['n_neighbors']}, min_dist={row['min_dist']}, n_components={row['n_components']}")
        print(f"   DBSCAN: eps={row['eps']}, min_samples={row['min_samples']}")
        print(f"   Clusters: {row['n_clusters']}, Noise: {row['n_noise']} ({row['noise_ratio']:.1%})")
        print(f"   Silhouette: {row['silhouette_score']:.3f}")
        print(f"   CH Score: {row['calinski_harabasz_score']:.2f}")
        print(f"   Mortality rates: {[f'{x:.3f}' for x in row['mortality_by_cluster']]}")
        print(f"   Cluster sizes: {row['cluster_sizes']}")
        print(f"   Size range: {row['min_cluster_size']}-{row['max_cluster_size']}")
    
    # Create summary visualization
    print(f"\n=== CREATING SUMMARY VISUALIZATION FOR DBD PATIENTS ===")
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Parameter Optimization Summary - DBD Patients Only', fontsize=16, fontweight='bold')
    
    # Plot 1: Mortality Range vs Silhouette Score
    scatter = axes[0,0].scatter(results_df['mortality_range'], results_df['silhouette_score'], 
                               c=results_df['n_clusters'], cmap='viridis', alpha=0.7, s=100)
    axes[0,0].set_xlabel('Mortality Range')
    axes[0,0].set_ylabel('Silhouette Score')
    axes[0,0].set_title('Mortality Range vs Silhouette Score (DBD Only)')
    plt.colorbar(scatter, ax=axes[0,0], label='Number of Clusters')
    
    # Plot 2: Parameter combinations heatmap
    param_matrix = results_df.pivot_table(values='mortality_range', 
                                        index='eps', 
                                        columns='min_samples', 
                                        aggfunc='mean')
    sns.heatmap(param_matrix, annot=True, fmt='.3f', cmap='YlOrRd', ax=axes[0,1])
    axes[0,1].set_title('Mortality Range by DBSCAN Parameters (DBD Only)')
    
    # Plot 3: Cluster count distribution
    axes[1,0].hist(results_df['n_clusters'], bins=range(1, results_df['n_clusters'].max()+2), 
                   alpha=0.7, color='skyblue', edgecolor='black')
    axes[1,0].set_xlabel('Number of Clusters')
    axes[1,0].set_ylabel('Frequency')
    axes[1,0].set_title('Distribution of Cluster Counts (DBD Only)')
    
    # Plot 4: Top 5 combinations comparison
    top_5 = results_df.nlargest(5, 'mortality_range')
    x_pos = range(len(top_5))
    bars = axes[1,1].bar(x_pos, top_5['mortality_range'], alpha=0.7, color='lightcoral')
    axes[1,1].set_xlabel('Top 5 Combinations')
    axes[1,1].set_ylabel('Mortality Range')
    axes[1,1].set_title('Top 5 Parameter Combinations (DBD Only)')
    axes[1,1].set_xticks(x_pos)
    axes[1,1].set_xticklabels([f'#{int(row["combination_id"])}' for _, row in top_5.iterrows()], rotation=45)
    
    # Add value labels
    for i, v in enumerate(top_5['mortality_range']):
        axes[1,1].text(i, v + max(top_5['mortality_range'])*0.01, f'{v:.3f}',
                      ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n=== BEST PARAMETERS RECOMMENDATION FOR DBD PATIENTS ===")
    best_combo = results_df.loc[results_df['mortality_range'].idxmax()]
    print(f"Best Combination: #{int(best_combo['combination_id'])}")
    print(f"UMAP: n_neighbors={best_combo['n_neighbors']}, min_dist={best_combo['min_dist']}, n_components={best_combo['n_components']}")
    print(f"DBSCAN: eps={best_combo['eps']}, min_samples={best_combo['min_samples']}")
    print(f"Mortality Range: {best_combo['mortality_range']:.3f}")
    print(f"Silhouette Score: {best_combo['silhouette_score']:.3f}")
    print(f"CH Score: {best_combo['calinski_harabasz_score']:.2f}")
    print(f"Cluster Sizes: {best_combo['cluster_sizes']}")
    print(f"Mortality Rates: {[f'{x:.3f}' for x in best_combo['mortality_by_cluster']]}")
    
    # Save best parameters for future use
    best_params_dbd = {
        'n_neighbors': best_combo['n_neighbors'],
        'min_dist': best_combo['min_dist'],
        'n_components': best_combo['n_components'],
        'eps': best_combo['eps'],
        'min_samples': best_combo['min_samples']
    }
    
    print(f"\n=== BEST PARAMETERS FOR DBD PATIENTS ===")
    print(f"Best UMAP + DBSCAN parameters: {best_params_dbd}")
    
else:
    print("No valid parameter combinations found for DBD patients!")

In [ ]:
# Complete UMAP + DBSCAN Parameter Optimization for DCD Patients Only
# Tests all combinations and creates graphs for each

import itertools
from sklearn.metrics import silhouette_score, calinski_harabasz_score
from sklearn.cluster import DBSCAN
import umap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from scipy.stats import chi2_contingency

print("=== Complete UMAP + DBSCAN Parameter Optimization for DCD Patients ===")
print("Goal: Find parameters that create distinct high/low mortality risk clusters for DCD patients")
print("Each combination will be tested and visualized")

# Create DCD mask
mask_dbd = (x["DONOR_TYPE_BINARY"].to_numpy() == 0)
mask_dcd = (x["DONOR_TYPE_BINARY"].to_numpy() == 1)

print(f"Total patients: {len(x)}")
print(f"DBD patients: {np.sum(mask_dbd)}")
print(f"DCD patients: {np.sum(mask_dcd)}")

# Filter data for DCD patients only
shap_values_dcd = shap_values[mask_dcd]
y_dcd = y[mask_dcd]

print(f"DCD SHAP values shape: {shap_values_dcd.shape}")
print(f"DCD outcome shape: {y_dcd.shape}")
print(f"DCD mortality rate: {np.mean(y_dcd):.3f}")

# Define parameter ranges to test
umap_params = {
    'n_neighbors': [10, 15, 20],  # Increased for more global structure
    'min_dist': [0.1, 0.2, 0.3],  # Increased for more spread
    'n_components': [2]
}

dbscan_params = {
    'eps': [0.3, 0.4, 0.5, 0.6],  # Larger eps for bigger clusters
    'min_samples': [10, 15, 20]  # Much larger min_samples
}
# Results storage
results = []
total_combinations = len(umap_params['n_neighbors']) * len(umap_params['min_dist']) * len(umap_params['n_components']) * len(dbscan_params['eps']) * len(dbscan_params['min_samples'])

print(f"Testing {total_combinations} parameter combinations for DCD patients...")

# Test all combinations
combination_count = 0
for n_neighbors in umap_params['n_neighbors']:
    for min_dist in umap_params['min_dist']:
        for n_components in umap_params['n_components']:
            for eps in dbscan_params['eps']:
                for min_samples in dbscan_params['min_samples']:
                    
                    combination_count += 1
                    print(f"\n--- Testing Combination {combination_count}/{total_combinations} ---")
                    print(f"UMAP: n_neighbors={n_neighbors}, min_dist={min_dist}, n_components={n_components}")
                    print(f"DBSCAN: eps={eps}, min_samples={min_samples}")
                    
                    try:
                        # Step 1: UMAP
                        umap_reducer = umap.UMAP(
                            n_components=n_components,
                            n_neighbors=n_neighbors,
                            min_dist=min_dist,
                            metric='euclidean',
                            random_state=42
                        )
                        
                        # Fit UMAP on DCD patients only
                        umap_embedding = umap_reducer.fit_transform(shap_values_dcd)
                        
                        # Step 2: DBSCAN
                        dbscan = DBSCAN(eps=eps, min_samples=min_samples)
                        clusters = dbscan.fit_predict(umap_embedding)
                        
                        # Calculate metrics
                        n_clusters = len(set(clusters)) - (1 if -1 in clusters else 0)
                        n_noise = list(clusters).count(-1)
                        
                        # Skip if too few clusters or too much noise
                        if n_clusters < 2 or n_noise > len(clusters) * 0.5:
                            print(f"❌ Skipped: {n_clusters} clusters, {n_noise} noise ({n_noise/len(clusters):.1%})")
                            continue
                        
                        # DEBUGGING: Cluster ID'leri kontrol et
                        print(f"\n=== CLUSTER ID DEBUGGING ===")
                        print(f"All cluster IDs: {sorted(set(clusters))}")
                        unique_clusters = sorted([c for c in set(clusters) if c != -1])
                        print(f"Unique clusters (excluding noise): {unique_clusters}")
                        print(f"Number of unique clusters: {len(unique_clusters)}")
                        
                        # Her cluster ID için sayıları kontrol edin
                        for cluster_id in sorted(set(clusters)):
                            count = np.sum(clusters == cluster_id)
                            if cluster_id == -1:
                                print(f"Noise (ID=-1): {count} patients")
                            else:
                                print(f"Cluster ID {cluster_id}: {count} patients")
                        
                        # Gerçek noise'u kontrol edin
                        noise_mask = clusters == -1
                        if np.sum(noise_mask) > 0:
                            print(f"Real noise points: {np.sum(noise_mask)}")
                            print(f"Noise mortality rate: {np.mean(y_dcd[noise_mask]):.3f}")
                        else:
                            print("No noise points found!")
                        
                        # Calculate clustering quality metrics - FIXED
                        if n_clusters > 1:
                            # Remove noise points for silhouette calculation
                            non_noise_mask = clusters != -1
                            if np.sum(non_noise_mask) > 1:
                                silhouette = silhouette_score(umap_embedding[non_noise_mask], 
                                                            clusters[non_noise_mask])
                                calinski = calinski_harabasz_score(umap_embedding[non_noise_mask], 
                                                                 clusters[non_noise_mask])
                            else:
                                silhouette = -1
                                calinski = 0
                        else:
                            silhouette = -1
                            calinski = 0
                        
                        # Calculate mortality risk separation for DCD patients
                        mortality_by_cluster = []
                        cluster_sizes = []
                        
                        for cluster_id in range(n_clusters):
                            cluster_mask = clusters == cluster_id
                            if np.sum(cluster_mask) > 0:
                                cluster_mortality = np.mean(y_dcd[cluster_mask])
                                cluster_size = np.sum(cluster_mask)
                                mortality_by_cluster.append(cluster_mortality)
                                cluster_sizes.append(cluster_size)
                        
                        # Calculate mortality risk variance and range
                        mortality_variance = np.var(mortality_by_cluster) if len(mortality_by_cluster) > 1 else 0
                        mortality_range = max(mortality_by_cluster) - min(mortality_by_cluster) if len(mortality_by_cluster) > 1 else 0
                        
                        # Print results
                        print(f"✅ Results: {n_clusters} clusters, {n_noise} noise ({n_noise/len(clusters):.1%})")
                        print(f"   Silhouette: {silhouette:.3f}")
                        print(f"   Calinski-Harabasz: {calinski:.2f}")
                        print(f"   Mortality Range: {mortality_range:.3f}")
                        print(f"   Mortality Rates: {[f'{x:.3f}' for x in mortality_by_cluster]}")
                        print(f"   Cluster Sizes: {cluster_sizes}")
                        
                        # Create visualization for this combination - NOISE SEPARATED
                        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
                        fig.suptitle(f'DCD Patients - Combination {combination_count}: UMAP({n_neighbors},{min_dist},{n_components}) + DBSCAN({eps},{min_samples})', 
                                   fontsize=16, fontweight='bold')
                        
                        # Get unique clusters (excluding noise points with -1)
                        n_unique_clusters = len(unique_clusters)
                        
                        # Plot 1: UMAP embedding with clusters - NOISE SEPARATED
                        # Create custom colormap that excludes noise
                        colors = cm.get_cmap('tab20', n_unique_clusters)
                        
                        # Plot clusters (non-noise points)
                        non_noise_mask = clusters != -1
                        scatter = axes[0,0].scatter(umap_embedding[non_noise_mask, 0], umap_embedding[non_noise_mask, 1], 
                                                   c=clusters[non_noise_mask], cmap=colors, alpha=0.7, s=20)
                        
                        # Plot noise points separately in black
                        if np.sum(noise_mask) > 0:
                            axes[0,0].scatter(umap_embedding[noise_mask, 0], umap_embedding[noise_mask, 1], 
                                             c='black', alpha=0.7, s=20, label='Noise')
                        
                        axes[0,0].set_title('UMAP Embedding of SHAP Values (Clustered) - DCD Only', fontsize=14, fontweight='bold')
                        axes[0,0].set_xlabel('UMAP 1')
                        axes[0,0].set_ylabel('UMAP 2')
                        
                        # Create custom colorbar with CORRECT labels - ONLY FOR CLUSTERS
                        cbar = plt.colorbar(scatter, ax=axes[0,0])
                        cbar.set_label('Cluster ID', rotation=270, labelpad=15)
                        # Set colorbar ticks to show REAL cluster IDs (excluding noise)
                        cbar.set_ticks(unique_clusters)
                        cbar.set_ticklabels([str(cluster_id) for cluster_id in unique_clusters])
                        # Adjust tick label position to be right next to colors
                        cbar.ax.tick_params(axis='y', which='major', pad=5)
                        
                        # Add legend for noise
                        if np.sum(noise_mask) > 0:
                            axes[0,0].legend(loc='upper right')
                        
                        # Plot 2: UMAP with outcome colors - NOISE SEPARATED
                        # Create color array for all points
                        colors_outcome = ['red' if outcome == 1 else 'blue' for outcome in y_dcd]
                        
                        # Plot clusters (non-noise points)
                        axes[0,1].scatter(umap_embedding[non_noise_mask, 0], umap_embedding[non_noise_mask, 1], 
                                        c=[colors_outcome[i] for i in range(len(colors_outcome)) if clusters[i] != -1], 
                                        alpha=0.7, s=20)
                        
                        # Plot noise points separately in black
                        if np.sum(noise_mask) > 0:
                            axes[0,1].scatter(umap_embedding[noise_mask, 0], umap_embedding[noise_mask, 1], 
                                             c='black', alpha=0.7, s=20, label='Noise')
                        
                        axes[0,1].set_title('UMAP Embedding Colored by Outcome - DCD Only', fontsize=14, fontweight='bold')
                        axes[0,1].set_xlabel('UMAP 1')
                        axes[0,1].set_ylabel('UMAP 2')
                        
                        # Add legend for noise
                        if np.sum(noise_mask) > 0:
                            axes[0,1].legend(loc='upper right')
                        
                        # Plot 3: Cluster size distribution - EXACT SAME AS WORKING CODE
                        cluster_counts = np.bincount(clusters + 1)  # +1 to handle -1 noise points
                        # Remove noise (index 0) and create labels starting from 1
                        cluster_counts_no_noise = cluster_counts[1:]  # Remove noise
                        cluster_labels = [f'Cluster {i+1}' for i in range(len(cluster_counts_no_noise))]
                        
                        axes[1,0].bar(cluster_labels, cluster_counts_no_noise, color='skyblue', alpha=0.7)
                        axes[1,0].set_title('Cluster Size Distribution - DCD Only', fontsize=14, fontweight='bold')
                        axes[1,0].set_xlabel('Cluster ID')
                        axes[1,0].set_ylabel('Number of Patients')
                        axes[1,0].tick_params(axis='x', rotation=0)
                        axes[1,0].grid(True, alpha=0.3)
                        
                        # Set x-axis ticks to show REAL cluster IDs
                        axes[1,0].set_xticks(range(len(cluster_counts_no_noise)))
                        axes[1,0].set_xticklabels([str(unique_clusters[i]) for i in range(len(cluster_counts_no_noise))])
                        
                        # Add value labels on bars
                        for i, v in enumerate(cluster_counts_no_noise):
                            axes[1,0].text(i, v + max(cluster_counts_no_noise)*0.01, str(v), ha='center', va='bottom', fontweight='bold')
                        
                        # Plot 4: Outcome distribution by cluster - EXACT SAME AS WORKING CODE
                        cluster_outcomes = []
                        cluster_ids = []
                        for i, cluster_id in enumerate(unique_clusters):
                            cluster_mask = clusters == cluster_id
                            if np.sum(cluster_mask) > 0:
                                cluster_outcome_rate = np.mean(y_dcd[cluster_mask])
                                cluster_outcomes.append(cluster_outcome_rate)
                                cluster_ids.append(cluster_id)
                        
                        axes[1,1].bar(cluster_ids, cluster_outcomes, color='lightcoral', alpha=0.7)
                        axes[1,1].set_title('Mortality Rate by Cluster - DCD Only', fontsize=14, fontweight='bold')
                        axes[1,1].set_xlabel('Cluster ID')
                        axes[1,1].set_ylabel('Mortality Rate')
                        axes[1,1].grid(True, alpha=0.3)
                        
                        # Set x-axis ticks to show REAL cluster IDs
                        axes[1,1].set_xticks(cluster_ids)
                        axes[1,1].set_xticklabels(cluster_ids)
                        
                        # Add value labels on bars
                        for i, v in enumerate(cluster_outcomes):
                            axes[1,1].text(cluster_ids[i], v + max(cluster_outcomes)*0.01, f'{v:.3f}',
                                           ha='center', va='bottom', fontweight='bold', fontsize=9)
                        
                        plt.tight_layout()
                        plt.show()
                        
                        # Store results
                        result = {
                            'combination_id': combination_count,
                            'n_neighbors': n_neighbors,
                            'min_dist': min_dist,
                            'n_components': n_components,
                            'eps': eps,
                            'min_samples': min_samples,
                            'n_clusters': n_clusters,
                            'n_noise': n_noise,
                            'noise_ratio': n_noise / len(clusters),
                            'silhouette_score': silhouette,
                            'calinski_harabasz_score': calinski,
                            'mortality_variance': mortality_variance,
                            'mortality_range': mortality_range,
                            'mortality_by_cluster': mortality_by_cluster,
                            'cluster_sizes': cluster_sizes,
                            'min_cluster_size': min(cluster_sizes) if cluster_sizes else 0,
                            'max_cluster_size': max(cluster_sizes) if cluster_sizes else 0
                        }
                        
                        results.append(result)
                        
                    except Exception as e:
                        print(f"❌ Error: {e}")
                        import traceback
                        traceback.print_exc()
                        continue

print(f"\n=== OPTIMIZATION COMPLETE FOR DCD PATIENTS ===")
print(f"Completed testing. Found {len(results)} valid parameter combinations.")

# Convert to DataFrame for analysis
results_df = pd.DataFrame(results)

if len(results_df) > 0:
    print(f"\n=== TOP 10 PARAMETER COMBINATIONS BY MORTALITY RANGE (DCD ONLY) ===")
    top_results = results_df.nlargest(10, 'mortality_range')
    
    for i, (_, row) in enumerate(top_results.iterrows()):
        print(f"\n{i+1}. Combination {row['combination_id']}: Mortality Range = {row['mortality_range']:.3f}")
        print(f"   UMAP: n_neighbors={row['n_neighbors']}, min_dist={row['min_dist']}, n_components={row['n_components']}")
        print(f"   DBSCAN: eps={row['eps']}, min_samples={row['min_samples']}")
        print(f"   Clusters: {row['n_clusters']}, Noise: {row['n_noise']} ({row['noise_ratio']:.1%})")
        print(f"   Silhouette: {row['silhouette_score']:.3f}")
        print(f"   CH Score: {row['calinski_harabasz_score']:.2f}")
        print(f"   Mortality rates: {[f'{x:.3f}' for x in row['mortality_by_cluster']]}")
        print(f"   Cluster sizes: {row['cluster_sizes']}")
        print(f"   Size range: {row['min_cluster_size']}-{row['max_cluster_size']}")
    
    # Create summary visualization
    print(f"\n=== CREATING SUMMARY VISUALIZATION FOR DCD PATIENTS ===")
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Parameter Optimization Summary - DCD Patients Only', fontsize=16, fontweight='bold')
    
    # Plot 1: Mortality Range vs Silhouette Score
    scatter = axes[0,0].scatter(results_df['mortality_range'], results_df['silhouette_score'], 
                               c=results_df['n_clusters'], cmap='viridis', alpha=0.7, s=100)
    axes[0,0].set_xlabel('Mortality Range')
    axes[0,0].set_ylabel('Silhouette Score')
    axes[0,0].set_title('Mortality Range vs Silhouette Score (DCD Only)')
    plt.colorbar(scatter, ax=axes[0,0], label='Number of Clusters')
    
    # Plot 2: Parameter combinations heatmap
    param_matrix = results_df.pivot_table(values='mortality_range', 
                                        index='eps', 
                                        columns='min_samples', 
                                        aggfunc='mean')
    sns.heatmap(param_matrix, annot=True, fmt='.3f', cmap='YlOrRd', ax=axes[0,1])
    axes[0,1].set_title('Mortality Range by DBSCAN Parameters (DCD Only)')
    
    # Plot 3: Cluster count distribution
    axes[1,0].hist(results_df['n_clusters'], bins=range(1, results_df['n_clusters'].max()+2), 
                   alpha=0.7, color='skyblue', edgecolor='black')
    axes[1,0].set_xlabel('Number of Clusters')
    axes[1,0].set_ylabel('Frequency')
    axes[1,0].set_title('Distribution of Cluster Counts (DCD Only)')
    
    # Plot 4: Top 5 combinations comparison
    top_5 = results_df.nlargest(5, 'mortality_range')
    x_pos = range(len(top_5))
    bars = axes[1,1].bar(x_pos, top_5['mortality_range'], alpha=0.7, color='lightcoral')
    axes[1,1].set_xlabel('Top 5 Combinations')
    axes[1,1].set_ylabel('Mortality Range')
    axes[1,1].set_title('Top 5 Parameter Combinations (DCD Only)')
    axes[1,1].set_xticks(x_pos)
    axes[1,1].set_xticklabels([f'#{int(row["combination_id"])}' for _, row in top_5.iterrows()], rotation=45)
    
    # Add value labels
    for i, v in enumerate(top_5['mortality_range']):
        axes[1,1].text(i, v + max(top_5['mortality_range'])*0.01, f'{v:.3f}',
                      ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n=== BEST PARAMETERS RECOMMENDATION FOR DCD PATIENTS ===")
    best_combo = results_df.loc[results_df['mortality_range'].idxmax()]
    print(f"Best Combination: #{int(best_combo['combination_id'])}")
    print(f"UMAP: n_neighbors={best_combo['n_neighbors']}, min_dist={best_combo['min_dist']}, n_components={best_combo['n_components']}")
    print(f"DBSCAN: eps={best_combo['eps']}, min_samples={best_combo['min_samples']}")
    print(f"Mortality Range: {best_combo['mortality_range']:.3f}")
    print(f"Silhouette Score: {best_combo['silhouette_score']:.3f}")
    print(f"CH Score: {best_combo['calinski_harabasz_score']:.2f}")
    print(f"Cluster Sizes: {best_combo['cluster_sizes']}")
    print(f"Mortality Rates: {[f'{x:.3f}' for x in best_combo['mortality_by_cluster']]}")
    
    # Save best parameters for future use
    best_params_dcd = {
        'n_neighbors': best_combo['n_neighbors'],
        'min_dist': best_combo['min_dist'],
        'n_components': best_combo['n_components'],
        'eps': best_combo['eps'],
        'min_samples': best_combo['min_samples']
    }
    
    print(f"\n=== BEST PARAMETERS FOR DCD PATIENTS ===")
    print(f"Best UMAP + DBSCAN parameters: {best_params_dcd}")
    
else:
    print("No valid parameter combinations found for DCD patients!")

In [ ]:
# Complete UMAP + DBSCAN Parameter Optimization for DCD Patients Only
# Tests all combinations and creates graphs for each

import itertools
from sklearn.metrics import silhouette_score, calinski_harabasz_score
from sklearn.cluster import DBSCAN
import umap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from scipy.stats import chi2_contingency

print("=== Complete UMAP + DBSCAN Parameter Optimization for DCD Patients ===")
print("Goal: Find parameters that create distinct high/low mortality risk clusters for DCD patients")
print("Each combination will be tested and visualized")

# Step 1: Combine SHAP values with outcome
print(f"\n=== Step 1: SHAP + Outcome Preparation ===")
# Normalize outcome to match SHAP scale
outcome_weight = 0.2  # Small weight to reduce dominance
shap_with_outcome = np.column_stack([shap_values, (y * outcome_weight)])
print(f"SHAP + Outcome shape: {shap_with_outcome.shape}")
print("Combined SHAP values with outcome as additional feature")

# Create DCD mask
mask_dbd = (x["DONOR_TYPE_BINARY"].to_numpy() == 0)
mask_dcd = (x["DONOR_TYPE_BINARY"].to_numpy() == 1)

print(f"Total patients: {len(x)}")
print(f"DBD patients: {np.sum(mask_dbd)}")
print(f"DCD patients: {np.sum(mask_dcd)}")

# Filter data for DCD patients only
shap_values_dcd = shap_with_outcome[mask_dcd]
y_dcd = y[mask_dcd]

print(f"DCD SHAP values shape: {shap_values_dcd.shape}")
print(f"DCD outcome shape: {y_dcd.shape}")
print(f"DCD mortality rate: {np.mean(y_dcd):.3f}")

# Define parameter ranges to test - ADJUSTED FOR DCD (smaller dataset)
umap_params = {
    'n_neighbors': [10, 15, 20],  # Increased for more global structure
    'min_dist': [0.1, 0.2, 0.3],  # Increased for more spread
    'n_components': [2]
}

dbscan_params = {
    'eps': [0.3, 0.4, 0.5, 0.6],  # Larger eps for bigger clusters
    'min_samples': [10, 15, 20]  # Much larger min_samples
}

# Results storage
results = []
total_combinations = len(umap_params['n_neighbors']) * len(umap_params['min_dist']) * len(umap_params['n_components']) * len(dbscan_params['eps']) * len(dbscan_params['min_samples'])

print(f"Testing {total_combinations} parameter combinations for DCD patients...")

# Test all combinations
combination_count = 0
for n_neighbors in umap_params['n_neighbors']:
    for min_dist in umap_params['min_dist']:
        for n_components in umap_params['n_components']:
            for eps in dbscan_params['eps']:
                for min_samples in dbscan_params['min_samples']:
                    
                    combination_count += 1
                    print(f"\n--- Testing Combination {combination_count}/{total_combinations} ---")
                    print(f"UMAP: n_neighbors={n_neighbors}, min_dist={min_dist}, n_components={n_components}")
                    print(f"DBSCAN: eps={eps}, min_samples={min_samples}")
                    
                    try:
                        # Step 1: UMAP
                        umap_reducer = umap.UMAP(
                            n_components=n_components,
                            n_neighbors=n_neighbors,
                            min_dist=min_dist,
                            metric='euclidean',
                            random_state=42
                        )
                        
                        # Fit UMAP on DCD patients only
                        umap_embedding = umap_reducer.fit_transform(shap_values_dcd)
                        
                        # Step 2: DBSCAN
                        dbscan = DBSCAN(eps=eps, min_samples=min_samples)
                        clusters = dbscan.fit_predict(umap_embedding)
                        
                        # Calculate metrics
                        n_clusters = len(set(clusters)) - (1 if -1 in clusters else 0)
                        n_noise = list(clusters).count(-1)
                        
                        # Skip if too few clusters or too much noise - ADJUSTED THRESHOLDS
                        if n_clusters < 2 or n_noise > len(clusters) * 0.8:  # Allow up to 80% noise
                            print(f"❌ Skipped: {n_clusters} clusters, {n_noise} noise ({n_noise/len(clusters):.1%})")
                            continue
                        
                        # DEBUGGING: Cluster ID'leri kontrol et
                        print(f"\n=== CLUSTER ID DEBUGGING ===")
                        print(f"All cluster IDs: {sorted(set(clusters))}")
                        unique_clusters = sorted([c for c in set(clusters) if c != -1])
                        print(f"Unique clusters (excluding noise): {unique_clusters}")
                        print(f"Number of unique clusters: {len(unique_clusters)}")
                        
                        # Her cluster ID için sayıları kontrol edin
                        for cluster_id in sorted(set(clusters)):
                            count = np.sum(clusters == cluster_id)
                            if cluster_id == -1:
                                print(f"Noise (ID=-1): {count} patients")
                            else:
                                print(f"Cluster ID {cluster_id}: {count} patients")
                        
                        # Gerçek noise'u kontrol edin
                        noise_mask = clusters == -1
                        if np.sum(noise_mask) > 0:
                            print(f"Real noise points: {np.sum(noise_mask)}")
                            print(f"Noise mortality rate: {np.mean(y_dcd[noise_mask]):.3f}")
                        else:
                            print("No noise points found!")
                        
                        # Calculate clustering quality metrics - FIXED
                        if n_clusters > 1:
                            # Remove noise points for silhouette calculation
                            non_noise_mask = clusters != -1
                            if np.sum(non_noise_mask) > 1:
                                silhouette = silhouette_score(umap_embedding[non_noise_mask], 
                                                            clusters[non_noise_mask])
                                calinski = calinski_harabasz_score(umap_embedding[non_noise_mask], 
                                                                 clusters[non_noise_mask])
                            else:
                                silhouette = -1
                                calinski = 0
                        else:
                            silhouette = -1
                            calinski = 0
                        
                        # Calculate mortality risk separation for DCD patients
                        mortality_by_cluster = []
                        cluster_sizes = []
                        
                        for cluster_id in range(n_clusters):
                            cluster_mask = clusters == cluster_id
                            if np.sum(cluster_mask) > 0:
                                cluster_mortality = np.mean(y_dcd[cluster_mask])
                                cluster_size = np.sum(cluster_mask)
                                mortality_by_cluster.append(cluster_mortality)
                                cluster_sizes.append(cluster_size)
                        
                        # Calculate mortality risk variance and range
                        mortality_variance = np.var(mortality_by_cluster) if len(mortality_by_cluster) > 1 else 0
                        mortality_range = max(mortality_by_cluster) - min(mortality_by_cluster) if len(mortality_by_cluster) > 1 else 0
                        
                        # Print results
                        print(f"✅ Results: {n_clusters} clusters, {n_noise} noise ({n_noise/len(clusters):.1%})")
                        print(f"   Silhouette: {silhouette:.3f}")
                        print(f"   Calinski-Harabasz: {calinski:.2f}")
                        print(f"   Mortality Range: {mortality_range:.3f}")
                        print(f"   Mortality Rates: {[f'{x:.3f}' for x in mortality_by_cluster]}")
                        print(f"   Cluster Sizes: {cluster_sizes}")
                        
                        # Create visualization for this combination - NOISE SEPARATED
                        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
                        fig.suptitle(f'DCD Patients - Combination {combination_count}: UMAP({n_neighbors},{min_dist},{n_components}) + DBSCAN({eps},{min_samples})', 
                                   fontsize=16, fontweight='bold')
                        
                        # Get unique clusters (excluding noise points with -1)
                        n_unique_clusters = len(unique_clusters)
                        
                        # Plot 1: UMAP embedding with clusters - NOISE SEPARATED
                        # Create custom colormap that excludes noise
                        colors = cm.get_cmap('tab20', n_unique_clusters)
                        
                        # Plot clusters (non-noise points)
                        non_noise_mask = clusters != -1
                        scatter = axes[0,0].scatter(umap_embedding[non_noise_mask, 0], umap_embedding[non_noise_mask, 1], 
                                                   c=clusters[non_noise_mask], cmap=colors, alpha=0.7, s=20)
                        
                        # Plot noise points separately in black
                        if np.sum(noise_mask) > 0:
                            axes[0,0].scatter(umap_embedding[noise_mask, 0], umap_embedding[noise_mask, 1], 
                                             c='black', alpha=0.7, s=20, label='Noise')
                        
                        axes[0,0].set_title('UMAP Embedding of SHAP Values (Clustered) - DCD Only', fontsize=14, fontweight='bold')
                        axes[0,0].set_xlabel('UMAP 1')
                        axes[0,0].set_ylabel('UMAP 2')
                        
                        # Create custom colorbar with CORRECT labels - ONLY FOR CLUSTERS
                        cbar = plt.colorbar(scatter, ax=axes[0,0])
                        cbar.set_label('Cluster ID', rotation=270, labelpad=15)
                        # Set colorbar ticks to show REAL cluster IDs (excluding noise)
                        cbar.set_ticks(unique_clusters)
                        cbar.set_ticklabels([str(cluster_id) for cluster_id in unique_clusters])
                        # Adjust tick label position to be right next to colors
                        cbar.ax.tick_params(axis='y', which='major', pad=5)
                        
                        # Add legend for noise
                        if np.sum(noise_mask) > 0:
                            axes[0,0].legend(loc='upper right')
                        
                        # Plot 2: UMAP with outcome colors - NOISE SEPARATED
                        # Create color array for all points
                        colors_outcome = ['red' if outcome == 1 else 'blue' for outcome in y_dcd]
                        
                        # Plot clusters (non-noise points)
                        axes[0,1].scatter(umap_embedding[non_noise_mask, 0], umap_embedding[non_noise_mask, 1], 
                                        c=[colors_outcome[i] for i in range(len(colors_outcome)) if clusters[i] != -1], 
                                        alpha=0.7, s=20)
                        
                        # Plot noise points separately in black
                        if np.sum(noise_mask) > 0:
                            axes[0,1].scatter(umap_embedding[noise_mask, 0], umap_embedding[noise_mask, 1], 
                                             c='black', alpha=0.7, s=20, label='Noise')
                        
                        axes[0,1].set_title('UMAP Embedding Colored by Outcome - DCD Only', fontsize=14, fontweight='bold')
                        axes[0,1].set_xlabel('UMAP 1')
                        axes[0,1].set_ylabel('UMAP 2')
                        
                        # Add legend for noise
                        if np.sum(noise_mask) > 0:
                            axes[0,1].legend(loc='upper right')
                        
                        # Plot 3: Cluster size distribution - EXACT SAME AS WORKING CODE
                        cluster_counts = np.bincount(clusters + 1)  # +1 to handle -1 noise points
                        # Remove noise (index 0) and create labels starting from 1
                        cluster_counts_no_noise = cluster_counts[1:]  # Remove noise
                        cluster_labels = [f'Cluster {i+1}' for i in range(len(cluster_counts_no_noise))]
                        
                        axes[1,0].bar(cluster_labels, cluster_counts_no_noise, color='skyblue', alpha=0.7)
                        axes[1,0].set_title('Cluster Size Distribution - DCD Only', fontsize=14, fontweight='bold')
                        axes[1,0].set_xlabel('Cluster ID')
                        axes[1,0].set_ylabel('Number of Patients')
                        axes[1,0].tick_params(axis='x', rotation=0)
                        axes[1,0].grid(True, alpha=0.3)
                        
                        # Set x-axis ticks to show REAL cluster IDs
                        axes[1,0].set_xticks(range(len(cluster_counts_no_noise)))
                        axes[1,0].set_xticklabels([str(unique_clusters[i]) for i in range(len(cluster_counts_no_noise))])
                        
                        # Add value labels on bars
                        for i, v in enumerate(cluster_counts_no_noise):
                            axes[1,0].text(i, v + max(cluster_counts_no_noise)*0.01, str(v), ha='center', va='bottom', fontweight='bold')
                        
                        # Plot 4: Outcome distribution by cluster - EXACT SAME AS WORKING CODE
                        cluster_outcomes = []
                        cluster_ids = []
                        for i, cluster_id in enumerate(unique_clusters):
                            cluster_mask = clusters == cluster_id
                            if np.sum(cluster_mask) > 0:
                                cluster_outcome_rate = np.mean(y_dcd[cluster_mask])
                                cluster_outcomes.append(cluster_outcome_rate)
                                cluster_ids.append(cluster_id)
                        
                        axes[1,1].bar(cluster_ids, cluster_outcomes, color='lightcoral', alpha=0.7)
                        axes[1,1].set_title('Mortality Rate by Cluster - DCD Only', fontsize=14, fontweight='bold')
                        axes[1,1].set_xlabel('Cluster ID')
                        axes[1,1].set_ylabel('Mortality Rate')
                        axes[1,1].grid(True, alpha=0.3)
                        
                        # Set x-axis ticks to show REAL cluster IDs
                        axes[1,1].set_xticks(cluster_ids)
                        axes[1,1].set_xticklabels(cluster_ids)
                        
                        # Add value labels on bars
                        for i, v in enumerate(cluster_outcomes):
                            axes[1,1].text(cluster_ids[i], v + max(cluster_outcomes)*0.01, f'{v:.3f}',
                                           ha='center', va='bottom', fontweight='bold', fontsize=9)
                        
                        plt.tight_layout()
                        plt.show()
                        
                        # Store results
                        result = {
                            'combination_id': combination_count,
                            'n_neighbors': n_neighbors,
                            'min_dist': min_dist,
                            'n_components': n_components,
                            'eps': eps,
                            'min_samples': min_samples,
                            'n_clusters': n_clusters,
                            'n_noise': n_noise,
                            'noise_ratio': n_noise / len(clusters),
                            'silhouette_score': silhouette,
                            'calinski_harabasz_score': calinski,
                            'mortality_variance': mortality_variance,
                            'mortality_range': mortality_range,
                            'mortality_by_cluster': mortality_by_cluster,
                            'cluster_sizes': cluster_sizes,
                            'min_cluster_size': min(cluster_sizes) if cluster_sizes else 0,
                            'max_cluster_size': max(cluster_sizes) if cluster_sizes else 0
                        }
                        
                        results.append(result)
                        
                    except Exception as e:
                        print(f"❌ Error: {e}")
                        import traceback
                        traceback.print_exc()
                        continue

print(f"\n=== OPTIMIZATION COMPLETE FOR DCD PATIENTS ===")
print(f"Completed testing. Found {len(results)} valid parameter combinations.")

# Convert to DataFrame for analysis
results_df = pd.DataFrame(results)

if len(results_df) > 0:
    print(f"\n=== TOP 10 PARAMETER COMBINATIONS BY MORTALITY RANGE (DCD ONLY) ===")
    top_results = results_df.nlargest(10, 'mortality_range')
    
    for i, (_, row) in enumerate(top_results.iterrows()):
        print(f"\n{i+1}. Combination {row['combination_id']}: Mortality Range = {row['mortality_range']:.3f}")
        print(f"   UMAP: n_neighbors={row['n_neighbors']}, min_dist={row['min_dist']}, n_components={row['n_components']}")
        print(f"   DBSCAN: eps={row['eps']}, min_samples={row['min_samples']}")
        print(f"   Clusters: {row['n_clusters']}, Noise: {row['n_noise']} ({row['noise_ratio']:.1%})")
        print(f"   Silhouette: {row['silhouette_score']:.3f}")
        print(f"   CH Score: {row['calinski_harabasz_score']:.2f}")
        print(f"   Mortality rates: {[f'{x:.3f}' for x in row['mortality_by_cluster']]}")
        print(f"   Cluster sizes: {row['cluster_sizes']}")
        print(f"   Size range: {row['min_cluster_size']}-{row['max_cluster_size']}")
    
    # Create summary visualization
    print(f"\n=== CREATING SUMMARY VISUALIZATION FOR DCD PATIENTS ===")
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Parameter Optimization Summary - DCD Patients Only', fontsize=16, fontweight='bold')
    
    # Plot 1: Mortality Range vs Silhouette Score
    scatter = axes[0,0].scatter(results_df['mortality_range'], results_df['silhouette_score'], 
                               c=results_df['n_clusters'], cmap='viridis', alpha=0.7, s=100)
    axes[0,0].set_xlabel('Mortality Range')
    axes[0,0].set_ylabel('Silhouette Score')
    axes[0,0].set_title('Mortality Range vs Silhouette Score (DCD Only)')
    plt.colorbar(scatter, ax=axes[0,0], label='Number of Clusters')
    
    # Plot 2: Parameter combinations heatmap
    param_matrix = results_df.pivot_table(values='mortality_range', 
                                        index='eps', 
                                        columns='min_samples', 
                                        aggfunc='mean')
    sns.heatmap(param_matrix, annot=True, fmt='.3f', cmap='YlOrRd', ax=axes[0,1])
    axes[0,1].set_title('Mortality Range by DBSCAN Parameters (DCD Only)')
    
    # Plot 3: Cluster count distribution
    axes[1,0].hist(results_df['n_clusters'], bins=range(1, results_df['n_clusters'].max()+2), 
                   alpha=0.7, color='skyblue', edgecolor='black')
    axes[1,0].set_xlabel('Number of Clusters')
    axes[1,0].set_ylabel('Frequency')
    axes[1,0].set_title('Distribution of Cluster Counts (DCD Only)')
    
    # Plot 4: Top 5 combinations comparison
    top_5 = results_df.nlargest(5, 'mortality_range')
    x_pos = range(len(top_5))
    bars = axes[1,1].bar(x_pos, top_5['mortality_range'], alpha=0.7, color='lightcoral')
    axes[1,1].set_xlabel('Top 5 Combinations')
    axes[1,1].set_ylabel('Mortality Range')
    axes[1,1].set_title('Top 5 Parameter Combinations (DCD Only)')
    axes[1,1].set_xticks(x_pos)
    axes[1,1].set_xticklabels([f'#{int(row["combination_id"])}' for _, row in top_5.iterrows()], rotation=45)
    
    # Add value labels
    for i, v in enumerate(top_5['mortality_range']):
        axes[1,1].text(i, v + max(top_5['mortality_range'])*0.01, f'{v:.3f}',
                      ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n=== BEST PARAMETERS RECOMMENDATION FOR DCD PATIENTS ===")
    best_combo = results_df.loc[results_df['mortality_range'].idxmax()]
    print(f"Best Combination: #{int(best_combo['combination_id'])}")
    print(f"UMAP: n_neighbors={best_combo['n_neighbors']}, min_dist={best_combo['min_dist']}, n_components={best_combo['n_components']}")
    print(f"DBSCAN: eps={best_combo['eps']}, min_samples={best_combo['min_samples']}")
    print(f"Mortality Range: {best_combo['mortality_range']:.3f}")
    print(f"Silhouette Score: {best_combo['silhouette_score']:.3f}")
    print(f"CH Score: {best_combo['calinski_harabasz_score']:.2f}")
    print(f"Cluster Sizes: {best_combo['cluster_sizes']}")
    print(f"Mortality Rates: {[f'{x:.3f}' for x in best_combo['mortality_by_cluster']]}")
    
    # Save best parameters for future use
    best_params_dcd = {
        'n_neighbors': best_combo['n_neighbors'],
        'min_dist': best_combo['min_dist'],
        'n_components': best_combo['n_components'],
        'eps': best_combo['eps'],
        'min_samples': best_combo['min_samples']
    }
    
    print(f"\n=== BEST PARAMETERS FOR DCD PATIENTS ===")
    print(f"Best UMAP + DBSCAN parameters: {best_params_dcd}")
    
else:
    print("No valid parameter combinations found for DCD patients!")

In [ ]:
# SHAP + Outcome → UMAP → DBSCAN Clustering Analysis
import umap
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from scipy.stats import chi2_contingency
import numpy as np

print("=== SHAP + Outcome Values Analysis ===")
print(f"SHAP values shape: {shap_values.shape}")
print(f"Outcome shape: {y.shape}")
print(f"Feature names: {len(feature_names)}")

# Step 1: Combine SHAP values with outcome
print(f"\n=== Step 1: SHAP + Outcome Preparation ===")
# Normalize outcome to match SHAP scale
outcome_weight = 0.1  # Small weight to reduce dominance
shap_with_outcome = np.column_stack([shap_values, (y * outcome_weight)])
print(f"SHAP + Outcome shape: {shap_with_outcome.shape}")
print("Combined SHAP values with normalized outcome as additional feature")

# Step 2: UMAP Dimensionality Reduction
print(f"\n=== Step 2: UMAP Dimensionality Reduction ===")
umap_reducer = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric='euclidean',
    random_state=42
)

# Fit UMAP on SHAP + outcome values
shap_umap_with_outcome = umap_reducer.fit_transform(shap_with_outcome)
print(f"UMAP embedding shape: {shap_umap_with_outcome.shape}")

# Step 3: DBSCAN Clustering
print(f"\n=== Step 3: DBSCAN Clustering ===")
dbscan = DBSCAN(eps=0.5, min_samples=100)
clusters = dbscan.fit_predict(shap_umap_with_outcome)

n_clusters = len(set(clusters)) - (1 if -1 in clusters else 0)
n_noise = list(clusters).count(-1)

print(f"Number of clusters: {n_clusters}")
print(f"Number of noise points: {n_noise}")
print(f"Cluster distribution: {np.bincount(clusters + 1)}")

# Step 4: Visualization
print(f"\n=== Step 4: Visualization ===")
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Get unique clusters (excluding noise points with -1)
unique_clusters = sorted([c for c in set(clusters) if c != -1])
n_unique_clusters = len(unique_clusters)
print(f"Unique clusters (excluding noise): {unique_clusters}")
print(f"Number of unique clusters: {n_unique_clusters}")

# Plot 1: UMAP embedding with clusters
colors = cm.get_cmap('tab20', n_unique_clusters)

scatter = axes[0,0].scatter(shap_umap_with_outcome[:, 0], shap_umap_with_outcome[:, 1], c=clusters,
                           cmap=colors, alpha=0.7, s=20)
axes[0,0].set_title('UMAP Embedding of SHAP + Outcome Values (Clustered)', fontsize=14, fontweight='bold')
axes[0,0].set_xlabel('UMAP 1')
axes[0,0].set_ylabel('UMAP 2')

# Create custom colorbar with proper labels - show 1-N for each color
cbar = plt.colorbar(scatter, ax=axes[0,0])
cbar.set_label('Cluster ID', rotation=270, labelpad=15)
# Set colorbar ticks to show 1-N for each color
cbar.set_ticks(unique_clusters)
cbar.set_ticklabels([str(i+1) for i in range(len(unique_clusters))])
# Adjust tick label position to be right next to colors
cbar.ax.tick_params(axis='y', which='major', pad=5)

# Plot 2: UMAP with outcome colors
# Create color array for all points
colors = ['red' if outcome == 1 else 'blue' for outcome in y]
axes[0,1].scatter(shap_umap_with_outcome[:, 0], shap_umap_with_outcome[:, 1], c=colors, alpha=0.7, s=20)
axes[0,1].set_title('UMAP Embedding Colored by Outcome', fontsize=14, fontweight='bold')
axes[0,1].set_xlabel('UMAP 1')
axes[0,1].set_ylabel('UMAP 2')

# Plot 3: Cluster size distribution - NO NOISE, consistent labeling (1-N)
cluster_counts = np.bincount(clusters + 1)  # +1 to handle -1 noise points
# Remove noise (index 0) and create labels starting from 1
cluster_counts_no_noise = cluster_counts[1:]  # Remove noise
cluster_labels = [f'Cluster {i+1}' for i in range(len(cluster_counts_no_noise))]

axes[1,0].bar(cluster_labels, cluster_counts_no_noise, color='skyblue', alpha=0.7)
axes[1,0].set_title('Cluster Size Distribution', fontsize=14, fontweight='bold')
axes[1,0].set_xlabel('Cluster ID')
axes[1,0].set_ylabel('Number of Patients')
axes[1,0].tick_params(axis='x', rotation=0)  # Changed from 45 to 0
axes[1,0].grid(True, alpha=0.3)

# Set x-axis ticks to show 1,2,3...N instead of "Cluster 1", "Cluster 2"
axes[1,0].set_xticks(range(len(cluster_counts_no_noise)))
axes[1,0].set_xticklabels([str(i+1) for i in range(len(cluster_counts_no_noise))])

# Add value labels on bars
for i, v in enumerate(cluster_counts_no_noise):
    axes[1,0].text(i, v + max(cluster_counts_no_noise)*0.01, str(v), ha='center', va='bottom', fontweight='bold')

# Plot 4: Outcome distribution by cluster - consistent labeling (1-N)
cluster_outcomes = []
cluster_ids = []
for i, cluster_id in enumerate(unique_clusters):
    cluster_mask = clusters == cluster_id
    if np.sum(cluster_mask) > 0:
        cluster_outcome_rate = np.mean(y[cluster_mask])
        cluster_outcomes.append(cluster_outcome_rate)
        cluster_ids.append(i + 1)  # Start from 1 instead of 0

axes[1,1].bar(cluster_ids, cluster_outcomes, color='lightcoral', alpha=0.7)
axes[1,1].set_title('Mortality Rate by Cluster', fontsize=14, fontweight='bold')
axes[1,1].set_xlabel('Cluster ID')
axes[1,1].set_ylabel('Mortality Rate')
axes[1,1].grid(True, alpha=0.3)

# Set x-axis ticks to show 1,2,3...N
axes[1,1].set_xticks(cluster_ids)
axes[1,1].set_xticklabels(cluster_ids)

# Add value labels on bars
for i, v in enumerate(cluster_outcomes):
    axes[1,1].text(cluster_ids[i], v + max(cluster_outcomes)*0.01, f'{v:.3f}',
                   ha='center', va='bottom', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.show()

print(f"\n=== SHAP + Outcome Cluster Analysis Summary ===")
print(f"Total clusters found: {n_clusters}")
print(f"Noise points: {n_noise}")
print(f"Unique cluster IDs: {unique_clusters}")

for i, cluster_id in enumerate(unique_clusters):
    cluster_mask = clusters == cluster_id
    cluster_size = np.sum(cluster_mask)
    cluster_outcome_rate = np.mean(y[cluster_mask]) if cluster_size > 0 else 0
    print(f"Cluster {i+1}: {cluster_size} patients, Mortality rate: {cluster_outcome_rate:.3f}")

if n_noise > 0:
    noise_mask = clusters == -1
    noise_outcome_rate = np.mean(y[noise_mask]) if n_noise > 0 else 0
    print(f"Noise: {n_noise} patients, Mortality rate: {noise_outcome_rate:.3f}")


In [ ]:
# SHAP → UMAP → DBSCAN Clustering Analysis
import umap
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from scipy.stats import chi2_contingency
import numpy as np

print("=== SHAP Values Analysis ===")
print(f"SHAP values shape: {shap_values.shape}")
print(f"Feature names: {len(feature_names)}")

# Step 1: Prepare SHAP values for UMAP
# Use original SHAP values (with signs) to preserve risk/protective information
print(f"\n=== Step 1: SHAP Values Preparation ===")
print(f"SHAP values shape: {shap_values.shape}")
print("Using original SHAP values (with signs) to preserve risk/protective information")

# Step 2: UMAP Dimensionality Reduction
print(f"\n=== Step 2: UMAP Dimensionality Reduction ===")
umap_reducer = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.3,
    metric='euclidean',
    random_state=42
)

# Fit UMAP on original SHAP values (preserving signs)
shap_umap = umap_reducer.fit_transform(shap_values)
print(f"UMAP embedding shape: {shap_umap.shape}")

# Step 3: DBSCAN Clustering
print(f"\n=== Step 3: DBSCAN Clustering ===")
dbscan = DBSCAN(eps=0.3, min_samples=100)
clusters = dbscan.fit_predict(shap_umap)

n_clusters = len(set(clusters)) - (1 if -1 in clusters else 0)
n_noise = list(clusters).count(-1)

print(f"Number of clusters: {n_clusters}")
print(f"Number of noise points: {n_noise}")
print(f"Cluster distribution: {np.bincount(clusters + 1)}")

# Step 4: Visualization
print(f"\n=== Step 4: Visualization ===")
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Get unique clusters (excluding noise points with -1)
unique_clusters = sorted([c for c in set(clusters) if c != -1])
n_unique_clusters = len(unique_clusters)
print(f"Unique clusters (excluding noise): {unique_clusters}")
print(f"Number of unique clusters: {n_unique_clusters}")

# Plot 1: UMAP embedding with clusters
colors = cm.get_cmap('tab20', n_unique_clusters)

scatter = axes[0,0].scatter(shap_umap[:, 0], shap_umap[:, 1], c=clusters,
                           cmap=colors, alpha=0.7, s=20)
axes[0,0].set_title('UMAP Embedding of SHAP Values (Clustered)', fontsize=14, fontweight='bold')
axes[0,0].set_xlabel('UMAP 1')
axes[0,0].set_ylabel('UMAP 2')

# Create custom colorbar with proper labels - show 1-11 for each color
cbar = plt.colorbar(scatter, ax=axes[0,0])
cbar.set_label('Cluster ID', rotation=270, labelpad=15)
# Set colorbar ticks to show 1-11 for each color
cbar.set_ticks(unique_clusters)
cbar.set_ticklabels([str(i+1) for i in range(len(unique_clusters))])
# Adjust tick label position to be right next to colors
cbar.ax.tick_params(axis='y', which='major', pad=5)

# Plot 2: UMAP with outcome colors
# Create color array for all points
colors = ['red' if outcome == 1 else 'blue' for outcome in y]
axes[0,1].scatter(shap_umap[:, 0], shap_umap[:, 1], c=colors, alpha=0.7, s=20)
axes[0,1].set_title('UMAP Embedding Colored by Outcome', fontsize=14, fontweight='bold')
axes[0,1].set_xlabel('UMAP 1')
axes[0,1].set_ylabel('UMAP 2')
axes[0,1].legend()

# Plot 3: Cluster size distribution - NO NOISE, consistent labeling (1-11)
cluster_counts = np.bincount(clusters + 1)  # +1 to handle -1 noise points
# Remove noise (index 0) and create labels starting from 1
cluster_counts_no_noise = cluster_counts[1:]  # Remove noise
cluster_labels = [f'Cluster {i+1}' for i in range(len(cluster_counts_no_noise))]

axes[1,0].bar(cluster_labels, cluster_counts_no_noise, color='skyblue', alpha=0.7)
axes[1,0].set_title('Cluster Size Distribution', fontsize=14, fontweight='bold')
axes[1,0].set_xlabel('Cluster ID')
axes[1,0].set_ylabel('Number of Patients')
axes[1,0].tick_params(axis='x', rotation=0)  # Changed from 45 to 0
axes[1,0].grid(True, alpha=0.3)

# Set x-axis ticks to show 1,2,3...11 instead of "Cluster 1", "Cluster 2"
axes[1,0].set_xticks(range(len(cluster_counts_no_noise)))
axes[1,0].set_xticklabels([str(i+1) for i in range(len(cluster_counts_no_noise))])

# Add value labels on bars
for i, v in enumerate(cluster_counts_no_noise):
    axes[1,0].text(i, v + max(cluster_counts_no_noise)*0.01, str(v), ha='center', va='bottom', fontweight='bold')

# Plot 4: Outcome distribution by cluster - consistent labeling (1-11)
cluster_outcomes = []
cluster_ids = []
for i, cluster_id in enumerate(unique_clusters):
    cluster_mask = clusters == cluster_id
    if np.sum(cluster_mask) > 0:
        cluster_outcome_rate = np.mean(y[cluster_mask])
        cluster_outcomes.append(cluster_outcome_rate)
        cluster_ids.append(i + 1)  # Start from 1 instead of 0

axes[1,1].bar(cluster_ids, cluster_outcomes, color='lightcoral', alpha=0.7)
axes[1,1].set_title('Mortality Rate by Cluster', fontsize=14, fontweight='bold')
axes[1,1].set_xlabel('Cluster ID')
axes[1,1].set_ylabel('Mortality Rate')
axes[1,1].grid(True, alpha=0.3)

# Set x-axis ticks to show 1,2,3...11
axes[1,1].set_xticks(cluster_ids)
axes[1,1].set_xticklabels(cluster_ids)

# Add value labels on bars
for i, v in enumerate(cluster_outcomes):
    axes[1,1].text(cluster_ids[i], v + max(cluster_outcomes)*0.01, f'{v:.3f}',
                   ha='center', va='bottom', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.show()

print(f"\n=== Cluster Analysis Summary ===")
print(f"Total clusters found: {n_clusters}")
print(f"Noise points: {n_noise}")
print(f"Unique cluster IDs: {unique_clusters}")

for i, cluster_id in enumerate(unique_clusters):
    cluster_mask = clusters == cluster_id
    cluster_size = np.sum(cluster_mask)
    cluster_outcome_rate = np.mean(y[cluster_mask]) if cluster_size > 0 else 0
    print(f"Cluster {i+1}: {cluster_size} patients, Mortality rate: {cluster_outcome_rate:.3f}")

if n_noise > 0:
    noise_mask = clusters == -1
    noise_outcome_rate = np.mean(y[noise_mask]) if n_noise > 0 else 0
    print(f"Noise: {n_noise} patients, Mortality rate: {noise_outcome_rate:.3f}")


In [ ]:
# Cluster Interpretation and Feature Importance Analysis
print("=== Cluster Feature Importance Analysis ===")

# Calculate mean SHAP values for each cluster (preserving signs)
cluster_shap_means = []
cluster_names = []

for cluster_id in range(n_clusters):
    cluster_mask = clusters == cluster_id
    if np.sum(cluster_mask) > 0:
        cluster_shap_mean = np.mean(shap_values[cluster_mask], axis=0)
        cluster_shap_means.append(cluster_shap_mean)
        cluster_names.append(f'Cluster {cluster_id}')

# Create heatmap of mean SHAP values by cluster
cluster_shap_matrix = np.array(cluster_shap_means)

# Get top features for each cluster (considering both positive and negative SHAP values)
print("\n=== Top Features by Cluster (with signs) ===")
top_n_features = 10

fig, axes = plt.subplots(2, 2, figsize=(20, 16))

# Plot 1: Heatmap of mean SHAP values by cluster (with signs)
im = axes[0,0].imshow(cluster_shap_matrix, cmap='RdBu_r', aspect='auto')
axes[0,0].set_title('Mean SHAP Values by Cluster (Red=Risk, Blue=Protective)')
axes[0,0].set_xlabel('Features')
axes[0,0].set_ylabel('Clusters')
axes[0,0].set_yticks(range(len(cluster_names)))
axes[0,0].set_yticklabels(cluster_names)

# Add colorbar
plt.colorbar(im, ax=axes[0,0])

# Plot 2: Top features for each cluster (considering magnitude and sign)
for i, cluster_name in enumerate(cluster_names):
    cluster_shap_mean = cluster_shap_means[i]
    # Get top features by absolute value but show actual SHAP values
    top_indices = np.argsort(np.abs(cluster_shap_mean))[-top_n_features:][::-1]
    top_features = [feature_names[idx] for idx in top_indices]
    top_values = cluster_shap_mean[top_indices]
    
    print(f"\n{cluster_name} - Top {top_n_features} Features:")
    for feat, val in zip(top_features, top_values):
        risk_type = "RISK" if val > 0 else "PROTECTIVE"
        print(f"  {feat}: {val:.4f} ({risk_type})")

# Plot 3: Feature importance comparison across clusters (absolute values)
feature_importance_by_cluster = np.abs(cluster_shap_matrix)
mean_importance = np.mean(feature_importance_by_cluster, axis=0)
top_global_features = np.argsort(mean_importance)[-top_n_features:][::-1]

axes[0,1].barh(range(top_n_features), mean_importance[top_global_features])
axes[0,1].set_yticks(range(top_n_features))
axes[0,1].set_yticklabels([feature_names[i] for i in top_global_features])
axes[0,1].set_title('Global Feature Importance (Mean Absolute SHAP)')
axes[0,1].set_xlabel('Mean Absolute SHAP Value')

# Plot 4: Risk vs Protective features by cluster
risk_features = np.sum(cluster_shap_matrix > 0, axis=1)
protective_features = np.sum(cluster_shap_matrix < 0, axis=1)

x = np.arange(len(cluster_names))
width = 0.35

axes[1,0].bar(x - width/2, risk_features, width, label='Risk Features', color='red', alpha=0.7)
axes[1,0].bar(x + width/2, protective_features, width, label='Protective Features', color='blue', alpha=0.7)
axes[1,0].set_xlabel('Clusters')
axes[1,0].set_ylabel('Number of Features')
axes[1,0].set_title('Risk vs Protective Features by Cluster')
axes[1,0].set_xticks(x)
axes[1,0].set_xticklabels(cluster_names)
axes[1,0].legend()

# Plot 5: SHAP value distribution by cluster
axes[1,1].boxplot([cluster_shap_means[i] for i in range(len(cluster_names))], 
                  labels=cluster_names)
axes[1,1].set_title('SHAP Value Distribution by Cluster')
axes[1,1].set_ylabel('SHAP Values')
axes[1,1].axhline(y=0, color='black', linestyle='--', alpha=0.5)
axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print(f"\n=== Cluster Risk Profile Summary ===")
for i, cluster_name in enumerate(cluster_names):
    cluster_shap_mean = cluster_shap_means[i]
    risk_features = np.sum(cluster_shap_mean > 0)
    protective_features = np.sum(cluster_shap_mean < 0)
    net_risk = np.sum(cluster_shap_mean)
    
    print(f"{cluster_name}:")
    print(f"  Risk features: {risk_features}")
    print(f"  Protective features: {protective_features}")
    print(f"  Net risk score: {net_risk:.4f}")
    print(f"  Risk level: {'HIGH' if net_risk > 0 else 'LOW' if net_risk < 0 else 'NEUTRAL'}")


In [ ]:
# Specific Interaction Analysis: EVLP × DCD/DBD
print("=== EVLP × DCD/DBD Interaction Analysis ===")

# variables exist in the dataset
perfusion_vars  = ['PERFUSED_PRIOR']
dcd_vbd_vars = ['DONOR_TYPE_BINARY']

print(f"Perfusion variables: {perfusion_vars}")
print(f"DCD/DBD variables: {dcd_vbd_vars}")


# Create interaction analysis
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Plot 1: EVLP vs Outcome by Donor Type
if len(perfusion_vars) > 0 and len(dcd_vbd_vars) > 0:
    perf_var = perfusion_vars[0]  # Use first perfusion variable
    donor_var = dcd_vbd_vars[0]  # Use first donor type variable
    
    # Create interaction plot
    interaction_data = df.groupby([perf_var, donor_var])['one_year_mortality'].agg(['mean', 'count']).reset_index()
    
    if len(interaction_data) > 0:
        pivot_table = interaction_data.pivot(index=perf_var, columns=donor_var, values='mean')
        sns.heatmap(pivot_table, annot=True, fmt='.3f', cmap='RdYlBu_r', ax=axes[0,0])
        axes[0,0].set_title(f'Outcome Rate: {perf_var} × {donor_var}')
    
    # Plot 2: Count heatmap
    count_table = interaction_data.pivot(index=perf_var, columns=donor_var, values='count')
    sns.heatmap(count_table, annot=True, fmt='d', cmap='Blues', ax=axes[0,1])
    axes[0,1].set_title(f'Sample Count: {perf_var} × {donor_var}')

# Plot 3: Cluster distribution by EVLP status
if len(perfusion_vars) > 0:
    perf_var = perfusion_vars[0]
    cluster_perf_crosstab = pd.crosstab(clusters, df[perf_var])
    sns.heatmap(cluster_perf_crosstab, annot=True, fmt='d', cmap='viridis', ax=axes[0,2])
    axes[0,2].set_title('Cluster Distribution by Perfusion Status')

# Plot 4: Outcome rate by cluster and perfusion
if len(perfusion_vars) > 0:
    perf_var = perfusion_vars[0]
    cluster_perf_outcome = df.groupby([clusters, df[perf_var]])['one_year_mortality'].mean().unstack()
    sns.heatmap(cluster_perf_outcome, annot=True, fmt='.3f', cmap='RdYlBu_r', ax=axes[1,0])
    axes[1,0].set_title('Outcome Rate: Cluster × Perfusion')

# Plot 5: Feature importance by perfusion status
if len(perfusion_vars) > 0:
    perf_var = perfusion_vars[0]
    perf_groups = df[perf_var].unique()
    
    for i, group in enumerate(perf_groups):
        if pd.notna(group):
            group_mask = df[perf_var] == group
            if np.sum(group_mask) > 0:
                group_shap_mean = np.mean(shap_values[group_mask], axis=0)
                top_indices = np.argsort(np.abs(group_shap_mean))[-5:][::-1]
                
                axes[1,1].barh(range(5), group_shap_mean[top_indices], 
                          label=f'{perf_var}={group}', alpha=0.7)
    
    axes[1,1].set_yticks(range(5))
    axes[1,1].set_yticklabels([feature_names[i] for i in top_indices])
    axes[1,1].set_title('Top Features by Perfusion Status (with signs)')
    axes[1,1].legend()

# Plot 6: Statistical significance test
if len(perfusion_vars) > 0 and len(dcd_vbd_vars) > 0:
    perf_var = perfusion_vars[0]
    donor_var = dcd_vbd_vars[0]
    
    # Chi-square test for independence
    contingency_table = pd.crosstab(df[perf_var], df[donor_var])
    chi2, p_value, dof, expected = chi2_contingency(contingency_table)
    
    axes[1,2].text(0.5, 0.7, f'Chi-square Test Results:', ha='center', va='center', 
                   fontsize=12, transform=axes[1,2].transAxes)
    axes[1,2].text(0.5, 0.5, f'χ² = {chi2:.4f}', ha='center', va='center', 
                   fontsize=12, transform=axes[1,2].transAxes)
    axes[1,2].text(0.5, 0.3, f'p-value = {p_value:.4f}', ha='center', va='center', 
                   fontsize=12, transform=axes[1,2].transAxes)
    axes[1,2].set_title('Statistical Significance')
    axes[1,2].axis('off')

plt.tight_layout()
plt.show()

# Print detailed interaction analysis
print(f"\n=== Detailed Interaction Analysis ===")
if len(perfusion_vars) > 0 and len(dcd_vbd_vars) > 0:
    perf_var = perfusion_vars[0]
    donor_var = dcd_vbd_vars[0]
    
    print(f"Analyzing interaction between {perf_var} and {donor_var}")
    
    for perf_val in df[perf_var].unique():
        if pd.notna(perf_val):
            for donor_val in df[donor_var].unique():
                if pd.notna(donor_val):
                    mask = (df[perf_var] == perf_val) & (df[donor_var] == donor_val)
                    if np.sum(mask) > 0:
                        outcome_rate = df[mask]['one_year_mortality'].mean()
                        count = np.sum(mask)
                        print(f"  {perf_var}={perf_val}, {donor_var}={donor_val}: {outcome_rate:.3f} ({count} patients)")
else:
    print("EVLP or DCD/DBD variables not found in dataset")
    print("Available variables:", df.columns.tolist())
